In [ ]:
import os
PATH = "/kaggle/input/datasets/jakeadam68/blackwell-deps-cu128/blackwell_deps_pack" # Le nom de ton dataset

# on installe le fichier exact de Torch 2.10.0 cu128 en premier
# Cela évite que pip ne choisisse la version 2.11 par erreur
%pip install --no-index --find-links={PATH} torch==2.10.0+cu128 torchvision torchaudio

import torch
print(f"PyTorch installé : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
print(f"GPU détecté : {torch.cuda.get_device_name(0)}")

In [ ]:
# Maintenant on installe les extensions qui dépendent de la 2.10
%pip install --no-index --find-links={PATH} torch-scatter torch-sparse torch-cluster torch-spline-conv
print("Extensions PyG installées.")

In [ ]:
# Ici on installe tout le reste (Transformers, RDKit, e3nn, etc.)
# pip verra que Torch est déjà installé et ne touchera plus à la version
%pip install --no-index --find-links={PATH} torch-geometric e3nn transformers accelerate bitsandbytes safetensors huggingface_hub fair-esm rdkit pubchempy py3Dmol biopython biotite joblib tqdm pandas numpy polars pyarrow fastparquet
print("Environnement complet opérationnel !")

In [ ]:
%%bash
echo "Début de l'installation Hors-Ligne"

# Remplacez ce chemin par le vrai chemin de votre dossier my_wheels uploadé
WHEELS_DIR="/kaggle/input/datasets/jakeadam68/blackwell-deps-cu128/offline_wheels/my_wheels"

# on installe explicitement juste ce dont on a besoin, sans casser le numpy de Kaggle
pip install freesasa openmm pdbfixer --no-index --find-links "$WHEELS_DIR" --quiet

echo "Tous les packages sont installés !"

In [ ]:
import os
import random
import numpy as np
import torch
import gc
import warnings

# on désactive les warnings 
warnings.filterwarnings('ignore')
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
# on crée une fonction pour la reproductibilité maximale 
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # on adopte la reproductibilité stricte
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    
    print(f"la graine aléatoire est fixée à {seed}")
    
seed_everything(42)
# on choisit une configuration optimisée pour GPU H100
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"\nle GPU détecté est : {gpu_name}")
    print(f"la VRAM disponible : {vram_gb:.2f} GB")
    # on choisit une optimisations Hopper / H100
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
    # on active le torch.compile pour un gain de vitesse important
    print(f"le TensorFloat-32 est actif : {torch.backends.cuda.matmul.allow_tf32}")
else:
    DEVICE = torch.device("cpu")
# on procède à un nettoyage de la mémoire
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print("la configuration de l'environnement est terminée")

In [ ]:
!echo "=== Modèle du CPU ==="
!cat /proc/cpuinfo | grep "model name" | uniq

!echo "=== Nombre de vCPUs disponibles ==="
!nproc

## **Partie Génération des Embeddings ESM-2 locaux pour chaque acide aminé et des structures 3D des protéines avec ESMfold**

In [ ]:
import torch
from transformers import AutoTokenizer, EsmModel
import numpy as np
import pickle
import pandas as pd
from tqdm.auto import tqdm
import re
import os
import gc
import logging
from transformers import logging as hf_logging


# 1. CONFIGURATION & NETTOYAGE

hf_logging.set_verbosity_error()
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("🧬 Génération des Embeddings ESM-2 Locaux (Centrés sur Mutation)")

# Chargement des données
df_patho = pd.read_parquet("/kaggle/input/datasets/rayanchakibidris/cosmic-genomescreen-v103/pathogenic_sequences_sota_final.parquet")
df_patho = df_patho[['uniprot_id', 'mutation_hgvsp', 'mutated_sequence', 'wt_sequence']].copy()

# on identifie les mutations uniques
unique_mt = df_patho[['uniprot_id', 'mutation_hgvsp', 'mutated_sequence', 'wt_sequence']].copy()
unique_mt['mt_key'] = unique_mt['uniprot_id'] + "_" + unique_mt['mutation_hgvsp']

print(f"Nombre de séquences mutées uniques à traiter : {len(unique_mt)}")

# Configuration du modèle ESM-2 (650M)
model_path = "/kaggle/input/datasets/rayanchakibidris/esm2-t33-650m-local" 
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = EsmModel.from_pretrained(
    model_path, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)
model.eval()
torch.backends.cuda.matmul.allow_tf32 = True

# 2. Fonctions de traitement

def get_mutation_pos(hgvsp_str, seq_len):
    """Extrait la position numérique de la mutation depuis la chaîne HGVS"""
    match = re.search(r'p\.[A-Z][a-z]{2}(\d+)|p\.[A-Z](\d+)', str(hgvsp_str))
    if match:
        pos = int(match.group(1) if match.group(1) else match.group(2))
        return pos - 1 # Conversion en index 0
    return seq_len // 2

def get_esm_local_embedding_centered(seq, mut_pos, model, tokenizer):
    """
    Calcule sur 1024 AA pour le contexte, mais ne retourne que 256 AA 
    autour de la mutation pour économiser le disque.
    """
    L = len(seq)
    context_len = 1024
    window_len = 256
    
    # 1. Définir la fenêtre de contexte (1024)
    if L <= context_len:
        start_ctx = 0
        input_seq = seq
    else:
        start_ctx = max(0, mut_pos - context_len // 2)
        end_ctx = min(L, start_ctx + context_len)
        if end_ctx == L: start_ctx = max(0, end_ctx - context_len)
        input_seq = seq[start_ctx:end_ctx]

    with torch.no_grad():
        inputs = tokenizer(input_seq, return_tensors="pt", padding=False, truncation=True).to(model.device)
        outputs = model(**inputs)
        
        # on récupère les embeddings (L_context, 1280)
        embeddings = outputs.last_hidden_state[0, 1:-1, :].detach().half().cpu().numpy()
        
    # 2. Calculer la position de la mutation relative à la fenêtre de 1024
    rel_mut_pos = mut_pos - start_ctx
    
    # 3. Extraire la fenêtre de 256 AA centrée sur la mutation
    s_win = max(0, rel_mut_pos - window_len // 2)
    e_win = s_win + window_len
    if e_win > embeddings.shape[0]:
        e_win = embeddings.shape[0]
        s_win = max(0, e_win - window_len)
        
    compact_emb = embeddings[s_win:e_win, :]
    
    # Padding si la protéine est trop courte pour faire 256 AA
    if compact_emb.shape[0] < window_len:
        pad_width = window_len - compact_emb.shape[0]
        compact_emb = np.pad(compact_emb, ((0, pad_width), (0, 0)), mode='constant')
        
    return compact_emb, start_ctx # retourne en float16


# 3. Boucle d'inférence 

mt_embs = {}
wt_embs = {}

print("\n🧠 Inférence ESM-2 en cours...")
for i, (_, row) in enumerate(tqdm(unique_mt.iterrows(), total=len(unique_mt), desc="Variants")):
    mkey = row['mt_key']
    pos = get_mutation_pos(row['mutation_hgvsp'], len(row['mutated_sequence']))
    
    # MT
    emb_mt, offset = get_esm_local_embedding_centered(row['mutated_sequence'], pos, model, tokenizer)
    mt_embs[mkey] = {'emb': emb_mt, 'offset': offset}
    
    # WT
    emb_wt, _ = get_esm_local_embedding_centered(row['wt_sequence'], pos, model, tokenizer)
    wt_embs[mkey] = {'emb': emb_wt, 'offset': offset}

    if i % 100 == 0:
        torch.cuda.empty_cache()
        gc.collect()

# Sauvegarde
final_data = {'wt': wt_embs, 'mt': mt_embs}
with open("esm2_pathogenic_bundle_local.pkl", "wb") as f:
    pickle.dump(final_data, f)

print(f"\nTerminé ! Embeddings Locaux sauvegardés.")
print(f"Le nombre de variants traités est : {len(mt_embs)}")

if len(mt_embs) > 0:
    first_key = list(mt_embs.keys())[0]
    print(f"Exemple de shape pour {first_key} : {mt_embs[first_key]['emb'].shape}")
    print(f"L'offset associé : {mt_embs[first_key]['offset']}")
else:
    print("Aucun variant n'a été traité.")

In [ ]:
import os
import shutil
import subprocess
import torch
from transformers import AutoTokenizer, EsmForProteinFolding
import freesasa
import numpy as np
import pandas as pd
import pickle
import re
import shutil
from tqdm.auto import tqdm
import gc

print("ESMFold (Backbone) + FoldX 5.1 (Side-chains)")

MAX_LEN = 256         
output_file = "esmfold_foldx_hybrid_5920.pkl" 

# =============================================================================
# 1. Récupération des 5 920 variants
# =============================================================================
path_silver = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/bindingdb_wildtype_mutated_fullseq.parquet"
df_all = pd.read_parquet(path_silver)
df_unique_variants = df_all.drop_duplicates(subset=['variant_id']).copy()
print(f"Nombre total de variants uniques à traiter : {len(df_unique_variants):,}")

# =============================================================================
# 2. Préparation de FoldX 
# =============================================================================
FOLDX_INPUT_PATH = "/kaggle/input/datasets/rayanchakibidris/foldx-5-1-linux/foldx_20270131"
FOLDX_WORKING_PATH = "/kaggle/working/foldx_5.1_linux"

print("Préparation de l'exécutable FoldX...")
if not os.path.exists(FOLDX_WORKING_PATH):
    shutil.copy(FOLDX_INPUT_PATH, FOLDX_WORKING_PATH)
os.system(f"chmod +x {FOLDX_WORKING_PATH}")

# =============================================================================
# 3. Chargement d'ESMfold en float32 
# =============================================================================
print("Chargement d'ESMFold (en Float32)...")
model_path = "/kaggle/input/datasets/jakeadam68/esmfold-v1-local-repo"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = EsmForProteinFolding.from_pretrained(model_path, low_cpu_mem_usage=True).to("cuda")
model.eval()

AA_3_TO_1 = {'Ala':'A', 'Arg':'R', 'Asn':'N', 'Asp':'D', 'Cys':'C', 'Glu':'E', 'Gln':'Q', 'Gly':'G', 'His':'H', 'Ile':'I', 'Leu':'L', 'Lys':'K', 'Met':'M', 'Phe':'F', 'Pro':'P', 'Ser':'S', 'Thr':'T', 'Trp':'W', 'Tyr':'Y', 'Val':'V'}

# =============================================================================
# 4. Fonctions de traitement
# =============================================================================
def get_window_indices(seq_len, hgvsp_str):
    match = re.search(r'p\.[A-Z][a-z]{2}(\d+)|p\.[A-Z](\d+)', str(hgvsp_str))
    pos = int(match.group(1) if match.group(1) else match.group(2)) if match else seq_len // 2
    center = pos - 1
    if seq_len <= MAX_LEN: return 0, seq_len
    start = max(0, center - MAX_LEN // 2)
    end = start + MAX_LEN
    if end > seq_len: end, start = seq_len, max(0, seq_len - MAX_LEN)
    return int(start), int(end)

def extract_ca_data(output):
    pos = output.positions.detach().cpu() 
    pld = output.plddt.detach().cpu()     
    if pos.ndim == 5: pos = pos[0] 
    elif pos.ndim == 4 and pos.shape[0] != 8: pos = pos[0] 
    if pos.ndim == 4: pos = pos[-1] 
    ca_coords = pos[:, 1, :].numpy() if pos.ndim == 3 else pos.numpy()
    if pld.ndim == 4: pld = pld[0]
    if pld.ndim == 3: pld = pld[-1] 
    ca_plddt = pld[:, 1].numpy() * 100 if pld.ndim == 2 else pld.flatten().numpy() * 100
    return ca_coords, ca_plddt

def run_foldx_mutation(pdb_file_wt, wt_aa, mt_aa, position_pdb):
    mutation_str = f"{wt_aa}A{position_pdb}{mt_aa};"
    with open("individual_list.txt", "w") as f: f.write(mutation_str)
    cmd = f"{FOLDX_WORKING_PATH} --command=BuildModel --pdb={pdb_file_wt} --mutant-file=individual_list.txt --water=CRYSTAL --out-pdb=1"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    mutant_filename = pdb_file_wt.replace(".pdb", "_1.pdb")

    ddg, electro, solv_hydro, clash = 0.0, 0.0, 0.0, 0.0
    dif_filename = "Dif_" + pdb_file_wt.replace(".pdb", ".fxout")
    
    if os.path.exists(dif_filename):
        try:
            with open(dif_filename, 'r') as f:
                lines = f.readlines()
                
            # Recherche dynamique de la ligne d'en-tête de FoldX 5.1
            header_idx = -1
            for idx, line in enumerate(lines):
                if "total energy" in line:
                    header_idx = idx
                    break
                    
            if header_idx != -1 and len(lines) > header_idx + 1:
                parts = lines[header_idx + 1].strip().split('\t')
                if len(parts) > 8:
                    ddg        = float(parts[1]) # Index [1] : total energy (ddG) [2]
                    electro    = float(parts[5]) # Index [5] : Electrostatics
                    solv_hydro = float(parts[7]) # Index [7] : Solvation Hydrophobic
                    clash      = float(parts[8]) # Index [8] : Van der Waals clashes
        except Exception:
            pass
            
    if os.path.exists(mutant_filename): 
        # on retourne le fichier ET les 4 variables calculées par FoldX 5.1 !
        return mutant_filename, ddg, electro, solv_hydro, clash
    return None, 0.0, 0.0, 0.0, 0.0

def get_sasa_from_pdb(pdb_filename, pdb_mut_pos):
    try:
        struct = freesasa.Structure(pdb_filename)
        result = freesasa.calc(struct)
        selections = freesasa.selectArea([f"target, resi {pdb_mut_pos} and chain A"], struct, result)
        return selections['target']
    except Exception:
        return 0.0

def get_full_atom_packing_from_pdb(pdb_filename, target_res_seq):
    target_atoms = []
    other_atoms = []
    with open(pdb_filename, 'r') as f:
        for line in f:
            if line.startswith("ATOM"):
                try:
                    res_seq = int(line[22:26].strip())
                    x, y, z = float(line[30:38].strip()), float(line[38:46].strip()), float(line[46:54].strip())
                    if res_seq == target_res_seq: target_atoms.append([x, y, z])
                    else: other_atoms.append([x, y, z])
                except ValueError: continue
    if not target_atoms or not other_atoms: return 0.0
    target_atoms, other_atoms = np.array(target_atoms), np.array(other_atoms)
    diff = target_atoms[:, np.newaxis, :] - other_atoms[np.newaxis, :, :]
    dist_sq = np.sum(diff ** 2, axis=-1)
    mask = (dist_sq <= 8.0**2) & (dist_sq > 0.01)
    if not np.any(mask): return 0.0
    return float(np.sum(1.0 / dist_sq[mask]))

# =============================================================================
# 5. La boucle hybride
# =============================================================================
master_dict = {'mt': {}, 'wt': {}}
wt_cache = {} 
SAVE_INTERVAL = 50 

output_file = "esmfold_foldx_hybrid_5920.pkl" 
INPUT_RESUME_FILE = "/kaggle/input/votre-dossier-dataset/esmfold_foldx_hybrid_5920.pkl"

# 1. Si le fichier n'est pas encore dans le dossier de travail, on va le copier depuis l'Input
if not os.path.exists(output_file) and os.path.exists(INPUT_RESUME_FILE):
    print(f"Copie du fichier de reprise depuis l'Input vers le dossier de travail...")
    shutil.copy(INPUT_RESUME_FILE, output_file)

# 2. Logique de chargement
if os.path.exists(output_file):
    print(f"\nFichier de sauvegarde trouvé ({output_file}). Chargement en cours...")
    with open(output_file, "rb") as f: 
        master_dict = pickle.load(f)
        
    if 'wt_cache' not in master_dict: master_dict['wt_cache'] = {}
    print(f"Reprise confirmée ! {len(master_dict['mt'])} variants déjà traités.")
else:
    print("\nAucun fichier existant. Lancement d'une nouvelle génération de zéro...")
    master_dict = {'mt': {}, 'wt': {}, 'wt_cache': {}}

wt_cache = master_dict['wt_cache']

try:
    for idx, row in tqdm(df_unique_variants.iterrows(), total=len(df_unique_variants), desc="Hybridation 3D"):
        vid = row['variant_id']
        if vid in master_dict['mt']: continue 

        seq_mt, seq_wt, hgvsp, uid = row['mutated_sequence'], row['wt_sequence'], row['mutation_hgvsp'], row['uniprot_id']
        
        indices = get_window_indices(len(seq_mt), hgvsp)
        if indices is None: continue
        s, e = indices
        win_seq_mt, win_seq_wt, win_key = seq_mt[s:e], seq_wt[s:e], (uid, s, e)
        
        match = re.search(r'p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})', str(hgvsp))
        if not match: continue
        wt_aa_3, pos_str, mt_aa_3 = match.groups()
        wt_aa, mt_aa = AA_3_TO_1.get(wt_aa_3), AA_3_TO_1.get(mt_aa_3)
        
        rel_idx = int(pos_str) - 1 - s
        pdb_mut_pos = rel_idx + 1

        with torch.no_grad():
            # A. ESMFold MT 
            out_mt = model(**tokenizer([win_seq_mt], return_tensors="pt", add_special_tokens=False).to("cuda"))
            coords_mt, plddt_mt = extract_ca_data(out_mt)

            # B. ESMFold WT + FoldX 
            wt_filename = f"wt_{uid}.pdb"
            if win_key not in wt_cache:
                out_wt = model(**tokenizer([win_seq_wt], return_tensors="pt", add_special_tokens=False).to("cuda"))
                coords_wt, plddt_wt = extract_ca_data(out_wt)
                
                pdb_wt_string = model.output_to_pdb(out_wt)[0]
                with open(wt_filename, "w") as f: f.write(pdb_wt_string)
                
                sasa_wt = get_sasa_from_pdb(wt_filename, pdb_mut_pos)
                packing_wt = get_full_atom_packing_from_pdb(wt_filename, pdb_mut_pos)
                
                wt_cache[win_key] = {'coords': coords_wt, 'plddt': plddt_wt, 'sasa': sasa_wt, 'packing': packing_wt}
            
            res_wt = wt_cache[win_key]
            
            vrai_delta_sasa = 0.0
            vrai_delta_packing = 0.0
            vrai_ddg = 0.0
            vrai_electro = 0.0
            vrai_solv_hydro = 0.0
            vrai_clash = 0.0
            
            mt_filename, vrai_ddg, vrai_electro, vrai_solv_hydro, vrai_clash = run_foldx_mutation(
                    wt_filename, wt_aa, mt_aa, pdb_mut_pos
                )
            
            if mt_filename is not None:
                sasa_mt = get_sasa_from_pdb(mt_filename, pdb_mut_pos)
                packing_mt = get_full_atom_packing_from_pdb(mt_filename, pdb_mut_pos)
                
                vrai_delta_sasa = sasa_mt - res_wt['sasa']
                vrai_delta_packing = packing_mt - res_wt['packing']
                os.remove(mt_filename) 
                
            import glob
            fichiers_inutiles = glob.glob(f"*{uid}*.fxout") + glob.glob(f"WT_wt_{uid}*.pdb")
            for f_poubelle in fichiers_inutiles:
                try:
                    os.remove(f_poubelle)
                except:
                    pass    

            # C. Stockage Final
            master_dict['mt'][vid] = {
                'coords': coords_mt,            
                'plddt': plddt_mt,              
                'mean_plddt': plddt_mt.mean(),  
                'true_delta_sasa': vrai_delta_sasa,     
                'true_delta_packing': vrai_delta_packing,
                'true_delta_ddg': vrai_ddg,             
                'true_delta_electro': vrai_electro,     
                'true_delta_solv_hydro': vrai_solv_hydro, 
                'true_delta_clash': vrai_clash          
            }
            master_dict['wt'][win_key] = {
                'coords': res_wt['coords'],
                'plddt': res_wt['plddt']
            }
                
        if len(master_dict['mt']) % SAVE_INTERVAL == 0:  
            with open(output_file + ".tmp", "wb") as f: pickle.dump(master_dict, f)
            os.replace(output_file + ".tmp", output_file)
            torch.cuda.empty_cache(); gc.collect()

except KeyboardInterrupt:
    print("\nbouton stop pressé ! sauvegarde d'urgence...")
except Exception as e:
    print(f"\nErreur inattendue : {e}. Sauvegarde de l'état actuel en cours...")
finally:
    with open(output_file, "wb") as f: pickle.dump(master_dict, f)
    print(f"Fichier final sauvegardé : {output_file}")

In [ ]:
import pickle
import numpy as np

print("Audit de la vérité terrain")

file_path = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/esmfold_foldx_hybrid_5920.pkl"

try:
    with open(file_path, "rb") as f:
        master_dict = pickle.load(f)
        
    mt_data = master_dict.get('mt', {})
    wt_data = master_dict.get('wt', {})
    print(f"Fichier chargé et contient : {len(mt_data)} variants Mutés et {len(wt_data)} protéines Sauvages.")

    if len(mt_data) > 0:
        sasa_vals = []
        pack_vals = []
        ddg_vals = []
        electro_vals = []
        solv_vals = []
        clash_vals = []
        
        print("\nÉchantillon de 5 variants:")
        for i, (vid, data) in enumerate(mt_data.items()):
            s_val = data.get('true_delta_sasa', 0.0)
            p_val = data.get('true_delta_packing', 0.0)
            d_val = data.get('true_delta_ddg', 0.0)
            e_val = data.get('true_delta_electro', 0.0)
            sh_val = data.get('true_delta_solv_hydro', 0.0)
            c_val = data.get('true_delta_clash', 0.0)
            
            sasa_vals.append(s_val)
            pack_vals.append(p_val)
            ddg_vals.append(d_val)
            electro_vals.append(e_val)
            solv_vals.append(sh_val)
            clash_vals.append(c_val)
            
            if i < 5:
                print(f"{vid:<20} | ΔSASA: {data['true_delta_sasa']:>8.4f} Å² | ΔPacking: {data['true_delta_packing']:>8.4f} | ΔΔG: {d_val:>6.2f} kcal/mol | ΔClash: {c_val:>6.2f}")

        sasa_vals = np.array(sasa_vals)
        pack_vals = np.array(pack_vals)
        ddg_vals = np.array(ddg_vals)
        electro_vals = np.array(electro_vals)
        solv_vals = np.array(solv_vals)
        clash_vals = np.array(clash_vals)
        
        print("\n📈 Statistiques sur ce début de dataset :")
        print("-" * 65)
        print("[TRUE DELTA SASA]")
        print(f"Moyenne  : {np.mean(sasa_vals):>8.4f} Å²")
        print(f"Ecart type  : {np.std(sasa_vals):>8.4f} Å²")
        print(f"Min/Max  : {np.min(sasa_vals):.4f} / {np.max(sasa_vals):.4f}")
        print(f"% Zéros  : {np.mean(sasa_vals == 0.0)*100:.1f} %")

        print("\n[TRUE DELTA PACKING]")
        print(f"Moyenne  : {np.mean(pack_vals):>8.4f}")
        print(f"Ecart type  : {np.std(pack_vals):>8.4f}")
        print(f"Min/Max  : {np.min(pack_vals):.4f} / {np.max(pack_vals):.4f}")
        print(f"% Zéros  : {np.mean(pack_vals == 0.0)*100:.1f} %")

        print("\n[3. TRUE DELTA DDG (Stabilité Globale)]")
        print(f"   Moyenne   : {np.mean(ddg_vals):>8.4f} kcal/mol")
        print(f"   Ecart type: {np.std(ddg_vals):>8.4f} kcal/mol")
        print(f"   Min/Max   : {np.min(ddg_vals):.4f} / {np.max(ddg_vals):.4f}")
        print(f"   % Zéros   : {np.mean(ddg_vals == 0.0)*100:.1f} %")

        print("\n[4. TRUE DELTA CLASH (Conflit Van der Waals)]")
        print(f"   Moyenne   : {np.mean(clash_vals):>8.4f} kcal/mol")
        print(f"   Ecart type: {np.std(clash_vals):>8.4f} kcal/mol")
        print(f"   Min/Max   : {np.min(clash_vals):.4f} / {np.max(clash_vals):.4f}")
        print(f"   % Zéros   : {np.mean(clash_vals == 0.0)*100:.1f} %")

        print("\n[5. TRUE DELTA ELECTROSTATICS]")
        print(f"   Moyenne   : {np.mean(electro_vals):>8.4f} kcal/mol")
        print(f"   Ecart type: {np.std(electro_vals):>8.4f} kcal/mol")
        print(f"   Min/Max   : {np.min(electro_vals):.4f} / {np.max(electro_vals):.4f}")
        print(f"   % Zéros   : {np.mean(electro_vals == 0.0)*100:.1f} %")

        print("\n[6. TRUE DELTA SOLVATION HYDROPHOBIC]")
        print(f"   Moyenne   : {np.mean(solv_vals):>8.4f} kcal/mol")
        print(f"   Ecart type: {np.std(solv_vals):>8.4f} kcal/mol")
        print(f"   Min/Max   : {np.min(solv_vals):.4f} / {np.max(solv_vals):.4f}")
        print(f"   % Zéros   : {np.mean(solv_vals == 0.0)*100:.1f} %")
        
        print(f"les clés principales détectées sont : {list(master_dict.keys())}")
        # Analyse du dictionnaire mutant (mt)
        if 'mt' in master_dict and len(master_dict['mt']) > 0:
            sample_vid = list(master_dict['mt'].keys())[1]
            sample_mt = master_dict['mt'][sample_vid]
            print(f"\nÉchantillon de la séquence protéique mutée (mt) : {sample_vid}")
            print(f"les clés internes sont : {list(sample_mt.keys())}")
            print(f"le shape des coordonnées : {np.array(sample_mt['coords']).shape}")
            print(f"le shape du pLDDT : {np.array(sample_mt['plddt']).shape}")
            print(f"le type de données : {np.array(sample_mt['coords']).dtype}")
        else:
            print("\nAucun variant muté n'a été trouvé dans 'mt'.")
            # Analyse du dictionnaire sauvage (wt)
        if 'wt' in master_dict and len(master_dict['wt']) > 0:
            sample_win = list(master_dict['wt'].keys())[1]
            sample_wt = master_dict['wt'][sample_win]
            print(f"\n🔹 Échantillon de la séquence protéique sauvage (wt) : {sample_win}")
            print(f"les clés internes : {list(sample_wt.keys())}")
            print(f"le shape des coordonnées : {np.array(sample_wt['coords']).shape}")
            print(f"le shape du pLDDT : {np.array(sample_wt['plddt']).shape}")
        else:
            print("\nAucune structure sauvage n'a été trouvée dans 'wt'.")

except Exception as e:
    print(f"Erreur lors de l'audit : {e}")

In [ ]:
import pickle
import py3Dmol
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# Configuration des couleurs officielles pour le seuil pLDDT 
COLORS = {
    'VERY_HIGH': '#0053D6', # Bleu (>90)
    'CONFIDENT': '#65CBF3', # Bleu ciel (70-90)
    'LOW':       '#FFDB13', # Jaune (50-70)
    'VERY_LOW':  '#FF7D45'  # Orange (< 50)
}
def get_plddt_color(score):
    """Assigne la couleur selon le score (gère 0-1 ou 0-100)"""
    val = score * 100 if score <= 1.0 else score
    if val > 90: return COLORS['VERY_HIGH']
    if val > 70: return COLORS['CONFIDENT']
    if val > 50: return COLORS['LOW']
    return COLORS['VERY_LOW']

def render_protein_trace(coords, plddt, title, variant_id):
    """Rendu 3D d'une trace CA-only avec cylindres et sphères"""
    view = py3Dmol.view(width=900, height=500)
    
    # Conversion en numpy et nettoyage des dimensions
    coords = np.array(coords).reshape(-1, 3)
    plddt = np.array(plddt).flatten()

    for i in range(len(coords)):
        color = get_plddt_color(plddt[i])
        pos = coords[i]
        
        # Sphère Carbone Alpha (on utilise Index 0.item() pour éviter les erreurs de type)
        x, y, z = pos[0].item(), pos[1].item(), pos[2].item()
        view.addSphere({
            'center': {'x': x, 'y': y, 'z': z},
            'radius': 0.7,
            'color': color
        })
        # Cylindre de liaison vers le résidu suivant
        if i < len(coords) - 1:
            next_pos = coords[i+1]
            xn, yn, zn = next_pos[0].item(), next_pos[1].item(), next_pos[2].item()
            view.addCylinder({
                'start': {'x': x, 'y': y, 'z': z},
                'end': {'x': xn, 'y': yn, 'z': zn},
                'radius': 0.35,
                'color': color,
                'fromCap': 1, 'toCap': 1
            })

    view.zoomTo()
    view.setBackgroundColor('#ffffff')
    # Affichage du titre et des statistiques
    display(HTML(f"<h4>🧬 {title} | {variant_id} | Résidus: {len(coords)} | pLDDT: {plddt.mean():.2f}</h4>"))
    return view.show()

def visualize_sota_mirror(variant_id, mode='mt', file_path="/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/esmfold_foldx_hybrid_5920.pkl"):
    # Chargement du dictionnaire maître
    with open(file_path, "rb") as f:
        master_dict = pickle.load(f)
    if variant_id not in master_dict['mt']:
        print(f"Variant {variant_id} non trouvé dans 'mt'.")
        return
    # Récupération du variant mutant
    data_mt = master_dict['mt'][variant_id]
    # Récupération du variant sauvage (wt) en cherchant la fenêtre correspondante
    uniprot_id = variant_id.split('_')[0]
    # on cherche dans master_dict['wt'] la clé qui commence par cet uniprot_id
    matching_wt_keys = [k for k in master_dict['wt'].keys() if k[0] == uniprot_id]
    
    if not matching_wt_keys:
        print(f"Pas de structure sauvage (wt) trouvée pour {uniprot_id}")
        return
    # on prend la première fenêtre sauvage trouvée pour ce gène
    data_wt = master_dict['wt'][matching_wt_keys[0]]
    display(HTML(f"<h2 style='color:#2c3e50'>🔍 Comparaison structurale entre la séquence wild-type et mutated-type</h2>"))
    print("Structure sauvage (wt) :")
    render_protein_trace(data_wt['coords'], data_wt['plddt'], "Référence de la protéine sauvage", uniprot_id)
    print("\nStructure mutée (mt) :")
    render_protein_trace(data_mt['coords'], data_mt['plddt'], "Variant pathogène", variant_id)
# on liste les variants disponibles dans le fichier
with open("/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/esmfold_foldx_hybrid_5920.pkl", "rb") as f:
    temp_dict = pickle.load(f)
# on charge les métadonnées pour identifier les drivers
df_meta = pd.read_parquet("/kaggle/input/datasets/rayanchakibidris/cosmic-genomescreen-v103/pathogenic_sequences_sota_final.parquet")
df_meta['variant_id'] = df_meta['uniprot_id'] + "_" + df_meta['mutation_hgvsp']
# on filtre pour ne garder que les drivers oncogéniques qui sont dans le dictionnaire 3D
available_vids = list(temp_dict['mt'].keys())
df_drivers_ready = df_meta[(df_meta['is_oncogenic'] == True) & (df_meta['variant_id'].isin(available_vids))]
if not df_drivers_ready.empty:
    # Priorité aux gènes drivers pour une belle démonstration
    stars = ['TP53', 'EGFR', 'KRAS', 'BRAF', 'PIK3CA']
    star_match = df_drivers_ready[df_drivers_ready['gene_symbol'].isin(stars)]
    # on sélectionne l'ID final
    if not star_match.empty:
        selected_vid = star_match.iloc[0]['variant_id']
        print(f"Gène oncogénique majeur détecté : {star_match.iloc[0]['gene_symbol']}")
    else:
        selected_vid = df_drivers_ready.iloc[6]['variant_id']
        print(f"Gène oncogénique driver détecté : {df_drivers_ready.iloc[6]['gene_symbol']}")
    # Visualisation en mode cylindre
    # on peux changer le mode='mt' par mode='wt' pour voir la version sauvage
    visualize_sota_mirror(selected_vid, mode='mt')
else:
    print("Aucun driver oncogénique trouvé dans les structures actuelles.")
    if available_vids:
        print(f"Affichage du premier variant disponible : {available_vids[0]}")
        visualize_sota_mirror(available_vids[0], mode='mt')
    

In [ ]:
import numpy as np
import pickle
import pandas as pd
import re
from Bio.SVDSuperimposer import SVDSuperimposer
from tqdm.auto import tqdm

# Chargement du fichier contenant les structures 3D des séquences protéiques wild-type et mutated-type
input_file = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/esmfold_foldx_hybrid_5920.pkl" 
with open(input_file, "rb") as f:
    master_dict = pickle.load(f)
# on a besoin des métadonnées pour savoir où est la mutation dans la fenêtre
df_patho = pd.read_parquet("/kaggle/input/datasets/rayanchakibidris/cosmic-genomescreen-v103/pathogenic_sequences_sota_final.parquet")
df_patho['variant_id'] = df_patho['uniprot_id'] + "_" + df_patho['mutation_hgvsp']
variant_info = df_patho.set_index('variant_id').to_dict('index')

def get_window_indices(seq_len, hgvsp_str, max_len=256):
    """on utilise la meme logique de fenêtrage que lors de la génération 3D"""
    match = re.search(r'p\.[A-Z][a-z]{2}(\d+)|p\.[A-Z](\d+)', str(hgvsp_str))
    pos_val = match.group(1) if match.group(1) else match.group(2)
    pos = int(pos_val) if pos_val else seq_len // 2
    center = pos - 1
    if seq_len <= max_len: return 0, seq_len, center
    start = max(0, center - max_len // 2)
    end = start + max_len
    if end > seq_len:
        end = seq_len
        start = max(0, end - max_len)
    # on renvoie aussi l'index local du centre dans la fenêtre
    local_center = center - start
    return start, end, local_center

def calculate_local_rmsd(coords_wt, coords_mt, center_idx, radius=15.0):
    """
    Calcule le RMSD uniquement pour les résidus dans un rayon de X Ångströms
    autour du Carbone Alpha muté.
    """
    # Trouver la position 3D du centre sur la structure wt
    mut_center_coord = coords_wt[center_idx]
    # Identifier les indices des résidus voisins
    distances = np.linalg.norm(coords_wt - mut_center_coord, axis=1)
    neighbor_indices = np.where(distances <= radius)[0]
    if len(neighbor_indices) < 3: # on adopte une sécurité mathématique pour l'algorithme de Kabsch
        return 0.0
    # Extraire les sous-ensembles de coordonnées
    local_wt = coords_wt[neighbor_indices]
    local_mt = coords_mt[neighbor_indices]
    # Superposition optimale et RMSD
    try:
        sup = SVDSuperimposer()
        sup.set(local_wt, local_mt)
        sup.run()
        return sup.get_rms()
    except Exception:
        return np.nan
# Boucle de calcul
rmsd_results = {}
print(f"🔬 Calcul du RMSD Local (Rayon: 15Å) pour {len(master_dict['mt'])} variants :")

for vid, data_mt in tqdm(master_dict['mt'].items()):
    info = variant_info.get(vid)
    if not info: continue
    
    uid = vid.split('_')[0]
    # Retrouver la fenêtre et l'index local de la mutation
    s, e, local_mut_idx = get_window_indices(len(info['mutated_sequence']), info['mutation_hgvsp'])
    win_key = (uid, s, e)
    
    if win_key in master_dict['wt']:
        data_wt = master_dict['wt'][win_key]
        
        try:
            # Coordonnées
            coords_wt = np.array(data_wt['coords'])
            coords_mt = np.array(data_mt['coords'])
            
            # Calcul du RMSD global
            sup_glob = SVDSuperimposer()
            sup_glob.set(coords_wt, coords_mt)
            sup_glob.run()
            global_rmsd = sup_glob.get_rms()
            # Calcul du RMSD local
            local_rmsd = calculate_local_rmsd(coords_wt, coords_mt, local_mut_idx, radius=15.0)
            rmsd_results[vid] = {
                'global_rmsd': float(global_rmsd),
                'local_rmsd_15A': float(local_rmsd),
                'n_neighbors': int(np.sum(np.linalg.norm(coords_wt - coords_wt[local_mut_idx], axis=1) <= 15.0))
            }
        except:
            continue
            
# Sauvegarde du fichier au format .pkl et affichage des statistiques
with open("structural_impact_labels_sota.pkl", "wb") as f:
    pickle.dump(rmsd_results, f)
    
# Extraction des valeurs pour faciliter les calculs
global_rmsd_vals = [v['global_rmsd'] for v in rmsd_results.values()]
local_rmsd_vals = [v['local_rmsd_15A'] for v in rmsd_results.values()]

# Calcul des moyennes
avg_global = np.mean(global_rmsd_vals)
avg_local = np.mean(local_rmsd_vals)

# Calcul des maximums (Indispensable pour valider la classe 2)
max_global = np.max(global_rmsd_vals)
avg_local = np.mean(local_rmsd_vals)

# Calcul des maximums (Indispensable pour valider la classe 2)
max_global = np.max(global_rmsd_vals)
max_local = np.max(local_rmsd_vals)

print("\nAnalyse terminée !")
print(f"RMSD Global -> Moyen : {avg_global:.4f} Å | Max : {max_global:.4f} Å")
print(f"RMSD Local -> Moyen : {avg_local:.4f} Å | Max : {max_local:.4f} Å")

In [ ]:
import pandas as pd
import gc

print("Fusion des liaison des interactions BindingDB aux variants pathogènes :")
# Chargement du réservoir d'interactions (1.67M paires WT)
df_bdb = pd.read_parquet("/kaggle/input/datasets/jakeadam68/bindingdb-onco-admet/bindingdb_onco_diamond_sota_fullseq.parquet")
# Chargement de ta bibliothèque de variants (5 920 mutations d'élite)
df_variants = pd.read_parquet("/kaggle/input/datasets/rayanchakibidris/cosmic-genomescreen-v103/pathogenic_sequences_sota_final.parquet")
# La Fusion (Inner Join sur uniprot_id)
# pour chaque ligand d'une protéine, on va créer autant de lignes qu'il y a de mutations pour cette protéine.
df_final_sota = pd.merge(
    df_bdb[['Ligand SMILES', 'uniprot_id', 'pAff', 'aff_type', 'gene_symbol', 'is_oncogenic', 'PubChem CID']], 
    df_variants[['uniprot_id', 'mutation_hgvsp', 'mutated_sequence', 'wt_sequence', 'ClinicalSignificance', 'is_confirmed_pathogenic']],
    on='uniprot_id',
    how='inner'
)
# Création d'une clé unique pour l'IA (Pair ID)
# Utile pour retrouver facilement les embeddings ESM-2 plus tard
df_final_sota['variant_id'] = df_final_sota['uniprot_id'] + "_" + df_final_sota['mutation_hgvsp']
# Bilan des statistiques
print(f"le nombre de paires [Ligand - Protéine Mutée] est : {len(df_final_sota):,} paires")
print(f"le nombre de gènes représentés est : {df_final_sota['gene_symbol'].nunique()} gènes")
print(f"le nombre de mutations uniques est : {df_final_sota['variant_id'].nunique()} mutations")
# Sauvegarde du Dataset d'Entraînement/Test Final
output_name = "bindingdb_wildtype_mutated_fullseq.parquet"
df_final_sota.to_parquet(output_name, index=False, compression='snappy')
print(f"le dataset est sauvegardé au chemin spécifié : {output_name}")
# Nettoyage mémoire immédiat
del df_bdb, df_variants
gc.collect()

In [ ]:
import pandas as pd

# Chemin du fichier généré par ta fusion précédente
file_path = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/bindingdb_wildtype_mutated_fullseq.parquet"

print(f"Lecture du fichier : {file_path}")
df_check = pd.read_parquet(file_path)

print(f"Apercu du dataset ({len(df_check):,} lignes)")

# 1. Liste des colonnes
print(f"\nListe des {len(df_check.columns)} colonnes :")
for i, col in enumerate(df_check.columns):
    print(f"  {i+1:>2}. {col}")

# 2. Types de données et valeurs manquantes
print("\nAnalyse des types et des NaNs :")
print(df_check.info())

# 3. Affichage des 5 premières lignes pour voir le contenu réel
print("\nÉchantillon des données :")
display(df_check.head())

# 4. Vérification statistique des colonnes de labels potentiels
print("\nStatistiques des colonnes numériques :")
display(df_check.describe())

In [ ]:
import pandas as pd
import numpy as np
import re
import gc
import torch

# 1. Préparation des cibles depuis ton fichier de variants
df_sota = pd.read_parquet("/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/bindingdb_wildtype_mutated_fullseq.parquet")
target_mutants = set(df_sota['mutated_sequence'].unique())
target_smiles = set(df_sota['Ligand SMILES'].unique())

print(f"Recherche de mesures pour {len(target_mutants):,} séquences mutées uniques...")

# 2. La fonction de nettoyage 
def get_best_affinity_mirrored(row):
    # Ta liste de colonnes
    for col in ['Ki (nM)', 'IC50 (nM)', 'Kd (nM)', 'EC50 (nM)']:
        val = row[col]
        if pd.notna(val):
            val_str = str(val).strip().upper()
            # Tes exclusions
            if val_str in ['ND', 'N.D.', 'N/A', 'NONE', 'NOT DETERMINED', '-', 'NA']:
                continue
            # Ton nettoyage de symboles
            val_str = re.sub(r'[<>≈~]', '', val_str)
            val_str = val_str.replace(' ', '').replace(',', '')
            val_str = val_str.replace('NM', '').replace('NМ', '')
            # Ta gestion des intervalles (Moyenne Géométrique)
            if '-' in val_str:
                try:
                    parts = [float(p) for p in val_str.split('-') if p.strip()]
                    if len(parts) == 2:
                        num_val = (parts[0] * parts[1]) ** 0.5
                        return num_val, col
                    elif len(parts) == 1:
                        num_val = parts[0]
                except: continue

            # Ta conversion standard et ton seuil de 5,000,000
            try:
                num_val = float(val_str)
                if 0 < num_val <= 5_000_000:
                    return num_val, col
            except: continue
    return np.nan, None

# 3. Scan du fichier de 8 Go par morceaux
file_path = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/BindingDB_All_202602_tsv/BindingDB_All.tsv"
cols = [
    'Ligand SMILES', 'Target Source Organism According to Curator or DataSource',
    'BindingDB Target Chain Sequence 1', 'Ki (nM)', 'IC50 (nM)', 'Kd (nM)', 'EC50 (nM)'
]
found_mt_list = []
reader = pd.read_csv(file_path, sep='\t', usecols=cols, chunksize=250000, low_memory=False, on_bad_lines='skip')

for i, chunk in enumerate(reader):
    # Application des filtres initiaux (Humain + SMILES + Séquence mutée cible)
    human_mask = chunk['Target Source Organism According to Curator or DataSource'].str.contains(r'human|homo sapiens', case=False, na=False)
    smiles_mask = chunk['Ligand SMILES'].isin(target_smiles)
    seq_mask = chunk['BindingDB Target Chain Sequence 1'].isin(target_mutants)
    
    found = chunk[human_mask & smiles_mask & seq_mask].copy()
    
    if not found.empty:
        # Application de la fonction get_best_affinity
        temp = found[['Ki (nM)', 'IC50 (nM)', 'Kd (nM)', 'EC50 (nM)']].apply(get_best_affinity_mirrored, axis=1, result_type='expand')
        found[['affinity_nM', 'aff_type']] = temp
        found = found[found['affinity_nM'] > 0].dropna(subset=['affinity_nM'])
        found_mt_list.append(found)
        
    if (i+1) % 4 == 0: print(f" ⏳ Lu : {(i+1)*250000:,} lignes", end='\r')

df_all_found_mt = pd.concat(found_mt_list)
del found_mt_list
gc.collect()

# 4. Logique d'agrégation sélective avec priorité
priority_map = {'Ki (nM)': 4, 'Kd (nM)': 3, 'IC50 (nM)': 2, 'EC50 (nM)': 1}
df_all_found_mt['priority'] = df_all_found_mt['aff_type'].map(priority_map).fillna(0)
# Calcul du pAff avant l'agrégation 
df_all_found_mt['pAff_MT'] = -np.log10(df_all_found_mt['affinity_nM'] * 1e-9)

# on groupe par (SMILES, Séquence) pour trouver la meilleure priorité
idx_best = df_all_found_mt.groupby(['Ligand SMILES', 'BindingDB Target Chain Sequence 1'])['priority'].transform('max')
df_best_mt = df_all_found_mt[df_all_found_mt['priority'] == idx_best].copy()

# Exécution de l'agrégation fianle (en appliquant la médiane)
df_mt_clean = df_best_mt.groupby(['Ligand SMILES', 'BindingDB Target Chain Sequence 1']).agg({
    'affinity_nM': 'median',
    'pAff_MT': 'median'
}).reset_index()

# Filtrage finale sur le pAff_MT [3, 11]
df_mt_clean = df_mt_clean[(df_mt_clean['pAff_MT'] >= 3.0) & (df_mt_clean['pAff_MT'] <= 11.0)].copy()

# 6. Fusion finale et calcul du delta
df_final_truth = pd.merge(
    df_sota,
    df_mt_clean,
    left_on=['Ligand SMILES', 'mutated_sequence'],
    right_on=['Ligand SMILES', 'BindingDB Target Chain Sequence 1'],
    how='inner'
)
df_final_truth['delta_pAff'] = df_final_truth['pAff_MT'] - df_final_truth['pAff']
df_final_truth.to_parquet("gold_standard_experimental_16k.parquet", index=False)
print(f"\nTerminé ! le dataset est 100% symétrique.")
print(f"le nombre de paires conservées est : {len(df_final_truth):,}")

In [ ]:
import pandas as pd

# on charge le résultat du scan
df_gold = df_final_truth 

print("🧪 Analyse du set contenant le delta pAff calculé à partir du pAff des séquences mt et du pAff des séquences wt:")
print(f"Nombre de paires conservées : {len(df_gold):,}")
print(f"Nombre de ligands uniques   : {df_gold['Ligand SMILES'].nunique():,}")
print(f"Nombre de gènes uniques     : {df_gold['gene_symbol'].nunique():,}")

# Vérification du Delta pAff
print("\n📊 Statistiques du delta pAff :")
print(df_gold['delta_pAff'].describe())

# Vérification de l'étanchéité
check_nan = df_gold['delta_pAff'].isna().sum()
print(f"\n🚫 Valeurs manquantes dans le label : {check_nan}")

if check_nan == 0:
    print("Le dataset est prêt pour l'entraînement.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Chargement du fichier final
file_path = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/gold_standard_experimental_16k.parquet"
df_gold = pd.read_parquet(file_path)

print("Analyse des résultats expérimental")
print("="*70)

# 2. Statistiques Globales de Volume
stats_global = {
    "Total Paires [Ligand-Mutation]": len(df_gold),
    "Ligands Uniques (Chimie)": df_gold['Ligand SMILES'].nunique(),
    "Mutations Uniques (Biologie)": df_gold['variant_id'].nunique(),
    "Gènes Uniques (Cibles)": df_gold['gene_symbol'].nunique()
}

for label, val in stats_global.items():
    print(f"{label:<35} : {val:,}")

# 3. Analyse par Gène 
print("\nTop des gènes représentés :")
gene_breakdown = df_gold.groupby('gene_symbol').agg(
    n_paires=('variant_id', 'count'),
    n_mutations=('variant_id', 'nunique'),
    n_ligands=('Ligand SMILES', 'nunique'),
    delta_moyen=('delta_pAff', 'mean'),
    pAff_WT_moyen=('pAff', 'mean')
).sort_values(by='n_paires', ascending=False)

display(gene_breakdown)

# 4. Statistiques de la Cible (delta_pAff)
print("\n📊 Distribution de delta_pAff :")
desc = df_gold['delta_pAff'].describe()
print(desc)

# 5. Visualisation pour l'article
plt.figure(figsize=(15, 5))

# Histogramme du Delta pAff
plt.subplot(1, 2, 1)
sns.histplot(df_gold['delta_pAff'], bins=50, color='gold', kde=True)
plt.title("Distribution du delta pAff (Expérimental)")
plt.xlabel("Changement d'affinité (MT - WT)")

# Boxplot par Gène (Top 10)
plt.subplot(1, 2, 2)
top_10_genes = gene_breakdown.index[:10]
sns.boxplot(x='gene_symbol', y='delta_pAff', data=df_gold[df_gold['gene_symbol'].isin(top_10_genes)], palette='viridis')
plt.xticks(rotation=45)
plt.title("Impact des mutations par Gène")

plt.tight_layout()
plt.savefig("distribution_variation_affinité_wt_mt.png", dpi=300, bbox_inches='tight')
plt.show()

# 6. Vérification de l'étanchéité pour le Q1
print("\nAudit de rigueur :")
range_check = (df_gold['delta_pAff'].min() > -10) and (df_gold['delta_pAff'].max() < 10)
nan_check = df_gold['delta_pAff'].isna().sum() == 0
print(f"  - Absence de NaNs dans la cible : {'Oui' if nan_check else 'Non'}")
print(f"  - Valeurs dans la plage réaliste [-10, 10] : {' Oui' if range_check else 'Non'}")

In [ ]:
import pandas as pd
import numpy as np

print("🔍 Audit du fichier Gold Standard...")

# 1. Chargement du fichier
path_gold = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/gold_standard_experimental_16k.parquet"
try:
    df_gold_audit = pd.read_parquet(path_gold)
    print("le fichier est chargé avec succès.")
except Exception as e:
    print(f"Erreur lors du chargement : {e}")
    # on s'arrête ici si le fichier est inaccessible
    raise

# 2. Vérification de la structure
print("\nStructure du DataFrame :")
print(f"Dimensions : {df_gold_audit.shape}")
print(f"Colonnes présentes : {df_gold_audit.columns.tolist()}")

# 3. Vérification de l'unicité (Hypothèse fondamentale)
# On vérifie si chaque variant_id n'apparaît qu'une seule fois
duplicates = df_gold_audit['variant_id'].duplicated().sum()
is_unique = duplicates == 0
print(f"\nUnicité des variant_id : {'Unique' if is_unique else 'Doublons Détéctés'}")
print(f"Nombre de doublons : {duplicates}")

# 4. Vérification des valeurs manquantes (NaN)
nan_paff = df_gold_audit['delta_pAff'].isna().sum()
print(f"Valeurs manquantes dans delta_pAff : {nan_paff} ({ (nan_paff/len(df_gold_audit))*100:.2f}%)")

# 5. Analyse statistique des valeurs de delta_pAff
print("\nStatistiques de delta_pAff :")
print(df_gold_audit['delta_pAff'].describe())

# 6. Calcul et vérification du Fold-Change maximum
# FC = 10^(-delta_pAff)
max_fc = 10**(-df_gold_audit['delta_pAff'].min())
print(f"\nFold-Change Maximum détecté : {max_fc:.4f}")

# 7. Aperçu des données
print("\nVoici un aperçu des 5 premières lignes :")
print(df_gold_audit.head())

In [ ]:
import pandas as pd
import numpy as np

print("Analyse comparative approfondie des datasets")

# Chemins des fichiers sur Kaggle
path_silver = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/bindingdb_wildtype_mutated_fullseq.parquet"
path_gold = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/gold_standard_experimental_16k.parquet"

# 1. Chargement des données
print("1. Chargement des fichiers...")
df_silver = pd.read_parquet(path_silver)
df_gold = pd.read_parquet(path_gold)

# 2. Analyse quantitative de base
print("2. Analyse Quantitative de base")
print(f"Dataset global (Silver)    : {len(df_silver):,} lignes")
print(f"Dataset Expérimental (Gold) : {len(df_gold):,} lignes")
print()

# 3. Unicité Biologique et Chimique (Unique counts)
print("3. Unicité Biologique et chimique")
print("[Dataset Global (SILVER)]")
print(f"➜ Mutations uniques (variant_id) : {df_silver['variant_id'].nunique():,}")
print(f"➜ Ligands uniques (SMILES)       : {df_silver['Ligand SMILES'].nunique():,}")
print("\n[Dataset Experimental (GOLD)]")
print(f"➜ Mutations uniques (variant_id) : {df_gold['variant_id'].nunique():,}")
print(f"➜ Ligands uniques (SMILES)       : {df_gold['Ligand SMILES'].nunique():,}")

# 4. Clé Unique de Liaison (Primary Key Check)
print("4. Diagnostic de la clé unique (variant_id + Ligand)")
# On vérifie si la combinaison variant_id + Ligand SMILES n'apparaît qu'une seule fois dans le Gold
gold_pair_dups = df_gold.duplicated(subset=['variant_id', 'Ligand SMILES']).sum()
print(f"   - Doublons exacts de la paire [variant_id + Ligand] dans le Gold : {gold_pair_dups}")
if gold_pair_dups == 0:
    print("Chaque ligne du fichier Éxpérimental de 16k est bien une interaction unique [Mutation <-> Médicament].")
else:
    print("Il y a des doublons d'interaction dans le fichier Éxpérimental.")

# 5. L'Audit de Réplication (L'explication des 149k)
# on récupère la liste des mutations uniques présentes dans le fichier éxpérimental 
gold_variants_set = set(df_gold['variant_id'].unique())

# on regarde combien de lignes du dataset global possèdent l'un de ces variant_id
silver_rows_matching_gold_variants = df_silver[df_silver['variant_id'].isin(gold_variants_set)]
print(f"le nombre de lignes dans le fichier global qui partagent un 'variant_id' avec le fichier éxpérimental : {len(silver_rows_matching_gold_variants):,} lignes")
print(f"le nombre de gènes uniques représentés dans ces lignes : {silver_rows_matching_gold_variants['gene_symbol'].nunique()}")

# 6. Intersection réelle (Combien de paires exactes du fichier éxpérimental sont dans le fichier global)
print("\n6. Intersection réel [variant_id + Ligand SMILES]")
intersection = pd.merge(
    df_silver[['variant_id', 'Ligand SMILES']], 
    df_gold[['variant_id', 'Ligand SMILES', 'delta_pAff']], 
    on=['variant_id', 'Ligand SMILES'], 
    how='inner'
)
print(f"le nombre de paires exactes [Mutation <-> Ligand] communes entre le fichier global et le fichier éxpérimental : {len(intersection):,} paires.")
print(f"le pourcentage de paires du fichier éxpérimental retrouvé à l'identique dans le fichier global : {len(intersection)/len(df_gold)*100:.2f}%\n")


## **Partie Génération 3D des Ligands - RDKit**

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # on supprime les messages warning de RDKit
from tqdm.auto import tqdm

# on charge le dataset BindingDB
df = pd.read_parquet("/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/bindingdb_wildtype_mutated_fullseq.parquet")
print(f"Avant d'avoir vérifier les Ligands SMILES on avait : {len(df):,} lignes\n")

# on crée une fonction qui permet de vérifier la validité des SMILES
def is_valid_smiles(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        return False, "vide ou non-string"
    try:
        mol = Chem.MolFromSmiles(smiles, sanitize=True)
        return mol is not None, "valide" if mol else "invalide (RDKit échoue)"
    except:
        return False, "exception RDKit"

# on procéde à la vérification complète des Ligand SMILES 
results = []
for smiles in tqdm(df['Ligand SMILES'], desc="Validation SMILES"):
    valid, reason = is_valid_smiles(smiles)
    results.append((valid, reason))

df_check = pd.DataFrame(results, columns=['valid', 'reason'])
df_check.index = df.index

# on calcule les statistiques des Ligand SMILES 
total = len(df_check)
valid_count = df_check['valid'].sum()

# Affichage des statistiques détaillées
invalid_count = total - valid_count
print("="*50)
print("📊 Bilan de validation des smiles :")
print("="*50)
print(f"le nombre totale de lignes analysées est : {total:,}")
print(f"le nombre de smiles valides est : {valid_count:,} avec un taux : ({valid_count/total*100:.2f}%)")
print(f"le nombre de smiles invalides est : {invalid_count:,} avec un taux : ({invalid_count/total*100:.2f}%)")
print("\n🔍 Raisons des échecs :")
print(df_check['reason'].value_counts())
# on ne garde que les lignes valides
df_final_clean = df[df_check['valid']].copy()
print(f"le nombre de lignes conservées est : {len(df_final_clean):,}")

In [ ]:
import torch
import numpy as np
import pandas as pd
import pickle
import traceback
import os
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Crippen, QED, rdMolDescriptors
from rdkit import RDLogger
from tqdm.auto import tqdm
from joblib import Parallel, delayed
import gc
# Désactivation des logs RDKit pour une exécution propre
RDLogger.DisableLog('rdApp.*') 

# on crée une fonction pour la génération 3D des ligands
def get_sota_3d_conformer(smiles, max_precision=True):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None, None, None, None
        
        # Ajout des Hydrogènes (indispensable pour une 3D et des charges correctes)
        mol = Chem.AddHs(mol)
        
        # 1. Génération de la géométrie 3D
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        params.maxIterations = 200 
        params.useRandomCoords = False
        
        res = AllChem.EmbedMolecule(mol, params)
        if res == -1:
            # Fallback en cas d'échec (molécules complexes)
            res = AllChem.EmbedMolecule(mol, useRandomCoords=True, randomSeed=42)
        
        if res == -1:
            return None, None, None, None
            
        # 2. Optimisation de la structure (MMFF94s)
        if max_precision:
            if AllChem.MMFFGetMoleculeProperties(mol) is not None:
                AllChem.MMFFOptimizeMolecule(mol, mmffVariant='MMFF94s', maxIters=200)
        
        # 3. Calcul des propriétés électroniques et physico-chimiques
        AllChem.ComputeGasteigerCharges(mol)
        pt = Chem.GetPeriodicTable()
        logp_contribs = rdMolDescriptors._CalcCrippenContribs(mol)
        
        # Features Globales (15 dimensions)
        logp = Crippen.MolLogP(mol)
        tpsa = Descriptors.TPSA(mol)
        mw = Descriptors.MolWt(mol)
        qed_val = QED.qed(mol)
        rot_bonds = Descriptors.NumRotatableBonds(mol)
        f_csp3 = Descriptors.FractionCSP3(mol)
        
        global_features = torch.tensor([
            logp / 10.0,            # 1. Lipophilie
            tpsa / 200.0,           # 2. Surface polaire
            mw / 1000.0,            # 3. Masse molaire
            qed_val,                # 4. Médicabilité (QED)
            rot_bonds / 20.0,       # 5. Flexibilité
            f_csp3,                  # 6. Caractère 3D (Fraction CSP3)
            float(Descriptors.NumHAcceptors(mol)) / 15.0,      # 7. Accepteurs H
            float(Descriptors.NumHDonors(mol)) / 10.0,         # 8. Donneurs H
            float(Descriptors.NumAromaticRings(mol)) / 10.0,   # 9. Cycles aromatiques
            float(Descriptors.NumAliphaticRings(mol)) / 10.0,  # 10. Cycles aliphatiques
            float(Descriptors.NumHeteroatoms(mol)) / 100.0,     # 11. Atomes lourds
            Descriptors.LabuteASA(mol) / 500.0,                # 12. Surface accessible 
            Descriptors.MolMR(mol) / 150.0,                    # 13. Réfractivité molaire
            float(rdMolDescriptors.CalcNumSpiroAtoms(mol)) / 5.0,      # 14. Atomes Spiro
            float(rdMolDescriptors.CalcNumBridgeheadAtoms(mol)) / 5.0  # 15. Atomes de pont
        ], dtype=torch.float32) 
        
        # 4. Extraction des coordonnées et features atomiques
        conf = mol.GetConformer(0)
        coords = conf.GetPositions()
        atoms_feat = []
        # Mapping atomique 
        atom_map = {1:0, 5:1, 6:2, 7:3, 8:4, 9:5, 14:6, 15:7, 16:8, 17:9, 35:10, 53:11}
        
        for atom in mol.GetAtoms():
            z = atom.GetAtomicNum()
            logp_idx = atom.GetIdx()

             # Récupération de la charge de Gasteiger
            try:
                q = float(atom.GetDoubleProp('_GasteigerCharge'))
                q = 0.0 if np.isnan(q) or np.isinf(q) else q
            except: q = 0.0
            
            # Vecteur de caractéristiques atomiques (23 dimensions)
            # Type, Degré, Charge formelle, Aromaticité, Dans un cycle, etc...
            a_feat = [
                float(atom_map.get(z, 12)),                    # 1. Identité (Index type)
                float(atom.GetMass() / 100.0),                 # 2. Masse atomique
                float(pt.GetRvdw(z)),                          # 3. Rayon de Van der Waals
                float(pt.GetRcovalent(z)),                     # 4. Rayon covalent
                float(atom.GetDegree()),                       # 5. Connectivité (Nombre de voisins)
                float(atom.GetTotalValence()),                 # 6. Valence totale
                float(atom.GetFormalCharge()),                 # 7. Charge nette
                float(1.0 if atom.GetIsAromatic() else 0.0),   # 8. Aromaticité (Nuage Pi)
                float(1.0 if atom.IsInRing() else 0.0),        # 9. Rigidité (Appartenance à un cycle)
                float(int(atom.GetHybridization())),           # 10. Géométrie (sp, sp2, sp3)
                float(atom.GetTotalNumHs()),                   # 11. Saturation (Nombre de H liés)
                # 12. Potentiel Accepteur de liaison H (N, O, F)
                float(1.0 if z in [7, 8, 9] else 0.0),         
                # 13. Potentiel Donneur de liaison H (N, O liés à au moins un H)
                float(1.0 if z in [7, 8] and atom.GetTotalNumHs() > 0 else 0.0),
                float(int(atom.GetChiralTag())),               # 14. Chiralité (Configuration spatiale)
                float(logp_contribs[logp_idx][0]),             # 15. Contrib locale LogP 
                float(logp_contribs[logp_idx][1]),             # 16. Contrib locale Réfractivité
                float(atom.GetExplicitValence()),              # 17. Valence (État chimique)
                float(atom.GetImplicitValence()),              # 18. Valence implicite
                float(atom.GetNumRadicalElectrons()),          # 19. Électrons radicaux
                float(pt.GetNOuterElecs(z)),                   # 20. Électrons de valence (Couche externe)
                float(pt.GetMostCommonIsotope(z)) / 100.0,     # 21. Isotope le plus commun
                float(pt.GetAbundanceForIsotope(z, 0)),        # 22. Abondance isotopique
                q                                              # 23. Charge de Gasteiger
            ]
            atoms_feat.append(a_feat)
       
        return (
            torch.tensor(coords, dtype=torch.float32),
            torch.tensor(atoms_feat, dtype=torch.float32), # Changé en float32 pour le multi-feature
            torch.tensor([a[-1] for a in atoms_feat], dtype=torch.float32), # Tenseur charges
            global_features
        )
    except Exception as e:
               print(f"\n🚨 Crash détecté pour le smiles : {smiles}")
               print(f"Type d'erreur : {type(e).__name__}")
               print(f"Message : {e}")
               print("Traceback complet :")
               traceback.print_exc()
               return None, None, None, None

INPUT_PARQUET = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/bindingdb_wildtype_mutated_fullseq.parquet"
output_file = "ligands_3d_rdkit_database.pkl"

print("📖 Chargement du dataset pour extraction des ligands...")
# Chargement optimisé (lecture d'une seule colonne)
df_smiles = pd.read_parquet(INPUT_PARQUET, columns=['Ligand SMILES'])
unique_smiles = df_smiles['Ligand SMILES'].unique().tolist()
unique_smiles = [s for s in unique_smiles if s is not None and s != '']

print(f"🧪 {len(unique_smiles):,} molécules uniques à traiter en 3D.")

# on crée une fonction de transition pour le multiprocessing
def process_single_ligand(smiles):
    from rdkit import rdBase
    rdBase.DisableLog('rdApp.*') # Silence total des warnings RDKit C++
    # on appelle ta fonction de génération 3D des ligands
    res_rdkit = get_sota_3d_conformer(smiles, max_precision=True)
    if res_rdkit[0] is not None:
        return {
            'coords': res_rdkit[0],           # Coordonnées (N, 3)
            'atom_features': res_rdkit[1],    # caractéristiques des atomes
            'global_rdkit': res_rdkit[3],     # LogP, TPSA, MW
        }
    return None

print(f"🧠 Génération 3D des ligands en cours :")
final_ligand_db = {}
SAVE_STEP = 20000

# on définit le nombre de cœurs (n_jobs=-1 utilise tout)
n_jobs = -1

for i in range(0, len(unique_smiles), SAVE_STEP):
    batch_smiles = unique_smiles[i:i+SAVE_STEP]
    
    
    # on calcul en parallèle avec la barre tqdm intégré
    # on enveloppe le générateur dans tqdm pour voir la progression réelle
    results = Parallel(n_jobs=n_jobs)(
        delayed(process_single_ligand)(s) 
        for s in tqdm(batch_smiles, 
                         desc=f"Lot {i//SAVE_STEP + 1}/{len(unique_smiles)//SAVE_STEP + 1}")
    )
    
    # Fusion dans le dictionnaire final
    for j, s in enumerate(batch_smiles):
        if results[j] is not None:
            final_ligand_db[s] = results[j]
            
    # Sauvegarde du progrès
    with open(output_file, "wb") as f:
        pickle.dump(final_ligand_db, f)
    
    print(f" ✅ Lot {i//SAVE_STEP + 1} sauvegardé | Total en base : {len(final_ligand_db):,}")
    gc.collect()

print(f"\n✨ la génération 3D des ligands est terminé ! Fichier : {output_file}")
    

In [ ]:
import pickle
import numpy as np
import torch
import os

file_path = "ligands_3d_rdkit_database.pkl"

if not os.path.exists(file_path):
    print(f"❌ Erreur : Le fichier {file_path} est introuvable.")
else:
    print(f"📖 Lecture du fichier : {file_path} ({os.path.getsize(file_path) / 1024**3:.2f} Go)")
    with open(file_path, "rb") as f:
        ligand_db = pickle.load(f)

    print(f"le nombre total de ligands uniques est : {len(ligand_db):,}")

    sample_smiles = list(ligand_db.keys())[0]
    sample_data = ligand_db[sample_smiles]

    print(f"🔍 Analyse de l'échantillon : {sample_smiles}")

    if sample_data is not None:
        components = {
            'coords': 'Coordonnées 3D (N, 3)',
            'atom_features': 'Features Atomiques (N, 23)', 
            'global_rdkit': 'Features Globales RDKit (15,)',
        }

        for key, description in components.items():
            val = sample_data.get(key)
            if val is not None:
                # on convertit le shape en string pour éviter l'erreur de formatage
                sh = str(list(val.shape) if hasattr(val, 'shape') else np.array(val).shape)
                print(f"✅ {key:<15} : présent | shape: {sh:<15} | ({description})")
            else:
                print(f"❌ {key:<15} : manquant dans cet échantillon.")
        c_array = sample_data['coords'].numpy() if torch.is_tensor(sample_data['coords']) else np.array(sample_data['coords'])
        
    else:
        print("❌ Données corrompues pour cet échantillon.")

In [ ]:
import pickle
import numpy as np
import torch
from rdkit import Chem
import py3Dmol
from IPython.display import HTML  

# Charger la base de données
with open("/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/ligands_3d_rdkit_database.pkl", "rb") as f:
    ligand_db = pickle.load(f)

def visualize_ligand_3d(smiles, style='stick', height=500, width=600, bg_color='white', show_info=True):
    """Visualise un ligand 3D avec style avancé et couleurs"""
    
    if smiles not in ligand_db:
        return HTML(f"<p>❌ SMILES introuvable</p>")
    
    ligand_data = ligand_db[smiles]
    
    # Reconstruire la molécule
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return HTML(f"<p>❌ SMILES invalide</p>")
    
    mol = Chem.AddHs(mol)
    
    # Récupérer et assigner les coordonnées
    coords = ligand_data['coords']
    if torch.is_tensor(coords):
        coords = coords.numpy()
    
    conf = Chem.Conformer(mol.GetNumAtoms())
    for i in range(mol.GetNumAtoms()):
        if i < len(coords):
            x, y, z = coords[i]
            conf.SetAtomPosition(i, (float(x), float(y), float(z)))
    
    mol.RemoveAllConformers()
    mol.AddConformer(conf, assignId=True)
    
    # Générer SDF
    sdf_block = Chem.MolToMolBlock(mol)
    
    view = py3Dmol.view(width=width, height=height)
    view.addModel(sdf_block, "sdf")
    
    if style == 'line':
        # Lignes avec atomes colorés
        view.setStyle({}, {'line': {
            'colorscheme': 'Jmol',
            'linewidth': 2.0
        }})
        view.setStyle({}, {'sphere': {
            'scale': 0.15,
            'colorscheme': 'Jmol'
        }})
    
    elif style == 'stick':
        # Bâtons colorés CPK
        view.setStyle({}, {'stick': {
            'colorscheme': 'Jmol',
            'radius': 0.2
        }})
    
    elif style == 'ball+stick':
        # Boules et bâtons
        view.setStyle({}, {'sphere': {
            'scale': 0.3,
            'colorscheme': 'Jmol'
        }})
        view.setStyle({}, {'stick': {
            'colorscheme': 'Jmol',
            'radius': 0.15
        }})
    
    elif style == 'sphere':
        # Boules colorées
        view.setStyle({}, {'sphere': {
            'scale': 0.5,
            'colorscheme': 'Jmol'
        }})
    
    else:
        # Par défaut
        view.setStyle({}, {style: {'colorscheme': 'Jmol'}})
    
    # Couleur de fond
    bg_colors = {
        'white': 'white',
        'black': '#000000',
        'lightgray': '#e8e8e8'
    }
    
    view.setBackgroundColor(bg_colors.get(bg_color, 'white'))
    view.zoomTo()
    view.render()
    
    # Informations du Ligand
    if show_info:
        global_features = ligand_data['global_rdkit']
        
        info_html = f"""
        <div style="background-color: #f0f0f0; padding: 10px; border-radius: 5px; margin-top: 10px;">
            <b>📊 Informations :</b><br>
            <small>
            • Atomes: <b>{mol.GetNumAtoms()}</b><br>
            • MW: <b>{global_features[2].item()*1000:.1f}</b> g/mol<br>
            • LogP: <b>{global_features[0].item()*10:.2f}</b><br>
            • TPSA: <b>{global_features[1].item()*200:.1f}</b> &Aring;<sup>2</sup><br>
            • QED: <b>{global_features[3].item():.2f}</b>
            </small>
        </div>
        """
    else:
        info_html = ""
    
    return HTML(view._make_html() + info_html)

import random

print("Visualisation de 3 Ligands aléatoires :")

random_smiles_list = random.sample(list(ligand_db.keys()), 3)

for idx, random_smiles in enumerate(random_smiles_list, 1):
    print(f"\n🔬 Ligand aléatoire {idx}/3 : {random_smiles[:80]}")
    display(visualize_ligand_3d(random_smiles, style='ball+stick', bg_color='white', show_info=True))

## **Partie Préparation des données** 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData, Data, Dataset
from torch_cluster import radius_graph, knn_graph
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import numpy as np
import pickle
import os
import re
import warnings
import gc
from tqdm.auto import tqdm

# Configuration
PATH_PREFIX       = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/"
RANDOM_SEED       = 42

# Alphabet des acides aminés pour le One-Hot Encoding
AA_CHARS  = "ACDEFGHIKLMNPQRSTVWYX"
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_CHARS)}
AA_VOCAB  = len(AA_CHARS)

print("🚀 Phase 1 : Chargement et Configuration")

def get_win_params(seq_len, hgvsp_str, max_len=256):
    match = re.search(r'p\.[A-Z][a-z]{2}(\d+)|p\.[A-Z](\d+)', str(hgvsp_str))
    pos = int(match.group(1) if match.group(1) else match.group(2)) if match else seq_len // 2
    center = pos - 1
    if seq_len <= max_len: return 0, seq_len, center
    s = max(0, center - max_len // 2)
    e = s + max_len
    if e > seq_len:
        e = seq_len
        s = max(0, e - max_len)
    return int(s), int(e), int(center - s)

def encode_sequence(sequence: str, mut_idx: int, plddt: torch.Tensor, metrics_vector: torch.Tensor = None, esm_vector: torch.Tensor = None) -> torch.Tensor:
    """Version optimisée sans boucle for"""
    L = len(sequence)
    if plddt.shape[0] != L:
        raise ValueError(f"Mismatch de taille : la séquence fait {L} mais le pLDDT fait {plddt.shape[0]}. "
                         f"Vérifiez le découpage de la fenêtre dans _build_data.")
    # 1. on convertit la séquence en indices en une seule fois
    indices = torch.tensor([AA_TO_IDX.get(aa.upper(), AA_TO_IDX['X']) for aa in sequence], dtype=torch.long)
    # 2. One-hot encoding vectorisé (très rapide)
    features = F.one_hot(indices, num_classes=AA_VOCAB).float()
    # 3. Ajout du pLDDT (on ne slice plus, on utilise le tensor tel quel)
    plddt_col = plddt.unsqueeze(1)
    # 4. Ajout du flag de mutation (colonne 22)
    mut_col = torch.zeros((L, 1), dtype=torch.float32)
    if 0 <= mut_idx < L:
        mut_col[mut_idx] = 1.0

    if metrics_vector is not None:
        num_metrics = metrics_vector.shape[0]
        metrics_block = torch.zeros((L, num_metrics), dtype=torch.float32)
        if 0 <= mut_idx < L:
            metrics_block[mut_idx] = metrics_vector
    else:
        # Si aucune métrique n'est fournie (ex: début de Phase 1), on met des zéros
        metrics_block = torch.zeros((L, 6), dtype=torch.float32)


    if esm_vector is not None:
        # on s'assure que l'embedding a la bonne taille (L, 1280)
        if esm_vector.shape[0] != L:
            # on ajuste la taille pour correspondre à L (padding ou truncation)
            if esm_vector.shape[0] > L:
                esm_vector = esm_vector[:L, :]
            else:
                pad = torch.zeros((L - esm_vector.shape[0], esm_vector.shape[1]), dtype=torch.float32)
                esm_vector = torch.cat([esm_vector, pad], dim=0)
    else:
        # Si pas d'ESM-2, on met des zéros
        esm_vector = torch.zeros((L, 2560), dtype=torch.float32)
        
    return torch.cat([features, plddt_col, mut_col, metrics_block, esm_vector], dim=-1)

def compute_biophysical_edges(data, radius_prot=8.0, radius_bind=5.0, k_prot=6):
    # 1. Protein -> Protein : on fusionne Radius et kNN plus efficacement
    # on utilise torch.cat puis unique pour éviter les doublons
    edge_p_p = torch.cat([
        radius_graph(data['protein'].pos, r=radius_prot, loop=False),
        knn_graph(data['protein'].pos, k=k_prot, loop=False)
    ], dim=1).unique(dim=0)
    data['protein', 'interacts', 'protein'].edge_index = edge_p_p

    # 2. Ligand -> Ligand : Graphe complet optimisé
    num_l = data['ligand'].pos.size(0)
    adj = torch.ones((num_l, num_l), device=data['ligand'].pos.device)
    data['ligand', 'interacts', 'ligand'].edge_index = adj.nonzero().t()

    # 3. Protein <-> Ligand : Bipartite Radius optimisé
    dist_mat = torch.cdist(data['protein'].pos, data['ligand'].pos)
    edge_p_l = (dist_mat < radius_bind).nonzero().t()
    data['protein', 'binds', 'ligand'].edge_index = edge_p_l
    data['ligand', 'binds', 'protein'].edge_index = edge_p_l[:, [1, 0]]

    return data
    
def extract_aa_from_hgvsp(hgvsp_str):
    """
    Extrait le résidu sauvage (WT) et le résidu muté (MT) d'une chaîne HGVS.
    Format supporté : 'ENSP...:p.Pro29Ser' ou 'p.Pro29Ser'
    """
    hgvsp_str = str(hgvsp_str)
    match = re.search(r'p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})', hgvsp_str)
    if match:
        wt_full = match.group(1)
        mt_full = match.group(3)
        aa_map = {
            'Ala':'A', 'Arg':'R', 'Asn':'N', 'Asp':'D', 'Cys':'C', 
            'Glu':'E', 'Gln':'Q', 'Gly':'G', 'His':'H', 'Ile':'I', 
            'Leu':'L', 'Lys':'K', 'Met':'M', 'Phe':'F', 'Pro':'P', 
            'Ser':'S', 'Thr':'T', 'Trp':'W', 'Tyr':'Y', 'Val':'V'
        }
        return aa_map.get(wt_full), aa_map.get(mt_full)
    return None, None

In [ ]:
print("🎯 Phase 1 & 2 : Calcul des métriques atomiques")

path_prefix = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/"
with open(path_prefix + "esmfold_foldx_hybrid_5920.pkl", "rb") as f:
    prot_3d_db = pickle.load(f)
with open(path_prefix + "structural_impact_labels_sota.pkl", "rb") as f:
    rmsd_labels = pickle.load(f)

df = pd.read_parquet(path_prefix + "bindingdb_wildtype_mutated_fullseq.parquet")

# on utilise un dictionnaire de recherche pour éviter df[df['variant_id']==vid]
seq_lookup = df.set_index('variant_id')['mutated_sequence'].to_dict()

unique_variants = df['variant_id'].unique()
variant_metrics = {}
error_log = {}

for vid in tqdm(unique_variants):
    try:
        uid, hgvsp = vid.split('_')[0], vid.split('_')[1]
        
        # 1. Récupération MT
        if vid not in prot_3d_db['mt']: continue
        mt_data = prot_3d_db['mt'][vid]
        coords_mt = np.array(mt_data['coords']) 
        plddt_mt = np.array(mt_data['plddt'])
        
        seq_mt_full = seq_lookup.get(vid)
        if seq_mt_full is None: continue
        
        # 2. Paramètres de fenêtre 
        s, e, rel_idx = get_win_params(len(seq_mt_full), hgvsp)
        
        # on vérifie que l'index relatif est valide pour MT et WT
        if rel_idx >= len(coords_mt):
            raise IndexError(f"Index relatif {rel_idx} hors limites pour MT ({len(coords_mt)})")

        # 3. Récupération WT
        win_key = (uid, s, e)
        if win_key not in prot_3d_db['wt']: continue
        coords_wt = np.array(prot_3d_db['wt'][win_key]['coords'])
        plddt_wt = np.array(prot_3d_db['wt'][win_key]['plddt'])
        
        if rel_idx >= len(coords_wt):
            raise IndexError(f"Index relatif {rel_idx} hors limites pour WT ({len(coords_wt)})")        
     
        # 4. Géométrie et Stabilité (Utilisation systématique de rel_idx)
        l_rmsd = rmsd_labels.get(vid, {}).get('local_rmsd_15A', 0.0)

        variant_metrics[vid] = {
            'structural_rmsd_A': l_rmsd,
            'true_delta_sasa': float(mt_data.get('true_delta_sasa', 0.0)),            
            'true_delta_packing': float(mt_data.get('true_delta_packing', 0.0)),      
            'true_delta_ddg': float(mt_data.get('true_delta_ddg', 0.0)),           
            'true_delta_electro': float(mt_data.get('true_delta_electro', 0.0)),
            'true_delta_solv_hydro': float(mt_data.get('true_delta_solv_hydro', 0.0)),
            'true_delta_clash': float(mt_data.get('true_delta_clash', 0.0)),      
            'plddt_confidence': mt_data['mean_plddt'] / 100.0
        }
    except Exception as e:
        err_name = type(e).__name__
        error_log[err_name] = error_log.get(err_name, 0) + 1
        continue

# Rapport d'erreurs
print(f"\nSuccès : {len(variant_metrics):,} / {len(unique_variants):,}")
if error_log:
    print("Erreurs rencontrées :")
    for err, count in error_log.items():
        print(f"{err}: {count} variants")

# Injection
if len(variant_metrics) > 0:
    metrics_df = pd.DataFrame.from_dict(variant_metrics, orient='index').reset_index().rename(columns={'index': 'variant_id'})
    cols_to_update = [col for col in metrics_df.columns if col in df.columns and col != 'variant_id']
    if cols_to_update:
        df = df.drop(columns=cols_to_update)
    df = df.merge(metrics_df, on='variant_id', how='left')
    print("Injection réussie.")
else:
    print("Aucune métrique calculée.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Vérification des distributions physiques générées :")

features_to_plot = ['true_delta_sasa', 'true_delta_packing', 
                    'true_delta_ddg', 'true_delta_solv_hydro', 'true_delta_electro', 'true_delta_clash', 'structural_rmsd_A']

fig, axes = plt.subplots(3, 3, figsize=(20, 15))
axes = axes.flatten()

for i, col in enumerate(features_to_plot):
    if col in df.columns:
        sns.histplot(df[col].dropna(), bins=50, kde=True, ax=axes[i], color='teal')
        axes[i].set_title(f"Distribution de {col}")
        axes[i].set_ylabel("Fréquence")
    else:
        print(f"Attention : la colonne {col} est introuvable dans df.")

# 3. Masquage des cases vides
for j in range(len(features_to_plot), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("distribution_features", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import re
import os
# Assure-toi que torch_cluster est installé : %pip install torch-cluster
from torch_cluster import radius_graph, knn_graph 

def precompute_all_edges_optimized(df_input, prot_3d_db, ligand_db):
    print("Démarrage du pré-calcul optimisé des graphes...")
    
    protein_edges_cache = {} #<-- Cache pour les arrêtes Protéine-Protéine
    binding_edges_cache = {} #<-- Cache pour les arrêtes Proteine-Ligand
    error_log = {} # Initialisation du log d'erreurs
    
    # Lookup unique pour les infos de séquences
    data_lookup = df_input.drop_duplicates(subset='variant_id').set_index('variant_id').to_dict('index')
    unique_pairs = df_input[['variant_id', 'Ligand SMILES']].drop_duplicates()

    for idx, row in tqdm(unique_pairs.iterrows(), total=len(unique_pairs), desc="Caching Edges"):
        vid = row['variant_id']
        smiles = row['Ligand SMILES']
        
        try:
            # 1. Vérification Protein DB
            if vid not in prot_3d_db['mt']: 
                 continue
                
            variant_info = data_lookup[vid]
            hgvsp = variant_info['mutation_hgvsp']
            seq_full = variant_info['mutated_sequence']
            
            # 2. Gestion Protéine
            if vid not in protein_edges_cache:
                # Récupération des coordonnées (Elles font déjà max 256 AA)
                coords_list = prot_3d_db['mt'][vid].get('coords', [])
                if len(coords_list) == 0: continue
                
                pos_p = torch.tensor(coords_list, dtype=torch.float32)
                actual_len = pos_p.shape[0]
                
                # Calcul de l'index local de la mutation
                # on doit retrouver exactement le 's' utilisé lors de la génération 
                # on utilise la même logique que la fonction get_window_indices de la cellule de génération des structures protéiques 3D ESMfold
                match = re.search(r'p\.[A-Z][a-z]{2}(\d+)|p\.[A-Z](\d+)', str(hgvsp))
                if match:
                    pos_val = match.group(1) if match.group(1) else match.group(2)
                    pos = int(pos_val)
                else:
                    pos = len(seq_full) // 2
                
                center = pos - 1
                if len(seq_full) <= 256:
                    s_gen = 0
                else:
                    s_gen = max(0, center - 256 // 2)
                    if s_gen + 256 > len(seq_full):
                        s_gen = max(0, len(seq_full) - 256)
                
                # L'index de la mutation dans le bloc de 256 est :
                local_mut_idx = (pos - 1) - s_gen
                
                # on s'assure que l'index est dans les bornes du tenseur réel
                if local_mut_idx < 0 or local_mut_idx >= actual_len:
                    # Si c'est hors borne, on tente de le clip l'index pour ne pas perdre la donnée
                    local_mut_idx = max(0, min(local_mut_idx, actual_len - 1))

                # Centrage sur le résidu muté
                pivot = pos_p[local_mut_idx]
                pos_p_centered = pos_p - pivot
                
                # Calcul des edges Protéine-Protéine
                edge_p_p = torch.cat([
                    radius_graph(pos_p_centered, r=8.0, loop=False),
                    knn_graph(pos_p_centered, k=6, loop=False)
                ], dim=1).unique(dim=0)
                
                # on stocke s=0, e=actual_len car on ne slice plus
                protein_edges_cache[vid] = {'edges': edge_p_p, 's': 0, 'e': actual_len, 'idx': local_mut_idx}

            # 3. Gestion Ligand
            if smiles not in ligand_db:
                continue
                
            pos_l = torch.tensor(ligand_db[smiles]['coords'], dtype=torch.float32)
            
            # 4. Gestion Liaison (Protein <-> Ligand)
            cache_p = protein_edges_cache[vid]
            
            # on récupère les coordonnées (déjà cropped)
            prot_coords_cropped = torch.tensor(prot_3d_db['mt'][vid]['coords'], dtype=torch.float32)
            
            # Le pivot est à l'index 'idx' du cache
            pivot = prot_coords_cropped[cache_p['idx']]
            pos_p_centered = prot_coords_cropped - pivot
            
            # Centrage du ligand
            pos_l_centered = pos_l - pos_l.mean(dim=0)
            
            # Calcul des distances et edges
            dist_mat = torch.cdist(pos_p_centered, pos_l_centered)
            edge_p_l = (dist_mat < 5.0).nonzero().t()

            binding_edges_cache[(vid, smiles)] = edge_p_l

        except Exception as e:
            err_name = type(e).__name__
            error_log[err_name] = error_log.get(err_name, 0) + 1
            continue
                    

    # Sauvegarde
    torch.save(protein_edges_cache, "protein_edges.pt")
    torch.save(binding_edges_cache, "binding_edges.pt")
    
    print(f"\nCache optimisé terminé !")
    print(f"Variants : {len(protein_edges_cache)} | Paires : {len(binding_edges_cache)}")
    if error_log:
        print("Erreurs rencontrées :", error_log)

# Appel de la fonction
# df, prot_3d_db et ligand_db sont bien chargés en mémoire
precompute_all_edges_optimized(df, prot_3d_db, ligand_db)

In [ ]:
import torch

# Charger le fichier d'origine
prot_in = "/kaggle/input/datasets/jakeadam68/binding-edges-sota/protein_edges.pt"
data_dict = torch.load(prot_in, map_location='cpu')

# Récupérer la première clé et sa valeur pour inspection
first_key = list(data_dict.keys())[0]
first_value = data_dict[first_key]

print("Diagnostique de la structure :")
print(f"Type de la clé principale : {type(first_key)}")
print(f"Exemple de clé : {first_key}")
print(f"Type de la valeur associée : {type(first_value)}")

if isinstance(first_value, dict):
    print(f"Clés du dictionnaire interne : {list(first_value.keys())}")
    for k, v in first_value.items():
        if isinstance(v, torch.Tensor):
            print(f"  -> '{k}' est un Tenseur de forme : {v.shape} et de type {v.dtype}")
        else:
            print(f"  -> '{k}' est de type : {type(v)}")
else:
    print("La valeur n'est pas un dictionnaire. Voici un aperçu :", first_value)

In [ ]:
import torch

# Charger le fichier d'origine
prot_in = "/kaggle/input/datasets/jakeadam68/binding-edges-sota/binding_edges.pt"
data_dict = torch.load(prot_in, map_location='cpu')

# Récupérer la première clé et sa valeur pour inspection
first_key = list(data_dict.keys())[0]
first_value = data_dict[first_key]

print("Diagnostique de la structure :")
print(f"Type de la clé principale : {type(first_key)}")
print(f"Exemple de clé : {first_key}")
print(f"Type de la valeur associée : {type(first_value)}")

if isinstance(first_value, dict):
    print(f"Clés du dictionnaire interne : {list(first_value.keys())}")
    for k, v in first_value.items():
        if isinstance(v, torch.Tensor):
            print(f"  -> '{k}' est un Tenseur de forme : {v.shape} et de type {v.dtype}")
        else:
            print(f"  -> '{k}' est de type : {type(v)}")
else:
    print("La valeur n'est pas un dictionnaire. Voici un aperçu :", first_value)

In [ ]:
import torch

def convert_to_flat_format(input_path, tensor_out, offset_out):
    print(f"Conversion de {input_path}...")
    data_dict = torch.load(input_path, map_location='cpu')
    
    all_edges = []
    offsets = {}
    current_offset = 0
    
    for key, value in data_dict.items():
        # 1. Extraction du tenseur d'arêtes
        if isinstance(value, dict) and 'edges' in value:
            edges_tensor = torch.as_tensor(value['edges'], dtype=torch.long)
        else:
            edges_tensor = torch.as_tensor(value, dtype=torch.long)
        
        # 2. Détection de la forme
        if edges_tensor.ndim == 2 and edges_tensor.size(0) == 2:
            num_edges = edges_tensor.size(1)
            concat_dim = 1
        else:
            num_edges = edges_tensor.size(0)
            concat_dim = 0
            
        all_edges.append(edges_tensor)
        
        # 3. Stockage des offsets et des métadonnées associées
        offsets[key] = {
            'start': current_offset,
            'end': current_offset + num_edges,
            's': value.get('s', None) if isinstance(value, dict) else None,
            'e': value.get('e', None) if isinstance(value, dict) else None,
            'idx': value.get('idx', None) if isinstance(value, dict) else None,
        }
        current_offset += num_edges
        
    flat_tensor = torch.cat(all_edges, dim=concat_dim)
    
    torch.save(flat_tensor, tensor_out)
    torch.save(offsets, offset_out)
    print(f"Terminé ! Fichier plat : {tensor_out} (Forme : {list(flat_tensor.shape)})")
    print(f"Fichier offsets : {offset_out}\n")

# Chemins sources (/kaggle/input)
prot_in = "/kaggle/input/datasets/jakeadam68/binding-edges-sota/protein_edges.pt"
bind_in = "/kaggle/input/datasets/jakeadam68/binding-edges-sota/binding_edges.pt"

# Chemins cibles (/kaggle/working)
prot_flat = "/kaggle/working/protein_edges_flat.pt"
prot_off = "/kaggle/working/protein_offsets.pt"
bind_flat = "/kaggle/working/binding_edges_flat.pt"
bind_off = "/kaggle/working/binding_offsets.pt"

convert_to_flat_format(prot_in, prot_flat, prot_off)
convert_to_flat_format(bind_in, bind_flat, bind_off)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd # Ajout de l'import pour être sûr
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

def audit_feature_redundancy(df, features, targets=None):
    # 1. Vérification des colonnes
    all_cols = features + (targets if targets else [])
    missing_cols = [col for col in all_cols if col not in df.columns]
    if missing_cols:
        print(f"Erreur : Les colonnes suivantes sont manquantes : {missing_cols}")
        return 
    
    print("Audit de Redondance des Métriques Biophysiques")

    # on supprime les NaN pour ne pas fausser les corrélations et le VIF
    df_clean = df[all_cols].dropna()
    
    # A. Matrice de corrélation (Features + Targets)
    # Ici, on peut garder les targets pour voir si les features sont liées à la cible

    plt.figure(figsize=(12, 10))
    corr_matrix = df_clean.corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title("Matrice de Corrélation : Features Biophysiques vs Cibles")
    plt.savefig("correlation_matrix_full.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    # B. Calcul du Variance Indicator Factor (Uniquement sur les features)
    
    print("\nCalcul du Variance Inflation Factor (VIF) sur les Features...")
    X = df_clean[features]
    X_vif = add_constant(X)
    
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X.columns))]
    
    print(vif_data.sort_values(by="VIF", ascending=False))
    print("\nInterprétation : VIF < 5 : Faible | 5 < VIF < 10 : Modéré | VIF > 10 : Forte multicolinéarité")

# Application

# 1. on définit uniquement les variables d'entrée (le "X")
features_to_audit = [
    'true_delta_sasa', 'true_delta_packing', 'true_delta_ddg', 
    'true_delta_solv_hydro', 'true_delta_electro', 'true_delta_clash'
]

# 2. on définit les cibles pour la visualisation de la corrélation
targets_to_audit = [
    'structural_rmsd_A'
]

audit_feature_redundancy(df, features_to_audit, targets_to_audit)

In [ ]:
class SOTAVariantDataset(Dataset):
    
  # Constructeur mis à jour pour accepter le partage de mémoire
 def __init__(self, dataframe, ligand_db, protein_db, esm2_bundle=None, phase=1,
              protein_offsets=None, binding_offsets=None,
              protein_edges_flat=None, binding_edges_flat=None, gene_to_id=None):
     super().__init__()
     self.df = dataframe.reset_index(drop=True)
     self.ligand_db = ligand_db
     self.protein_db = protein_db
     self.esm2_bundle = esm2_bundle
     self.phase = phase
     self.ligand_edges_ram_cache = {}
     self.gene_to_id = gene_to_id if gene_to_id is not None else {}

     # Configuration Memory-Mapped 
     # Chemin d'accès Kaggle pour le modèle attaché
     model_dir = "/kaggle/input/datasets/jakeadam68/binding-edges-flat"

     # S'ils sont passés en argument, on utilise les objets partagés (Évite la duplication RAM)
     self.protein_offsets = protein_offsets
     self.binding_offsets = binding_offsets
     self.protein_edges_flat = protein_edges_flat
     self.binding_edges_flat = binding_edges_flat
      
 def __len__(self):
     return len(self.df)
 
 def __getitem__(self, idx):
     row = self.df.iloc[idx]
     vid = row['variant_id']
     smiles = row.get('Ligand SMILES', '')
     uid = row['uniprot_id']
     try:
         return self._build_data(row, vid, smiles, uid)
     except Exception as e:
         raise RuntimeError(f"Erreur d'extraction pour {vid} : {e}")
 
 def _build_data(self, row, vid, smiles, uid):

     # 1. Récupération des données brutes (Protéine entière)
     prot = self.protein_db['mt'][vid]
     full_coords = torch.tensor(prot['coords'], dtype=torch.float32)
     full_plddt = torch.tensor(prot['plddt'], dtype=torch.float32) / 100.0
     full_seq = row['mutated_sequence']

     s, e, local_mut_idx = None, None, None

     # on essaie de lire directement s, e et idx depuis mes offsets en RAM 
     if vid in self.protein_offsets:
         meta = self.protein_offsets[vid]
         if isinstance(meta, dict):
             s = meta.get('s')
             e = meta.get('e')
             local_mut_idx = meta.get('idx')

     # si absent ou non trouvé (Fallback), on calcule à la volée via HGVSP
     if s is None or e is None or local_mut_idx is None:
         s, e, local_mut_idx = get_win_params(len(full_seq), row['mutation_hgvsp'])
        
     # 3. Découpage synchronisé 
     # on ne garde que la fenêtre pour tout : séquence, coordonnées et pLDDT
     window_seq = full_seq[s:e]
     # 3. Découpage intelligent 
     # on vérifie si la base de données contient la protéine entière ou juste la fenêtre
     # si la taille est > 256, on découpe. si elle est <= 256, on considère que c'est déjà la fenêtre.
     if full_coords.shape[0] > 256:
         # Cas 1 : Protéine entière dans la DB
         pos_p_window = full_coords[s:e]
         plddt_window = full_plddt[s:e]
     else:
         # Cas 2 : Déjà découpé en fenêtre dans la DB (le cas probable ici)
         pos_p_window = full_coords
         plddt_window = full_plddt

     # on s'assure que la taille du pLDDT correspond exactement à la séquence
     # Si la DB est un peu plus courte que 256 (ex: protéine courte), on ajuste la séquence
     L_actual = plddt_window.shape[0]
     if L_actual != len(window_seq):
         window_seq = window_seq[:L_actual]

     if self.esm2_bundle is not None and vid in self.esm2_bundle['mt'] and vid in self.esm2_bundle['wt']:
         esm_data_mt = self.esm2_bundle['mt'][vid]
         esm_data_wt = self.esm2_bundle['wt'][vid]
         
         full_esm_mt = torch.tensor(esm_data_mt['emb'], dtype=torch.float32)
         full_esm_wt = torch.tensor(esm_data_wt['emb'], dtype=torch.float32)
         
         offset = esm_data_mt['offset']
         s_rel, e_rel = max(0, s - offset), min(full_esm_mt.shape[0], e - offset)
         
         esm_window_mt = full_esm_mt[s_rel:e_rel]
         esm_window_wt = full_esm_wt[s_rel:e_rel]
         
         # 1. Calcul du Delta (L'impact de la mutation sur le langage des protéines)
         delta_esm_window = esm_window_mt - esm_window_wt
         
         # 2. Concaténation (MT + Delta) -> Dimension 2560
         esm_combined = torch.cat([esm_window_mt, delta_esm_window], dim=-1)
         
         if esm_combined.shape[0] < 256:
             padding = torch.zeros((256 - esm_combined.shape[0], 2560), dtype=torch.float32)
             esm_combined = torch.cat([esm_combined, padding], dim=0)
         else:
             esm_combined = esm_combined[:256]
     else:
         esm_combined = torch.zeros((256, 2560), dtype=torch.float32)

     metrics_cols = ['true_delta_sasa', 'true_delta_packing', 'true_delta_solv_hydro', 'true_delta_electro', 'true_delta_clash', 'true_delta_ddg']

     if self.phase == 1:
         # En Phase 1 : Aucun Ligand. on construit un graphe homogène.
         data = Data()
         x_local = encode_sequence(window_seq, local_mut_idx, plddt_window, metrics_vector=torch.zeros(6), esm_vector=esm_combined)
         
         data.x = x_local
         data.pos = pos_p_window - pos_p_window[local_mut_idx].clone() # Centrage
         data.local_mut_idx = torch.tensor([local_mut_idx], dtype=torch.long)
         data.weight = torch.tensor([row['informational_weight']], dtype=torch.float32) # Poids info
         
         # Arrêtes protéine-protéine
         if vid in self.protein_offsets and self.protein_edges_flat is not None:
             meta = self.protein_offsets[vid]
             start, end = (meta['start'], meta['end']) if isinstance(meta, dict) else meta
             if self.protein_edges_flat.ndim == 2 and self.protein_edges_flat.size(0) == 2:
                 data.edge_index = self.protein_edges_flat[:, start:end]
             else:
                 data.edge_index = self.protein_edges_flat[start:end]

         else:
             # Fallback propre pour la Phase 1
             data.edge_index = torch.cat([
                 radius_graph(data.pos, r=8.0, loop=False),
                 knn_graph(data.pos, k=6, loop=False)
             ], dim=1).unique(dim=0)
         

         # Cibles (Targets) pour la Phase 1 : RMSD Class + Les 7 régressions
         #data.y_rmsd_class = torch.tensor([row['l_rmsd_class']], dtype=torch.long)
         
         # on groupe les 7 deltas dans un seul tenseur pour la Multitask Loss
         #deltas = [row['delta_plddt']] + [row.get(col, 0.0) for col in metrics_cols]
         #data.y_deltas = torch.tensor([deltas], dtype=torch.float32) # Shape [1, 7]
         targets = [row['structural_rmsd_A']] + [row.get(col, 0.0) for col in metrics_cols]
         data.y_deltas = torch.tensor([targets], dtype=torch.float32) # Shape [1, 7]
         # Création du masque de perte (Loss Masking)
         # Indique si FoldX a planté (ex: généré via une colonne 'foldx_valid' définie avant le Dataset)
         foldx_valid = 1.0 if row.get('foldx_valid', True) else 0.0
            
         # Indices : [RMSD, SASA, Packing, Solv, Electro, Clash, DDG]
         # RMSD, SASA et Packing sont toujours valides (1.0). FoldX dépend du flag.
         data.target_mask = torch.tensor([[1.0, 1.0, 1.0, foldx_valid, foldx_valid, foldx_valid, foldx_valid]], dtype=torch.float32)

         
         return data


     else:
         # Phase 2 : on injecte les valeurs comme features
         data = HeteroData()
         # Extraire les valeurs réelles du dataframe
         metrics_values = torch.tensor([row.get(col, 0.0) for col in metrics_cols], dtype=torch.float32)
         x_local = encode_sequence(window_seq, local_mut_idx, plddt_window, metrics_vector=metrics_values, esm_vector=esm_combined)

         data['protein'].x = x_local
         pivot = pos_p_window[local_mut_idx].clone()
         data['protein'].pos = pos_p_window - pivot
         data.local_mut_idx = torch.tensor([local_mut_idx], dtype=torch.long)
         
         if smiles in self.ligand_db:
             # Géométrie et Features Ligands 
             lig = self.ligand_db[smiles]
             data['ligand'].x = torch.tensor(lig['atom_features'], dtype=torch.float32)
             data['ligand'].pos = torch.tensor(lig['coords'], dtype=torch.float32) - torch.tensor(lig['coords'], dtype=torch.float32).mean(dim=0)
             data.ligand_global_feat = torch.tensor(lig['global_rdkit'], dtype=torch.float32)
         else:
                raise ValueError(f"SMILES introuvable dans la base de données : {smiles}")


         # 1. Ligand-Ligand
         if smiles not in self.ligand_edges_ram_cache:
             # on génère le graphe complet une seule fois pour ce SMILES
             num_l = data['ligand'].pos.size(0)
             adj = torch.ones((num_l, num_l))
             adj.fill_diagonal_(0)
             self.ligand_edges_ram_cache[smiles] = adj.nonzero().t()

         # on récupère l'index depuis la RAM 
         data['ligand', 'interacts', 'ligand'].edge_index = self.ligand_edges_ram_cache[smiles]

         # 1. Protéine-Protéine (Slicing du tenseur plat mappé)
         if vid in self.protein_offsets and self.protein_edges_flat is not None:
             meta = self.protein_offsets[vid]
             # Gestion de la rétrocompatibilité (si vos anciens offsets étaient de simples tuples)
             start, end = (meta['start'], meta['end']) if isinstance(meta, dict) else meta
            
             # Découpage dynamique de l'intervalle
             if self.protein_edges_flat.ndim == 2 and self.protein_edges_flat.size(0) == 2:
                 data['protein', 'interacts', 'protein'].edge_index = self.protein_edges_flat[:, start:end]
             else:
                 data['protein', 'interacts', 'protein'].edge_index = self.protein_edges_flat[start:end]
         else:
             # Fallback si manque dans le cache
             data = compute_biophysical_edges(data)
    
         # 3. Protéine-Ligand (Slicing du tenseur plat mappé)
         pair_key = (vid, smiles)
         if pair_key in self.binding_offsets and self.binding_edges_flat is not None:
             meta = self.binding_offsets[pair_key]
             if isinstance(meta, dict):
                 start, end = meta['start'], meta['end']
             else:
                 start, end = meta
            
             # Slicing robuste selon les dimensions réelles des arêtes
             if self.binding_edges_flat.ndim == 2 and self.binding_edges_flat.size(0) == 2:
                 edge_index = self.binding_edges_flat[:, start:end]
                 data['protein', 'binds', 'ligand'].edge_index = edge_index
                 # Pour inverser au format [2, N], on permute les lignes 0 et 1
                 data['ligand', 'binds', 'protein'].edge_index = edge_index[[1, 0], :]
             else:
                 edge_index = self.binding_edges_flat[start:end]
                 data['protein', 'binds', 'ligand'].edge_index = edge_index
                 # Pour inverser au format [N, 2], on permute les colonnes 0 et 1
                 data['ligand', 'binds', 'protein'].edge_index = edge_index[:, [1, 0]]
         else:
             # Calcul on-the-fly si manquant
             dist_mat = torch.cdist(data['protein'].pos, data['ligand'].pos)
             edge_p_l = (dist_mat < 5.0).nonzero().t()
             data['protein', 'binds', 'ligand'].edge_index = edge_p_l
             # Comme nonzero().t() renvoie un tenseur de forme [2, N], l'inversion permute les lignes 0 et 1
             data['ligand', 'binds', 'protein'].edge_index = edge_p_l[[1, 0], :]

        
         # Cibles d'affinité (Phase 2)
         data.y_delta_pAff = torch.tensor([row['delta_pAff']], dtype=torch.float32)
         data.weight = torch.tensor([row['label_weight']], dtype=torch.float32)

         # Conversion du gène en entier pour la pairwise ranking loss
         g_id = self.gene_to_id.get(row['gene_symbol'], 0)
         data.gene_id = torch.tensor([g_id], dtype=torch.long) 

         return data

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import numpy as np

print("Phase Split : Généralisation en utilisant une stratégie stratified Group Split")

# 1. Calcul de la popularité des gènes
gene_counts = df['gene_symbol'].value_counts()
# on crée 3 catégories de popularité : Rare, Moyen, Fréquent
# on utilise les quantiles pour que les groupes soient équilibrés en nombre de gènes
low = gene_counts.quantile(0.33)
high = gene_counts.quantile(0.66)

def assign_strat(count):
    if count <= low: return 'rare'
    elif count <= high: return 'medium'
    else: return 'frequent'

# on crée un mapping gène -> strate
gene_strat_map = gene_counts.apply(assign_strat).to_dict()
df['gene_strat'] = df['gene_symbol'].map(gene_strat_map)

# 2. Split stratifié par groupe
# on fait le split pour chaque strate séparément pour garantir l'équilibre
df_train, df_val, df_test = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

for strat in ['rare', 'medium', 'frequent']:
    df_strat = df[df['gene_strat'] == strat].copy()
    
    # Split 1: Train vs (Val + Test)
    gss1 = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=RANDOM_SEED)
    train_idx, rem_idx = next(gss1.split(df_strat, groups=df_strat['gene_symbol']))
    
    df_train_s = df_strat.iloc[train_idx]
    df_rem_s = df_strat.iloc[rem_idx]
    
    # Split 2: Val vs Test (50/50 du reste)
    gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=RANDOM_SEED)
    val_idx, test_idx = next(gss2.split(df_rem_s, groups=df_rem_s['gene_symbol']))
    
    df_val_s = df_rem_s.iloc[val_idx]
    df_test_s = df_rem_s.iloc[test_idx]
    
    # Accumulation
    df_train = pd.concat([df_train, df_train_s])
    df_val = pd.concat([df_val, df_val_s])
    df_test = pd.concat([df_test, df_test_s])

# 3. Vérification finale
genes_train = set(df_train['gene_symbol'].unique())
genes_val = set(df_val['gene_symbol'].unique())
genes_test = set(df_test['gene_symbol'].unique())

assert genes_train.isdisjoint(genes_val), "Fuite Train/Val !"
assert genes_train.isdisjoint(genes_test), "Fuite Train/Test !"
assert genes_val.isdisjoint(genes_test), "Fuite Val/Test !"

total_pairs = len(df)
print(f"Répartition équilibrée et stratifiée (Total: {total_pairs:,} paires)")

# Affichage détaillé pour chaque set
print(f"Train set : {len(df_train)} paires | {len(genes_train)} gènes | {len(df_train)/total_pairs:.2%} du total")
print(f"Val set   : {len(df_val):} paires | {len(genes_val)} gènes | {len(df_val)/total_pairs:.2%} du total")
print(f"Test set  : {len(df_test):} paires | {len(genes_test)} gènes | {len(df_test)/total_pairs:.2%} du total")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

print("Phase Split gold : Stratified Group Split Indépendant du fichier globale pour la phase 2")

# 1. Chargement du vrai Gold Standard de 16k lignes (strictement expérimental)
df_gold_raw = pd.read_parquet("/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/gold_standard_experimental_16k.parquet")

# =============================================================================
# 2. Group Split Indépendant et équilibré sur le gold standard (70/15/15)
# =============================================================================
# Calcul de la popularité des gènes uniquement sur les paires Gold
gold_gene_counts = df_gold_raw['gene_symbol'].value_counts()
low = gold_gene_counts.quantile(0.33)
high = gold_gene_counts.quantile(0.66)

def assign_gold_strat(count):
    if count <= low: return 'rare'
    elif count <= high: return 'medium'
    else: return 'frequent'

gold_gene_strat_map = gold_gene_counts.apply(assign_gold_strat).to_dict()
df_gold_raw['gene_strat'] = df_gold_raw['gene_symbol'].map(gold_gene_strat_map)

# Split stratifié par gène pour un équilibre parfait de l'évaluation
df_gold_train, df_gold_val, df_gold_test = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
RANDOM_SEED = 42

for strat in ['rare', 'medium', 'frequent']:
    df_strat = df_gold_raw[df_gold_raw['gene_strat'] == strat].copy()
    
    # Split 1: Train vs (Val + Test) - 70% Train, 30% Reste
    gss1 = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=RANDOM_SEED)
    train_idx, rem_idx = next(gss1.split(df_strat, groups=df_strat['gene_symbol']))
    
    df_train_s = df_strat.iloc[train_idx]
    df_rem_s = df_strat.iloc[rem_idx]
    
    # Split 2: Val vs Test (50/50 du reste, donc 15% / 15%)
    gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=RANDOM_SEED)
    val_idx, test_idx = next(gss2.split(df_rem_s, groups=df_rem_s['gene_symbol']))
    
    df_val_s = df_rem_s.iloc[val_idx]
    df_test_s = df_rem_s.iloc[test_idx]
    
    df_gold_train = pd.concat([df_gold_train, df_train_s])
    df_gold_val = pd.concat([df_gold_val, df_val_s])
    df_gold_test = pd.concat([df_gold_test, df_test_s])

# Vérification d'absence de fuite génétique
genes_gold_train = set(df_gold_train['gene_symbol'].unique())
genes_gold_val = set(df_gold_val['gene_symbol'].unique())
genes_gold_test = set(df_gold_test['gene_symbol'].unique())

assert genes_gold_train.isdisjoint(genes_gold_val), "Fuite Train/Val dans le Gold !"
assert genes_gold_train.isdisjoint(genes_gold_test), "Fuite Train/Test dans le Gold !"
assert genes_gold_val.isdisjoint(genes_gold_test), "Fuite Val/Test dans le Gold !"

total_gold = len(df_gold_raw)
print(f"Répartition équilibrée et stratifiée (Total: {total_gold:,} paires)")

print(f"Train Gold : {len(df_gold_train):,} paires | {len(genes_gold_train)} gènes | {len(df_gold_train)/total_gold:.2%}")
print(f"Val Gold   : {len(df_gold_val):,} paires | {len(genes_gold_val)} gènes | {len(df_gold_val)/total_gold:.2%}")
print(f"Test Gold  : {len(df_gold_test):,} paires | {len(genes_gold_test)} gènes | {len(df_gold_test)/total_gold:.2%}")

# Nettoyage des colonnes temporaires
df_gold_train.drop(columns=['gene_strat'], errors='ignore', inplace=True)
df_gold_val.drop(columns=['gene_strat'], errors='ignore', inplace=True)
df_gold_test.drop(columns=['gene_strat'], errors='ignore', inplace=True)

# 4. Calcul des quantiles RMSD 
rmsd_train_unique = df_train.drop_duplicates(subset=['variant_id'])['structural_rmsd_A'].dropna()
q33 = rmsd_train_unique.quantile(0.33333)
q66 = rmsd_train_unique.quantile(0.66667)


print("\nPhase 3 exécutée avec succès.")

In [ ]:
from sklearn.preprocessing import QuantileTransformer
import pickle

print("Correction du biais de gène ")

# 1. Injection de la physique dans le gold standard 

print("Injection des métriques physiques dans le fichier expérimentale qui servira pour le fine-tunning...")

# on supprime delta_pAff de metrics_df au cas où, pour éviter les doublons
metrics_df_clean = metrics_df.drop(columns=['delta_pAff'], errors='ignore')

# Les 7 colonnes physiques cibles d'AEGIS-GT
cols_physics = [
    'structural_rmsd_A', 'true_delta_sasa', 'true_delta_packing', 
    'true_delta_solv_hydro', 'true_delta_electro', 'true_delta_clash', 'true_delta_ddg'
]

# on supprime toutes les colonnes physiques et les résidus de fusions précédents (_x, _y)
# pour éviter tout conflit de merge lors de la ré-exécution de la cellule.
print("🧹 Nettoyage défensif des colonnes physiques et résidus de fusions...")

for name, dset in zip(['df_train', 'df_val', 'df_test', 'df_gold_train', 'df_gold_val', 'df_gold_test'], 
                      [df_train, df_val, df_test, df_gold_train, df_gold_val, df_gold_test]):
    
    # on repère toutes les colonnes de physique ou polluées par des suffixes _x/_y
    cols_to_drop = [
        col for col in dset.columns 
        if col in cols_physics or col == 'plddt_confidence' or col.endswith('_x') or col.endswith('_y')
    ]
    if cols_to_drop:
        dset.drop(columns=cols_to_drop, errors='ignore', inplace=True)

print("Injection propre des métriques physiques...")
df_train = df_train.merge(metrics_df_clean, on='variant_id', how='left')
df_val = df_val.merge(metrics_df_clean, on='variant_id', how='left')
df_test = df_test.merge(metrics_df_clean, on='variant_id', how='left')

# on fusionne la table metrics_df sur le variant_id pour apporter 'plddt_confidence' et la physique
df_gold_train = df_gold_train.merge(metrics_df_clean, on='variant_id', how='left')
df_gold_val   = df_gold_val.merge(metrics_df_clean, on='variant_id', how='left')
df_gold_test  = df_gold_test.merge(metrics_df_clean, on='variant_id', how='left')

# on détecte les plantages de FoldX (quand ddg et solv_hydro valent exactement 0.0)
# on applique ça sur les jeux Silver (Phase 1) et Gold (Phase 2)
for dset in [df_train, df_val, df_test, df_gold_train, df_gold_val, df_gold_test]:
    dset['foldx_valid'] = ~((dset['true_delta_ddg'] == 0.0) & (dset['true_delta_solv_hydro'] == 0.0))

# 1. Paramètres de l'Information Effective
beta = 0.9999  # plus beta est proche de 1, plus on s'approche de l'ICF classique
epsilon = 1e-6

# 2. Calcul des fréquences
unique_variants_df = df.drop_duplicates(subset=['variant_id'])
gene_counts = unique_variants_df.groupby('gene_symbol').size()

global_std = df_gold_train['delta_pAff'].std()
print(f"Écart-type réel de delta_pAff détecté : {global_std:.4f}")

# 3. Calcul du Nombre Effectif (Effective Number of Samples)
# Formule : En = (1 - beta^n) / (1 - beta)
# on intègre la diversité (global_std) pour pondérer l'importance
eff_num = (1.0 - np.power(beta, gene_counts)) / (1.0 - beta)

# on pondère l'information effective par la diversité structurelle
# on utilise l'inverse car plus l'information effective est grande, plus le poids doit être bas
gene_weights = 1.0 / (eff_num * global_std + epsilon)

# 4. Normalisation Rigoureuse (Moyenne = 1.0)
# c'est l'étape la plus importante pour la stabilité du gradient
gene_weights = gene_weights / gene_weights.mean()

# Stabilisation du Gradient
# Même avec l'information effective, un ratio de 700k est trop dangereux.
# on applique un "Smooth Capping" : on limite le poids max à 100x la moyenne.
# cela ne détruit pas l'information, mais empêche un seul exemple de faire exploser le gradient.
max_weight = 100.0 * gene_weights.mean()

gene_weights = np.clip(gene_weights, a_min=None, a_max=max_weight)

# on renormalise une dernière fois après le clip
gene_weights = gene_weights / gene_weights.mean()

# Mappage sur l'intégralité du dataset
df_train['informational_weight'] = df_train['gene_symbol'].map(gene_weights).astype(np.float32)
df_val['informational_weight'] = df_val['gene_symbol'].map(gene_weights).astype(np.float32)
df_test['informational_weight'] = df_test['gene_symbol'].map(gene_weights).astype(np.float32)

# Mappage des poids d'information sur les 3 Gold sets réels
df_gold_train['informational_weight'] = df_gold_train['gene_symbol'].map(gene_weights).astype(np.float32)
df_gold_val['informational_weight']   = df_gold_val['gene_symbol'].map(gene_weights).astype(np.float32)
df_gold_test['informational_weight']  = df_gold_test['gene_symbol'].map(gene_weights).astype(np.float32)

# 2. Poids de la Source et de la Structure 
source_map = {'Ki (nM)': 1.0, 'Kd (nM)': 1.0, 'IC50 (nM)': 0.5, 'EC50 (nM)': 0.2}

CLINVAR_CONFIDENCE_MAP = {
    'Pathogenic': 0.99, 
    'Pathogenic/Likely pathogenic': 0.95, 
    'Likely pathogenic': 0.90, 
    'Unknown': 0.50
}

def structural_confidence(plddt):
    return 1.0 / (1.0 + np.exp(-20 * (plddt - 0.7)))

for target_df in [df_gold_train, df_gold_val, df_gold_test]:
    # 1. Poids Expérimental
    target_df['source_weight'] = target_df['aff_type'].map(source_map).fillna(0.2)
    # 2. Poids Structurel (ESMFold)
    target_df['struct_weight'] = structural_confidence(target_df['plddt_confidence'].fillna(0))
    target_df['label_weight'] = (target_df['informational_weight'] * target_df['source_weight'] * target_df['struct_weight']).astype(np.float32)

# 2. on utilise uniquement le QuantileTransformer 
scaler = QuantileTransformer(output_distribution='normal', n_quantiles=300, random_state=42)

train_unique_for_scaler = df_train.drop_duplicates(subset=['variant_id'])
scaler.fit(train_unique_for_scaler[cols_physics].fillna(0))

# 3. Application
df_train[cols_physics] = scaler.transform(df_train[cols_physics].fillna(0))
df_val[cols_physics]   = scaler.transform(df_val[cols_physics].fillna(0))
df_test[cols_physics]  = scaler.transform(df_test[cols_physics].fillna(0))

# Normalisation du Gold Standard réel
df_gold_train[cols_physics] = scaler.transform(df_gold_train[cols_physics].fillna(0))
df_gold_val[cols_physics]   = scaler.transform(df_gold_val[cols_physics].fillna(0))
df_gold_test[cols_physics]  = scaler.transform(df_gold_test[cols_physics].fillna(0))

# Sauvegarde du scaler pour inverser les prédictions plus tard
with open("phase1_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Sauvegarde des fichiers gold sets réels normalisés
df_gold_train.to_parquet("df_gold_train_clean.parquet", index=False)
df_gold_val.to_parquet("df_gold_val_clean.parquet", index=False)
df_gold_test.to_parquet("df_gold_test_clean.parquet", index=False)

print("Normalisation Quantile globale et Pondérations appliquées avec succès.")

In [ ]:
import pandas as pd
import numpy as np

print("Audit des Statistiques Post-Normalisation Quantile Transformer")

cols_to_scale = ['structural_rmsd_A', 'true_delta_sasa', 'true_delta_packing', 'true_delta_ddg', 
                 'true_delta_solv_hydro', 'true_delta_electro', 'true_delta_clash']

audit_stats = []

# on extrait les variants uniques du train set pour vérifier le scaler pur
df_train_unique = df_train.drop_duplicates(subset=['variant_id'])

for col in cols_to_scale:
    audit_stats.append({
        'Feature': col,
        'Train Unique Mean': df_train_unique[col].mean(), # La vraie moyenne sans biais de ligand
        'Train Unique Std': df_train_unique[col].std(),
        'Train Full Mean': df_train[col].mean(),          # La moyenne pondérée par les ligands
        'Train Full Std': df_train[col].std()
    })

audit_df = pd.DataFrame(audit_stats).set_index('Feature')

# on affiche le tableau arrondi à 4 décimales pour la lisibilité
display(audit_df.round(4))

# Vérification algorithmique sur le Train Set
train_mean_ok = np.allclose(audit_df['Train Unique Mean'], 0, atol=0.15)
train_std_ok = np.allclose(audit_df['Train Unique Std'], 1, atol=0.15) # L'écart-type pandas vs sklearn peut avoir une micro-différence due au degré de liberté ( ddof = 1 pour pandas & ddof = 0 pour scikit learn)

if train_mean_ok and train_std_ok:
    print("\nSuccès absolu : L'espace latent est parfaitement conditionné.")
    print("Moyennes proches de 0 et écarts-types proches de 1. SwiGLU et la Huber Loss vont converger sans problème !")
else:
    print("\nAttention : Les données semblent s'écarter de la loi normale.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def audit_gene_weights(df):
    print("⚖️ Audit des Poids d'Importance par Gène")
    
    # 1. on crée un DataFrame résumé par gène
    # Mon utilise 'nunique' pour compter la biologie (variations), pas la chimie
    gene_analysis = df.groupby('gene_symbol').agg({
        'informational_weight': 'mean',
        'variant_id': 'nunique'  
    }).rename(columns={'variant_id': 'unique_variants'})
    
    # 2. Statistiques descriptives
    print("\nStatistiques des Poids")
    print(f"Poids Minimum : {gene_analysis['informational_weight'].min():.6f}")
    print(f"Poids Maximum : {gene_analysis['informational_weight'].max():.6f}")
    print(f"Poids Moyen   : {gene_analysis['informational_weight'].mean():.6f}")
    print(f"Ratio Max/Min  : {gene_analysis['informational_weight'].max() / gene_analysis['informational_weight'].min():.2f}x")
    
    # 3. Identification des gènes extrêmes
    print("\nTop 5 Gènes les plus diversifiés (Poids Faible)")
    print(gene_analysis.sort_values('informational_weight').head(5))
    print("\nTop 5 Gènes les plus rares (Poids Fort)")
    print(gene_analysis.sort_values('informational_weight', ascending=False).head(5))
    
    # 4. Visualisation : Corrélation Fréquence vs Poids
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=gene_analysis, x='unique_variants', y='informational_weight', alpha=0.7, color='purple')
    plt.xscale('log') # Échelle logarithmique
    plt.yscale('log') # Échelle logarithmique
    plt.title("Relation entre la Diversité du Gène et son Poids d'Importance")
    plt.xlabel("Nombre de variants uniques (échelle log)")
    plt.ylabel("Poids attribué (échelle log)")
    plt.grid(True, which="both", ls="-", alpha=0.2)
    plt.savefig("distribution_poids_par_gène.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    return gene_analysis

# Application sur le set d'entraînement
gene_stats = audit_gene_weights(df_train)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Visualisation des distributions post normalisation :")

features_to_plot = [
    'true_delta_sasa', 'true_delta_packing', 
    'true_delta_solv_hydro', 'true_delta_electro', 
    'true_delta_clash', 'true_delta_ddg', 'structural_rmsd_A'
]

# Grille 3x3 pour nos 9 variables
fig, axes = plt.subplots(3, 3, figsize=(20, 15))
axes = axes.flatten()

# on isole les variants uniques pour voir la vraie loi normale sans le biais des ligands
df_train_unique = df_train.drop_duplicates(subset=['variant_id'])

for i, col in enumerate(features_to_plot):
    # on utilise df_train qui contient maintenant les données transformées
    sns.histplot(df_train_unique[col].dropna(), bins=50, kde=True, ax=axes[i], color='indigo')
    
    # Ajout d'une ligne verticale rouge pour marquer le zéro parfait
    axes[i].axvline(0, color='red', linestyle='--', alpha=0.5)
    
    axes[i].set_title(f"Post-Norm: {col}")
    axes[i].set_ylabel("Fréquence")

# Masquage des cases vides (indices 7 et 8)
for j in range(len(features_to_plot), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("distribution_features_normalized", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import torch
from torch.utils.data import WeightedRandomSampler
import pickle


print("Création des Datasets PyTorch")

# Chargement direct des gold sets réels, normalisés et nettoyés
df_gold_train_clean = pd.read_parquet("df_gold_train_clean.parquet")
df_gold_val_clean   = pd.read_parquet("df_gold_val_clean.parquet")
df_gold_test_clean  = pd.read_parquet("df_gold_test_clean.parquet")

# 2. Chargement de la base de données géométrique des ligands (RDKit) [2.1] nécessaire uniquement en Phase 2 pour construire les graphes moléculaires
print("Chargement de la base de données géométrique des ligands...")
try:
    with open(path_prefix + "ligands_3d_rdkit_database.pkl", "rb") as f:
        ligand_db = pickle.load(f)
    print("Base de données des ligands chargée avec succès.")
except Exception as e:
    print(f"Erreur lors du chargement des ligands : {e}")
    ligand_db = {}

def filter_valid_samples(df, prot_3d_db, ligand_db=None, phase=1):
    """
    Filtre vectoriel intelligent selon la phase.
    Phase 1 : Vérifie uniquement la présence de la protéine.
    Phase 2 : Vérifie Protéine + Ligand.
    """
    valid_proteins = set(prot_3d_db['mt'].keys())
    initial_len = len(df)
    
    # Masque de base (Protéines)
    mask = df['variant_id'].isin(valid_proteins)
    
    # Masque additionnel (Ligands) uniquement si Phase 2
    if phase == 2 and ligand_db is not None:
        valid_ligands = set(ligand_db.keys())
        mask = mask & df['Ligand SMILES'].isin(valid_ligands)
        
    filtered_df = df[mask].copy()
    print(f"Filtrage (Phase {phase}) : {len(filtered_df):,} / {initial_len:,} paires valides conservées "
          f"({len(filtered_df)/initial_len*100:.2f}%)")
    return filtered_df


esm_path = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/esm2_pathogenic_bundle_local.pkl"

try:
    with open(esm_path, "rb") as f:
        esm2_bundle = pickle.load(f)
    print("ESM-2 Bundle chargé avec succès.")
except FileNotFoundError:
    print(f"Erreur : Le fichier {esm_path} est introuvable. Vérifiez le chemin !")
    esm2_bundle = None
except Exception as e:
    print(f"Erreur lors du chargement de ESM-2 : {e}")
    esm2_bundle = None

# Chargement unique et partagé des offsets et tenseurs (
print("\nChargement unique des offsets et mappage des fichiers plats...")
model_dir = "/kaggle/input/datasets/jakeadam68/binding-edges-flat"

shared_protein_offsets = torch.load(f"{model_dir}/protein_offsets.pt", map_location='cpu')
shared_binding_offsets = torch.load(f"{model_dir}/binding_offsets.pt", map_location='cpu')
shared_protein_edges_flat = torch.load(f"{model_dir}/protein_edges_flat.pt", mmap=False)
shared_binding_edges_flat = torch.load(f"{model_dir}/binding_edges_flat.pt", mmap=False)

print("Création du dictionnaire des gènes...")
unique_genes = df['gene_symbol'].unique()
global_gene_to_id = {gene: idx for idx, gene in enumerate(unique_genes)}

# ==========================================
# 4. Datasets pour Phase 1 (Phase 1 : Pré-entraînement par apprentissage de représentation multi-tâches informé par la biophysique.)
# ==========================================
print("\nPréparation des Datasets pour la Phase 1...")

# Extraction des variants uniques pour éviter de faire tourner le filtre sur 5 millions de lignes !
df_silver_train_unique = df_train.drop_duplicates(subset=['variant_id']).copy()
df_silver_val_unique   = df_val.drop_duplicates(subset=['variant_id']).copy()
df_test_unique         = df_test.drop_duplicates(subset=['variant_id']).copy()

print("Nettoyage des paires avec données manquantes...")
df_silver_train_clean = filter_valid_samples(df_silver_train_unique, prot_3d_db, phase=1)
df_silver_val_clean   = filter_valid_samples(df_silver_val_unique, prot_3d_db, phase=1)
df_test_clean         = filter_valid_samples(df_test_unique, prot_3d_db, phase=1)

train_dataset = SOTAVariantDataset(
    df_silver_train_clean, ligand_db, prot_3d_db, esm2_bundle, phase=1,
    protein_offsets=shared_protein_offsets,
    protein_edges_flat=shared_protein_edges_flat
)

val_dataset = SOTAVariantDataset(
    df_silver_val_clean, ligand_db, prot_3d_db, esm2_bundle, phase=1,
    protein_offsets=shared_protein_offsets,
    protein_edges_flat=shared_protein_edges_flat
)

test_dataset = SOTAVariantDataset(
    df_test_clean, ligand_db, prot_3d_db, esm2_bundle, phase=1,
    protein_offsets=shared_protein_offsets,
    protein_edges_flat=shared_protein_edges_flat
)

# Le sampler utilise l'informational_weight pour équilibrer les gènes
# tout en exposant le modèle aux 4200 variants mutées.
weights_p1 = torch.tensor(df_silver_train_clean['informational_weight'].values, dtype=torch.float32)
train_sampler = WeightedRandomSampler(
    weights=weights_p1, 
    num_samples=len(weights_p1), # on parcourt tout le dataset
    replacement=True          # obligatoire pour le WeightedRandomSampler
)

print(f"\nDatasets PyTorch générés avec succès :")
print(f"➜ Phase 1 (Train) : {len(train_dataset):,} mutations uniques")
print(f"➜ Phase 1 (Val)   : {len(val_dataset):,} mutations uniques")
print(f"➜ Phase 1 (Test)  : {len(test_dataset):,} mutations uniques")

## **Partie Architecture du modèle Transformer Géométrique**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_mean, scatter_add, scatter_softmax
from torch_geometric.nn import radius_graph
from torch_geometric.utils import to_dense_batch, to_dense_adj
from torch.utils.checkpoint import checkpoint

# =============================================================================
# 1. RMSNORM & SWIGLU
# =============================================================================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps, self.weight = eps, nn.Parameter(torch.ones(dim))
    def forward(self, x):
        # on sauvegarde le type d'origine (ex: float16 ou bfloat16)
        orig_dtype = x.dtype
        # on passe temporairement en float32 pour sécuriser le calcul carrés et la moyenne
        x_fp32 = x.to(torch.float32)
        variance = x_fp32.pow(2).mean(-1, keepdim=True)
        x_norm = x_fp32 * torch.rsqrt(variance + self.eps)
        # on repasse dans le type d'origine avant d'appliquer le poids
        return self.weight * x_norm.to(orig_dtype)

class SwiGLU(nn.Module):
    def __init__(self, dim, inter_dim=None, dropout=0.1): # Ajout du paramètre dropout
        super().__init__()
        # Si aucun inter_dim n'est donné, on utilise la formule standard de LLaMA
        self.inter_dim = inter_dim if inter_dim is not None else int(dim * 4 * 2 / 3)
        # Une seule couche linéaire qui remplace w1 et w2 pour optimiser le GEMM (General Matrix to Matrix Multiplication)
        self.w12 = nn.Linear(dim, self.inter_dim * 2)
        self.w3 = nn.Linear(self.inter_dim, dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # on projette en une seule fois (1 seule multiplication matricielle au lieu de 2)
        x12 = self.w12(x)
        # on découpe le tenseur en deux portions égales sur la dernière dimension
        x1, x2 = x12.chunk(2, dim=-1)
        # on applique la formule SwiGLU + Dropout + Projection finale
        return self.w3(self.dropout(F.silu(x1) * x2))

In [ ]:
# =============================================================================
# 1. Radial Basis Function 
# =============================================================================
class RadialBasisFunction(nn.Module):
    def __init__(self, num_gbasis=16, r_max=8.0):
        super().__init__()
        self.num_gbasis = num_gbasis
        self.r_max = r_max

        # on initialise les centres et les largeurs, mais en nn.Parameter
        # Le modèle va ajuster sa rétine sur les échelles de distance clés du dataset [7]
        initial_centers = torch.linspace(0.0, r_max, num_gbasis)
        initial_widths = torch.ones(num_gbasis) * (r_max / num_gbasis)
        
        self.centers = nn.Parameter(initial_centers)
        # epsilon de sécurité 1e-6 pour éviter toute division par zéro
        self.widths = nn.Parameter(initial_widths)

    def forward(self, dist):
        # 1. Calcul des Gaussiennes apprenables
        # on ajoute un epsilon de sécurité au dénominateur
        gaussians = torch.exp(-((dist - self.centers) ** 2) / (self.widths.pow(2) + 1e-6))
        
        # 2. Enveloppe de coupure douce (Cosine Cutoff) [7]
        # f_c(d) = 0.5 * (cos(pi * d / r_max) + 1)
        # on clampe la distance au maximum à r_max pour éviter les retours de phase du cosinus
        clamped_dist = torch.clamp(dist, max=self.r_max)
        cutoff = 0.5 * (torch.cos(clamped_dist * torch.pi / self.r_max) + 1.0)
        
        # on multiplie l'activation par l'enveloppe de coupure
        # Garantit que les arêtes s'éteignent en douceur à la frontière des 8.0 Å [7]
        return gaussians * cutoff

# =============================================================================
# 2. EGNN LAYER (Avec Dampening 0.1)
# =============================================================================
class AEGIS_GT_Layer(nn.Module):
    def __init__(self, d_model, update_coords=False):
        super().__init__()
        self.update_coords = update_coords

        # 16 bases radiales de 0 à 8.0 Å
        self.rbf = RadialBasisFunction(num_gbasis=16, r_max=8.0)

        # Norms de sécurité (Pre-Norm) pour stabiliser le SwiGLU
        self.norm_edge_in = RMSNorm(2 * d_model + 16)
        self.norm_node_in = RMSNorm(2 * d_model)
        
        self.edge_mlp = nn.Sequential(nn.Linear(2 * d_model + 16, d_model), SwiGLU(d_model, inter_dim=d_model, dropout=0.1))
        self.node_mlp = nn.Sequential(nn.Linear(2 * d_model, d_model), SwiGLU(d_model, inter_dim=d_model, dropout=0.1))
        self.norm = RMSNorm(d_model)
        if update_coords:
            self.coord_mlp = nn.Sequential(nn.Linear(d_model, d_model // 4), nn.SiLU(), nn.Linear(d_model // 4, 1), nn.Tanh())

    def forward(self, h, x, edge_index):
        if edge_index.numel() == 0: 
            return h, x
        
        row, col = edge_index

        # on effectue une seule soustraction et indexation en mémoire
        coord_diff = x[row] - x[col]
        coord_diff_f32 = coord_diff.to(torch.float32) # Passage en float32 pour la stabilité de la norme

        dist_sq = torch.sum(coord_diff_f32.pow(2), dim=-1, keepdim=True).clamp(min=1e-6, max=1e3)
        #dist_embed = torch.log1p(dist_sq).to(x.dtype) # on revient en bfloat16 pour la suite
        dist = torch.sqrt(dist_sq) # [Edges, 1]
        dist_embed = self.rbf(dist).to(h.dtype) # Shape: [Edges, 16]

        # Message avec Pre-Norm
        edge_input = torch.cat([h[row], h[col], dist_embed], dim=-1)
        msg = self.edge_mlp(self.norm_edge_in(edge_input))
        
        if self.update_coords:
            # on réutilise directement 'coord_diff' calculé précédemment
            pos_update = scatter_mean(coord_diff * self.coord_mlp(msg), row, dim=0, dim_size=x.size(0))
            x = x + pos_update * 0.1 # Dampening de 0.1 pour la stabilité
        # Node update avec Pre-Norm
        node_input = torch.cat([h, scatter_mean(msg, row, dim=0, dim_size=h.size(0))], dim=-1)
        h = h + self.node_mlp(self.norm_node_in(node_input))
        
        return self.norm(h), x

# =============================================================================
# 3. Cross-Attention avec distance bias 
# =============================================================================
class AEGIS_GT_CrossAttention(nn.Module):
    def __init__(self, h_dim, dist_temp=12.0):
        super().__init__()
        self.h_dim = h_dim
        self.head_dim = 64
        self.num_heads = h_dim // self.head_dim 

        self.dist_temp = nn.Parameter(torch.tensor([float(dist_temp)])) 
        
        # on remplace nn.MHA par des projections manuelles pour utiliser SDPA (Single Dot Product Attention)
        self.q_proj = nn.Linear(h_dim, h_dim)
        self.kv_proj = nn.Linear(h_dim, h_dim * 2)
        self.out_proj = nn.Linear(h_dim, h_dim)
        
        self.norm = RMSNorm(h_dim)

    def forward(self, hp, hl, p_batch, l_batch, pos_p, pos_l):
        # 1. Conversion en format Dense
        hp_d, p_mask = to_dense_batch(hp, p_batch) # [B, Seq_P, D]
        hl_d, l_mask = to_dense_batch(hl, l_batch) # [B, Seq_L, D]
        pp_d, _ = to_dense_batch(pos_p, p_batch)   # [B, Seq_P, 3]
        pl_d, _ = to_dense_batch(pos_l, l_batch)   # [B, Seq_L, 3]

        # 2. Calcul de la distance en float32
        # on force le calcul en f32 pour éviter les NaNs en bfloat16
        dist = torch.cdist(pl_d.float(), pp_d.float()) 
        
        # Softplus pour garantir une température toujours positive > 0
        safe_temp = F.softplus(self.dist_temp) + 1e-6
        bias = (-dist / safe_temp).to(hl_d.dtype) # on revient au dtype du modèle
        
        # 3. Projections Q, K, V pour l'attention
        # Forme cible : [B, num_heads, Seq, head_dim]
        q = self.q_proj(hl_d).view(hl_d.size(0), hl_d.size(1), self.num_heads, self.head_dim).transpose(1, 2)

        # Projection KV unique pour la protéine
        kv = self.kv_proj(hp_d) # [B, Seq_P, D * 2]
        k_raw, v_raw = kv.chunk(2, dim=-1) # Découpage en K et V
        
        k = k_raw.view(hp_d.size(0), hp_d.size(1), self.num_heads, self.head_dim).transpose(1, 2)
        v = v_raw.view(hp_d.size(0), hp_d.size(1), self.num_heads, self.head_dim).transpose(1, 2)
        
        # 4. Préparation du bias et du masque
        # bias shape: [B, Seq_L, Seq_P] -> on ajoute la dim des têtes : [B, 1, Seq_L, Seq_P]
        # SDPA fera le broadcasting automatique sur les 'num_heads' sans copier les données
        bias = bias.unsqueeze(1) 
        
        # Masque de padding pour la protéine (key_padding_mask)
        # p_mask: [B, Seq_P] -> [B, 1, 1, Seq_P]
        p_attn_mask = p_mask.unsqueeze(1).unsqueeze(2)

        # on force les scores d'attention des nœuds de padding à une valeur très basse
        # -1e4 pour float16 (pour éviter l'overflow négatif) ou -1e9 pour bfloat16/float32
        pad_value = -1e4 if hl_d.dtype == torch.float16 else -1e9
        combined_mask = bias.masked_fill(~p_attn_mask, pad_value)

        # 5. Flash Attention / SDPA
        # on combine le bias de distance et le masque de padding
        # on utilise l'astuce : mask = (mask_bool) + bias
        attn_out = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, 
            attn_mask=combined_mask, 
            dropout_p=0.0, 
            is_causal=False
        )
        
        # 6. Reprojection et Norm
        # [B, num_heads, Seq_L, head_dim] -> [B, Seq_L, D]
        attn_out = attn_out.transpose(1, 2).contiguous().view(hl_d.size(0), hl_d.size(1), -1)
        attn_out = self.out_proj(attn_out)
        
        return self.norm(attn_out[l_mask])           

In [ ]:
# =============================================================================
# 4. Pooling intelligent (Gated Attention)
# =============================================================================
class BillionScaleGatedPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.gate_dim = d_model // 2
        
        # Fusion des premières projections linéaires
        self.fused_proj = nn.Linear(d_model, self.gate_dim + d_model)
        self.gate_linear2 = nn.Linear(self.gate_dim, 1)
        self.norm = RMSNorm(d_model)
        
    def forward(self, x, batch):
        # Projection unique (1 seule opération GEMM sur le GPU)
        proj = self.fused_proj(x)
        
        # Découpage du tenseur fusionné
        gate_in, feat_in = proj.split([self.gate_dim, self.d_model], dim=-1)
        
        # Passage dans les fonctions d'activation respectives
        gate_out = self.gate_linear2(F.silu(gate_in))
        feat_out = F.silu(feat_in)
        
        # Calcul de l'attention Softmax et réduction
        weights = scatter_softmax(gate_out, batch, dim=0)
        pooled = scatter_add(weights * feat_out, batch, dim=0)
        
        return self.norm(pooled), weights

# =============================================================================
# 5. Architecture finale AEGIS_GT
# =============================================================================
class AEGIS_GT_Block_Logic(nn.Module):
    
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.head_dim = 64
        self.num_heads = d_model // self.head_dim

        # Le biais spatial apprenable (1 scalaire par tête d'attention)
        self.spatial_bias = nn.Parameter(torch.zeros(1, self.num_heads, 1, 1))
        
        # Projections d'attention (plus rapide que nn.MHA)
        self.qkv_p = nn.Linear(d_model, d_model * 3)
        self.out_p = nn.Linear(d_model, d_model)
        self.egnn_p = AEGIS_GT_Layer(d_model, update_coords=False)

        # Modules Ligands (Instanciés mais utilisés uniquement si hl_d n'est pas None)
        self.qkv_l = nn.Linear(d_model, d_model * 3)
        self.out_l = nn.Linear(d_model, d_model)
        self.egnn_l = AEGIS_GT_Layer(d_model, update_coords=True)
        
        self.ff = SwiGLU(d_model, dropout=0.1)
        self.n = RMSNorm(d_model)
    
    def forward(self, hp_d, p_mask, ep, xp_d, hl_d=None, l_mask=None, el=None, xl_d=None, p_batch=None):
        
        # Flux Protéine  
        # 1. Création du masque de Padding en Float
        # 0.0 pour les vrais acides aminés, -infinity pour le padding
        pad_value = -1e4 if hp_d.dtype == torch.float16 else -1e9
        float_attn_mask = torch.zeros_like(p_mask, dtype=hp_d.dtype)
        float_attn_mask = float_attn_mask.masked_fill(~p_mask, pad_value)
        float_attn_mask = float_attn_mask.unsqueeze(1).unsqueeze(2) # [B, 1, 1, L]

        # 2. Injection du Spatial Bias via les arêtes (ep)
        if ep is not None and ep.numel() > 0 and p_batch is not None:
            # on convertit les arêtes sparse en matrice d'adjacence dense [B, L, L]
            adj = to_dense_adj(ep, batch=p_batch, max_num_nodes=hp_d.size(1))
            adj = adj.unsqueeze(1) # [B, 1, L, L]
            
            # on ajoute le biais appris partout où adj == 1
            float_attn_mask = float_attn_mask + (adj * self.spatial_bias)

        # 3. Calcul Q, K, V
        qkv_p = self.qkv_p(self.n(hp_d)).reshape(hp_d.size(0), hp_d.size(1), 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        # on force la contiguïté mémoire de Q, K, V
        q_p, k_p, v_p = qkv_p[0].contiguous(), qkv_p[1].contiguous(), qkv_p[2].contiguous()
        # Flash Attention avec le dropout intégré 
        hp_attn = torch.nn.functional.scaled_dot_product_attention(
            q_p, k_p, v_p, 
            attn_mask=float_attn_mask,
            dropout_p=0.1 if self.training else 0.0  
        )
        hp_d = hp_d + self.out_p(hp_attn.permute(0, 2, 1, 3).reshape(hp_d.size(0), hp_d.size(1), -1))

        # EGNN Protéine
        hp_s = hp_d[p_mask]
        xp_s = xp_d[p_mask]
        hp_n, _ = self.egnn_p(hp_s, xp_s, ep)
        
        hp_out = hp_d.clone()
        hp_out[p_mask] = hp_s + self.ff(hp_n)

        # Flux Ligand (Actif uniquement en phase 2)
        hl_out, xl_out = None, None

        if hl_d is not None and l_mask is not None:
            
            l_float_mask = torch.zeros_like(l_mask, dtype=hl_d.dtype)
            l_float_mask = l_float_mask.masked_fill(~l_mask, pad_value)
            l_float_mask = l_float_mask.unsqueeze(1).unsqueeze(2)
            
            # 1. Calcul Q, K, V
            qkv_l = self.qkv_l(self.n(hl_d)).reshape(hl_d.size(0), hl_d.size(1), 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
            # on force la contiguïté mémoire de Q, K, V
            q_l, k_l, v_l = qkv_l[0].contiguous(), qkv_l[1].contiguous(), qkv_l[2].contiguous()
            # Flash Attention avec le dropout intégré
            hl_attn = torch.nn.functional.scaled_dot_product_attention(
                q_l, k_l, v_l, 
                attn_mask=l_float_mask, 
                dropout_p=0.1 if self.training else 0.0 
            )
            
            hl_d = hl_d + self.out_l(hl_attn.permute(0, 2, 1, 3).reshape(hl_d.size(0), hl_d.size(1), -1))
        
            # EGNN Ligand 
            # on extrait les nodes pour le calcul géométrique
            hl_s = hl_d[l_mask]
            xl_s = xl_d[l_mask]
            hl_n, xl_new_s = self.egnn_l(hl_s, xl_s, el)
        
            # 3. Mise à jour et Feed-Forward (Format DENSE) 
            # on utilise .clone() pour être compatible avec Gradient Checkpointing
        
            hl_out = hl_d.clone()
            xl_out = xl_d.clone()
        
            # Ré-injection du signal affiné par SwiGLU (self.ff)
            hl_out[l_mask] = hl_s + self.ff(hl_n)
            xl_out[l_mask] = xl_new_s # Mise à jour des positions du ligand

        return hp_out, hl_out, xl_out

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.utils import to_dense_batch
from torch.utils.checkpoint import checkpoint

class AEGIS_GT(nn.Module):
    def __init__(self, p_dim=2589, l_dim=23, h_dim=768, n_layers=6, rdkit_dim=15):
        super().__init__()
        
        # 1. Encodeurs Initiaux
        self.p_proj = nn.Linear(p_dim, h_dim)
        self.l_proj = nn.Linear(l_dim, h_dim)

        self.physics_adapter = nn.Sequential(
            nn.Linear(6, h_dim // 2),
            SwiGLU(h_dim // 2, dropout=0.1),
            nn.Linear(h_dim // 2, h_dim)
        )

        nn.init.constant_(self.physics_adapter[-1].weight, 0.0)
        nn.init.constant_(self.physics_adapter[-1].bias, 0.0)
        
        # 2. Le Backbone Géométrique
        self.layers = nn.ModuleList([AEGIS_GT_Block_Logic(h_dim) for _ in range(n_layers)])
        
        # 3. Cross-Attention & Pooling (Réservé Phase 2)
        self.cross_attn_l2p = AEGIS_GT_CrossAttention(h_dim) 
        self.cross_attn_p2l = AEGIS_GT_CrossAttention(h_dim) 
        self.l_pool = BillionScaleGatedPooling(h_dim) # Uniquement pour le Ligand

        # ==========================================
        # 4. Tetes de prédiction de la phase 1 (Physique de la Mutation)
        # ==========================================

        # Remet la variance à 1.0 après les 6 couches
        self.phase1_final_norm = RMSNorm(h_dim)

        self.phase1_heads = nn.ModuleDict({
            'structural_rmsd_A': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            ),
            'true_delta_sasa': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            ),
            'true_delta_packing': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            ),
            'true_delta_solv_hydro': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            ),
            'true_delta_electro': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            ),
            'true_delta_clash': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            ),
            'true_delta_ddg': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            )
        })

        # ==========================================
        # 5. Tetes de prédiction de la phase 2 (Affinité + Résistance clinique)
        # ==========================================
        # Fusion : Protéine (768) + Ligand (768) + RDKit (15) = 1551
        fusion_input_dim = (h_dim * 2) + rdkit_dim
        # on utilise SwiGLU pour la fusion, 
        self.fusion = nn.Sequential(
            nn.Linear(fusion_input_dim, 1024), 
            SwiGLU(1024, dropout=0.2), 
            RMSNorm(1024)
        )
        
        # Régression de l'affinité (delta_pAff) -> 1 dimension 
        self.phase2_pAff_head = nn.Sequential(
            nn.Linear(1024, 512),
            SwiGLU(512, dropout=0.1),
            nn.Linear(512, 1)
        )
        

    def forward(self, data):
        # ---------------------------------------------------------
        # Detection de la phase (Data = Phase 1, HeteroData = Phase 2)
        # ---------------------------------------------------------
        is_phase2 = hasattr(data, 'ligand_global_feat')

        # 1. Extraction Protéine (Commune aux deux phases)
        p_x = data['protein'].x if is_phase2 else data.x
        xp  = data['protein'].pos if is_phase2 else data.pos
        p_batch = data['protein'].batch if is_phase2 else getattr(data, 'batch', torch.zeros(p_x.size(0), dtype=torch.long, device=p_x.device))
        ep  = data['protein', 'interacts', 'protein'].edge_index if is_phase2 else data.edge_index
        local_mut_idx = data.local_mut_idx.view(-1) # Shape [Batch]

        if is_phase2:
            physics_features = p_x[:, 23:29] # Extraction des 6 métriques physiques
            p_x_base = p_x.clone()
            p_x_base[:, 23:29] = 0.0         # Le backbone continue de voir des zéros
            
            hp = self.p_proj(p_x_base)
            hp = hp + self.physics_adapter(physics_features) # on additionne le savoir
        else:
            hp = self.p_proj(p_x) # Phase 1 classique
            
        hp_d, p_mask = to_dense_batch(hp, p_batch)
        xp_d, _ = to_dense_batch(xp, p_batch)
        
        # Coordinate jittering
        if self.training:
            # on ajoute un micro-bruit de 0.12 Å uniquement pendant l'entraînement pour empêcher la mémorisation de la géométrie 3D des protéines
            xp_d = xp_d + torch.randn_like(xp_d) * 0.12

        # Filet de sécurité arêtes Protéine
        if ep.numel() > 0:
            row_p, col_p = ep
            mask_p = (row_p >= 0) & (row_p < hp.size(0)) & (col_p >= 0) & (col_p < hp.size(0))
            ep = ep[:, mask_p]

        # 2. Extraction Ligand (Seulement si Phase 2)
        hl_d, xl_d, el, l_mask = None, None, None, None
        if is_phase2:
            hl = self.l_proj(data['ligand'].x)
            xl = data['ligand'].pos
            l_batch = data['ligand'].batch
            el = data['ligand', 'interacts', 'ligand'].edge_index
            
            if el.numel() > 0:
                row_l, col_l = el
                mask_l = (row_l >= 0) & (row_l < hl.size(0)) & (col_l >= 0) & (col_l < hl.size(0))
                el = el[:, mask_l]

            hl_d, l_mask = to_dense_batch(hl, l_batch)
            xl_d, _ = to_dense_batch(xl, l_batch)

        # ========================================================
        # 3. Le Backbone (Gradient Checkpointing Interleaved)
        # ========================================================
        for i, layer in enumerate(self.layers):
            if i % 2 == 1 and self.training:
                # on doit passer les arguments dans l'ordre exact de la classe BGTBlock_Logic
                hp_d, hl_d, xl_d = torch.utils.checkpoint.checkpoint(
                    layer, hp_d, p_mask, ep, xp_d, hl_d, l_mask, el, xl_d, p_batch,
                    use_reentrant=False
                )
            else:
                hp_d, hl_d, xl_d = layer(hp_d, p_mask, ep, xp_d, hl_d, l_mask, el, xl_d, p_batch)


        # ========================================================
        # 5. Bifurcation des prédictions (Phase 1 vs Phase 2)
        # ========================================================
        outputs = {}

        if not is_phase2:

            # ========================================================
            # 4. Target Pooling (Extraction du vecteur de la mutation)
            # ========================================================
            # hp_d shape: [Batch, MaxSeq, 768]
            # on extrait précisément le vecteur à l'index de la mutation pour chaque élément du batch
            batch_indices = torch.arange(hp_d.size(0), device=hp_d.device)
            p_mut_vec = hp_d[batch_indices, local_mut_idx, :] # Shape: [Batch, 768]
            
            # Normalisation finale obligatoire
            p_mut_vec = self.phase1_final_norm(p_mut_vec) # Normalisation de sortie

            # Inférence propre et vectorisée par dictionnaire sur les 7 têtes indépendantes
            outputs = {name: head(p_mut_vec) for name, head in self.phase1_heads.items()}

            outputs['pred_deltas_tensor'] = torch.cat([
                outputs['structural_rmsd_A'], outputs['true_delta_sasa'], 
                outputs['true_delta_packing'], outputs['true_delta_solv_hydro'], 
                outputs['true_delta_electro'], outputs['true_delta_clash'], 
                outputs['true_delta_ddg']
            ], dim=-1) # Shape final: [B, 7]
            
            return outputs, None, None

        else:
            # En mode phase 2 
            hp_sparse = hp_d[p_mask]
            hl_sparse = hl_d[l_mask]
            xl_sparse = xl_d[l_mask]
            xp_sparse = xp_d[p_mask]

            # Cross-Attention
            # le ligand regarde la protéine
            hl_final = self.cross_attn_l2p(hp_sparse, hl_sparse, data['protein'].batch, data['ligand'].batch, xp_sparse, xl_sparse)

            # La protéine regarde le ligand
            hp_final = self.cross_attn_p2l(hl_sparse, hp_sparse, data['ligand'].batch, data['protein'].batch, xl_sparse, xp_sparse)


            # 2. Target pooling (sur la protéine qui a vu le ligand)
            # on remet la protéine en Dense pour pouvoir utiliser l'index de la mutation
            hp_final_d, _ = to_dense_batch(hp_final, data['protein'].batch)
            batch_indices = torch.arange(hp_final_d.size(0), device=hp_final_d.device)
            p_mut_vec_aware = hp_final_d[batch_indices, local_mut_idx, :] # Le vecteur est "conscient" du ligand !  
            
            # Gated pooling (sur le ligand qui a vu la protéine)
            l_vec, l_w = self.l_pool(hl_final, data['ligand'].batch)
            
            # Fusion : Target Pooled Protein + Gated Pooled Ligand + RDKit
            rdkit_global = data.ligand_global_feat.view(p_mut_vec_aware.size(0), -1)
            combined = torch.cat([p_mut_vec_aware, l_vec, rdkit_global], dim=-1)
            
            feat = self.fusion(combined)
            
            outputs = {
                'delta_pAff': self.phase2_pAff_head(feat)
            }
            return outputs, None, l_w # p_w n'est plus pertinent via Target Pooling
            
    def prepare_for_phase2(self):
        """
        Gèle le savoir de la protéine mutée (Phase 1) mais garde 
        actifs le ligand, la physique et la fusion pour la Phase 2.
        """
        #1. on gèle tout par défaut (le backbone Phase 1)
        for param in self.parameters():
            param.requires_grad = False
            
        # 2. on débloque l'adaptateur physique 
        for param in self.physics_adapter.parameters():
            param.requires_grad = True
            
        # 3. on débloque l'encodeur du Ligand
        for param in self.l_proj.parameters():
            param.requires_grad = True
            
        # 4. on débloque uniquement la partie Ligand dans le Backbone
        for layer in self.layers:
            for param in layer.qkv_l.parameters(): param.requires_grad = True
            for param in layer.out_l.parameters(): param.requires_grad = True
            for param in layer.egnn_l.parameters(): param.requires_grad = True
            
        # 5. on débloque la Cross-Attention et le Pooling Phase 2
        for param in self.cross_attn_l2p.parameters(): param.requires_grad = True
        for param in self.cross_attn_p2l.parameters(): param.requires_grad = True
        for param in self.l_pool.parameters(): param.requires_grad = True  
            
        # 6. on débloque les têtes de prédiction Phase 2
        for param in self.fusion.parameters(): param.requires_grad = True
        for param in self.phase2_pAff_head.parameters(): param.requires_grad = True
        
        # Affichage du statut dans la console
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        frozen = sum(p.numel() for p in self.parameters() if not p.requires_grad)
        print(f"Modèle préparé pour la Phase 2.")
        print(f"➜ Paramètres gelés (Savoir Phase 1) : {frozen:,}")
        print(f"➜ Paramètres entraînables (Ligand + Fusion) : {trainable:,}")

In [ ]:
# =============================================================================
# 6. Pairwise Ranking Loss
# =============================================================================
def pairwise_ranking_loss(pred, target, genes, margin=0.1):
    """
    Perte de classement par paires. 
    l'objectif est que si Target_A > Target_B, alors Pred_A > Pred_B.
    Sert à affiner la direction de l'affinité (Phase 2).
    """
    # Squeeze pour s'assurer d'avoir des vecteurs 1D
    pred = pred.view(-1)
    target = target.detach().view(-1)

    # s'assurer que genes est un tenseur 1D (ex: des IDs de gènes entiers)
    genes = genes.view(-1)
    
    if pred.size(0) < 2: 
        return torch.tensor(0.0, device=pred.device, requires_grad=True)
    
    # Calcul des différences par paires (Matrice N x N)
    # on utilise le broadcasting pour créer la matrice de différences
    diff_pred = pred.unsqueeze(1) - pred.unsqueeze(0)
    diff_target = target.unsqueeze(1) - target.unsqueeze(0)
    
    # Masque pour ne considérer que les paires où la différence de cible est significative (supérieure à la marge)
    mask_margin = diff_target.abs() > margin 

    # Masque de Gène (Séquence d'identité)
    # on crée une matrice booléenne où [i, j] est True si gene[i] == gene[j]
    gene_mask = (genes.unsqueeze(1) == genes.unsqueeze(0))

    # on combine les deux masques, même gène et différence significative
    final_mask = mask_margin & gene_mask
    
    # Perte de classement : on pénalise si la direction de la prédiction est opposée à la direction de la cible
    # Formule : relu(margin - sign(diff_target) * diff_pred)
    loss = F.relu(margin - diff_target.sign() * diff_pred)
    
    # Moyenne pondérée par le masque
    return (loss * final_mask.float()).sum() / (final_mask.float().sum() + 1e-6)

# =============================================================================
# 3. Initialisation des Poids (Kaiming/Xavier)
# =============================================================================
def init_weights(m):
    if isinstance(m, nn.Linear):
        # Têtes de sortie (Linear heads) 
        # Pour les sorties finales (Classification ou Régression ), 
        # on utilise Xavier pour une distribution centrée et stable.
        if m.out_features in [1, 3, 7, 8]:
            nn.init.xavier_uniform_(m.weight)
        
        # Couches Internes (Séquence/Graphe/Fusion) 
        # Pour tout le reste, on utilise Kaiming (He) car on utilise les fonctions d'activation SiLU/Swiglu.
        # Cela évite la disparition du gradient dans les réseaux profonds.
        else:
            nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
        
        # Initialisation du biais à zéro pour éviter tout décalage initial
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)


In [ ]:
import torch
import logging

if __name__ == "__main__":
    
    # 1. Création du modèle en Float32 (par défaut)
    model = AEGIS_GT(p_dim=2589, h_dim=256, n_layers=4)
    
    # 2. Initialisation statistique des poids en Float32
    model.apply(init_weights)
    
    # 3. Transfert vers le GPU en float32 (l'autocast s'occupera du bfloat16 plus tard) 
    model = model.to("cuda")
    
    # 4. Mesure et affichage des paramètres
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Le modèle AEGIS_GT est initialisé !")
    print(f"Le nombre de paramètres entraînables est : {total_params:,}")

In [ ]:
import graphviz
from IPython.display import display

def draw_bgt_architecture():
    # Création du graphe dirigé
    dot = graphviz.Digraph('AEGIS_GT', comment='Affinity Evaluation & Geometric Induced-fit Screening', format='png')
    dot.attr(rankdir='TB', size='12,15', fontname='Helvetica', fontsize='14', nodesep='0.6', ranksep='0.8')
    
    # Styles globaux
    dot.attr('node', shape='box', style='filled, rounded', fontname='Helvetica', fontsize='12', margin='0.2')
    dot.attr('edge', fontname='Helvetica', fontsize='10', color='#555555')

    # ==========================================
    # 1. ENTRÉES (INPUTS)
    # ==========================================
    with dot.subgraph(name='cluster_inputs') as c:
        c.attr(label="Données d'Entrée (Inputs)", style='dashed', color='grey', bgcolor='#f9f9f9')
        # Protéine divisée proprement pour l'adaptation physique
        c.node('P_Seq_Feat', 'Séquence Protéine\n[2583 dims]\n(ESM2 + pLDDT + OneHot)', fillcolor='#d0e8f2', shape='cylinder')
        c.node('P_Phys_Feat', 'Métriques Physiques\n[6 dims]\n(SASA, Packing, ddG...)', fillcolor='#bbdefb', shape='cylinder')
        c.node('P_Geom', 'Géométrie Protéine\n[Coords 3D + Edges]', fillcolor='#d0e8f2', shape='cylinder')
        
        # Ligand
        c.node('L_Feat', 'Features Ligand\n[23 dims]\n(Gasteiger, Types...)', fillcolor='#ffe4c4', shape='cylinder')
        c.node('L_Geom', 'Géométrie Ligand\n[Coords 3D + Edges]', fillcolor='#ffe4c4', shape='cylinder')
        c.node('L_Global', 'Features Globales RDKit\n[15 dims]', fillcolor='#ffe4c4', shape='cylinder')

    # ==========================================
    # 2. ENCODEURS (PROJECTIONS / ADAPTATION)
    # ==========================================
    dot.node('P_Proj', 'Projection Linéaire Protéine\n(Linear: 2589 → h_dim)', fillcolor='#a6dcef')
    dot.node('Phys_Adapt', 'Adaptateur Physique (Phase 2)\n(Linear + SwiGLU: 6 → h_dim)', fillcolor='#90caf9')
    dot.node('P_Sum', 'Sommation des Embeddings\nhp = hp_base + hp_phys', fillcolor='#e3f2fd', shape='circle')
    
    dot.node('L_Proj', 'Projection Linéaire Ligand\n(Linear: 23 → h_dim)', fillcolor='#ffcba4')

    # Connexions des Projections
    dot.edge('P_Seq_Feat', 'P_Proj', label=' hp_base')
    dot.edge('P_Phys_Feat', 'Phys_Adapt', label=' Phase 2 Only')
    dot.edge('P_Proj', 'P_Sum')
    dot.edge('Phys_Adapt', 'P_Sum')
    
    dot.edge('L_Feat', 'L_Proj', label=' Phase 2 Only')

    # ==========================================
    # 3. BACKBONE (4 LAYERS)
    # ==========================================
    with dot.subgraph(name='cluster_backbone') as c:
        c.attr(label='Backbone Géométrique (x4 Layers)\nGradient Checkpointing', style='solid', color='purple', bgcolor='#f3e8ff', penwidth='2')
        
        c.node('SpatialBias', 'Spatial Bias (Arêtes)\n[to_dense_adj]', fillcolor='#e9d5ff')
        c.node('P_Attn', 'Flash Attention (SDPA)\nProtéine (Self)', fillcolor='#d8b4fe')
        c.node('P_EGNN', 'EGNN Layer (Protéine)\n[Sans update_coords]', fillcolor='#c084fc')
        c.node('P_SwiGLU', 'RMSNorm + SwiGLU\n[Dropout 0.1]', fillcolor='#a855f7', fontcolor='white') 

        c.node('L_Attn', 'Flash Attention (SDPA)\nLigand (Self)', fillcolor='#d8b4fe')
        c.node('L_EGNN', 'EGNN Layer (Ligand)\n[Avec update_coords]', fillcolor='#c084fc')
        c.node('L_SwiGLU', 'RMSNorm + SwiGLU\n[Dropout 0.1]', fillcolor='#a855f7', fontcolor='white')

    # Flux Protéine Backbone
    dot.edge('P_Sum', 'SpatialBias')
    dot.edge('P_Geom', 'SpatialBias', label=' ep')
    dot.edge('SpatialBias', 'P_Attn', label=' mask + bias')
    dot.edge('P_Attn', 'P_EGNN')
    dot.edge('P_Geom', 'P_EGNN', label=' coords')
    dot.edge('P_EGNN', 'P_SwiGLU')

    # Flux Ligand Backbone
    dot.edge('L_Proj', 'L_Attn')
    dot.edge('L_Attn', 'L_EGNN')
    dot.edge('L_Geom', 'L_EGNN', label=' coords')
    dot.edge('L_EGNN', 'L_SwiGLU')

    # ==========================================
    # 4. EXTRACTION CHIRURGICALE (PHASE 1)
    # ==========================================
    dot.node('TargetPool_1', 'Target Pooling (Phase 1)\nExtraction Index Mutation\n[Batch, h_dim]', fillcolor='#ffb347', shape='diamond')
    dot.edge('P_SwiGLU', 'TargetPool_1', label=' Phase 1 (No Ligand)')

    # ==========================================
    # 5. PHASE 1 HEADS
    # ==========================================
    with dot.subgraph(name='cluster_phase1') as c:
        c.attr(label='Phase 1 : Lois de la Physique', style='dashed', color='red', bgcolor='#ffeded')
        c.node('P1_Norm', 'RMSNorm', fillcolor='#ff9999')
        c.node('P1_MLP', 'SwiGLU MLP\n[h_dim → h_dim//4 → 7]', fillcolor='#ff6666', fontcolor='white')
        c.node('P1_Out', '7 Régressions Cibles\n(RMSD, SASA, Packing, DDG...)', fillcolor='#ff3333', fontcolor='white', shape='note')

    dot.edge('TargetPool_1', 'P1_Norm')
    dot.edge('P1_Norm', 'P1_MLP')
    dot.edge('P1_MLP', 'P1_Out')

    # ==========================================
    # 6. PHASE 2 : CROSS-ATTENTION (Symétrique)
    # ==========================================
    with dot.subgraph(name='cluster_cross') as c:
        c.attr(label='Phase 2 : Ajustement Induit (Induced Fit)', style='dashed', color='green', bgcolor='#e8f5e9')
        c.node('Cross_L2P', 'Cross-Attention L2P\n(Ligand regarde Protéine)', fillcolor='#a5d6a7')
        c.node('Cross_P2L', 'Cross-Attention P2L\n(Protéine regarde Ligand)', fillcolor='#a5d6a7')
    dot.edge('P_SwiGLU', 'Cross_L2P', label=' K, V')
    dot.edge('L_SwiGLU', 'Cross_L2P', label=' Q')
    
    dot.edge('L_SwiGLU', 'Cross_P2L', label=' K, V')
    dot.edge('P_SwiGLU', 'Cross_P2L', label=' Q')

    # ==========================================
    # 7. PHASE 2 : POOLING & FUSION
    # ==========================================
    dot.node('TargetPool_2', 'Target Pooling (Phase 2)\nProtéine Consciente du Ligand\n[Batch, h_dim]', fillcolor='#ffb347', shape='diamond')
    dot.node('GatedPool', 'Gated Pooling\nRéduction Séquence Ligand\n[Batch, h_dim]', fillcolor='#ffb347', shape='diamond')
    
    dot.edge('Cross_P2L', 'TargetPool_2')
    dot.edge('Cross_L2P', 'GatedPool')
    dot.node('Concat', 'Concaténation (Fusion)\n[h_dim*2 + 15 = 527 dims]', fillcolor='#cccccc', shape='invhouse')
    dot.edge('TargetPool_2', 'Concat')
    dot.edge('GatedPool', 'Concat')
    dot.edge('L_Global', 'Concat', style='dashed')

    # ==========================================
    # 8. PHASE 2 HEADS 
    # ==========================================
    with dot.subgraph(name='cluster_phase2') as c:
        c.attr(label='Phase 2 : Régression d\'Affinité Pure', style='dashed', color='blue', bgcolor='#e3f2fd')
        c.node('P2_Fusion_MLP', 'Fusion SwiGLU + RMSNorm\n[527 → 1024]', fillcolor='#90caf9')
        c.node('P2_pAff_MLP', 'pAff SwiGLU Head\n[1024 → 512]', fillcolor='#64b5f6')
        c.node('P2_Out1', 'delta_pAff (Régression)\n[1 dim]', fillcolor='#1e88e5', fontcolor='white', shape='note')

    dot.edge('Concat', 'P2_Fusion_MLP')
    dot.edge('P2_Fusion_MLP', 'P2_pAff_MLP')
    dot.edge('P2_pAff_MLP', 'P2_Out1')

    # Rendu et affichage
    dot.render('AEGIS_GT_Architecture', view=False)
    display(dot)

# Exécution de la fonction pour dessiner le graphe
draw_bgt_architecture()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import colorsys

# 1. Configuration de l'environnement graphique Premium (Échelle augmentée à 24 pour la régression pure)
fig, ax = plt.subplots(figsize=(24, 12), dpi=300) # Haute résolution
ax.set_xlim(-1, 24)
ax.set_ylim(-1, 11)
ax.axis('off')

# Convertisseur Hex -> RGB
def hex_to_rgb(hex_str):
    hex_str = hex_str.lstrip('#')
    return [int(hex_str[i:i+2], 16)/255.0 for i in (0, 2, 4)]

# Ajustement de la luminosité pour l'effet de matière 3D
def adjust_lightness(color, amount=0.5):
    rgb = hex_to_rgb(color)
    c = colorsys.rgb_to_hls(*rgb)
    new_rgb = colorsys.hls_to_rgb(c[0], max(0, min(1, c[1] * amount)), c[2])
    return new_rgb

# Fonction avancée pour dessiner un tenseur 3D ombré et élégant
def draw_premium_tensor(ax, x, y, w, h, d, color, label=None, dims=None):
    theta = np.radians(25)  # Angle isométrique doux
    dx = d * np.cos(theta) * 0.4
    dy = d * np.sin(theta) * 0.4
    
    # Déclinaisons de couleurs pour l'effet 3D
    c_top = adjust_lightness(color, 1.15)
    c_right = adjust_lightness(color, 0.85)
    c_border = adjust_lightness(color, 0.6)  # Bordure très fine de la même teinte
    
    # 1. Ombre portée au sol (Drop Shadow)
    sh_y = -0.6
    shadow = patches.Polygon([
        [x, y+sh_y], 
        [x+w, y+sh_y], 
        [x+w+dx, y+dy+sh_y], 
        [x+dx, y+dy+sh_y]
    ], facecolor='#e2e8f0', edgecolor='none', alpha=0.6, zorder=0)
    ax.add_patch(shadow)
    
    # 2. Face avant (Front)
    front = patches.Polygon([[x, y], [x+w, y], [x+w, y+h], [x, y+h]], 
                            facecolor=color, edgecolor=c_border, linewidth=0.5, zorder=2)
    ax.add_patch(front)
    
    # 3. Face supérieure (Top)
    top = patches.Polygon([[x, y+h], [x+dx, y+h+dy], [x+w+dx, y+h+dy], [x+w, y+h]], 
                                            facecolor=c_top, edgecolor=c_border, linewidth=0.5, zorder=2)
    ax.add_patch(top)
    # 4. Face latérale droite (Right)
    right = patches.Polygon([[x+w, y], [x+w+dx, y+dy], [x+w+dx, y+h+dy], [x+w, y+h]], 
                            facecolor=c_right, edgecolor=c_border, linewidth=0.5, zorder=2)
    ax.add_patch(right)
    
    # 5. Textes & Étiquettes
    if label:
        ax.text(x + w/2, y + h/2, label, ha='center', va='center', 
                fontsize=9, fontweight='bold', color='#1e293b', wrap=True, zorder=3)
        
    if dims:
        ax.text(x + w/2, y - 0.25, dims[0], ha='center', va='top', fontsize=7.5, color='#64748b', zorder=3)
        ax.text(x - 0.2, y + h/2, dims[1], ha='right', va='center', fontsize=7.5, color='#64748b', zorder=3)
        ax.text(x + w + dx/2 + 0.1, y + dy/2 - 0.1, dims[2], ha='left', va='center', 
                fontsize=7.5, color='#64748b', rotation=20, zorder=3)

# Fonction de connexion stylisée
def draw_premium_connection(ax, start, end, label=None, label_pos=None, style='-'):
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle="->", lw=1.2, color='#64748b', ls=style,
                                shrinkA=4, shrinkB=4, mutation_scale=12))
    if label and label_pos:
        ax.text(label_pos[0], label_pos[1], label, fontsize=8, ha='center', va='bottom', color='#475569')

# =============================================================================
# Rendu de l'architecture AEGIS-GT (Régression Pure)
# =============================================================================
# Palette de couleurs "Modern Biotech"
C_PROT   = '#a5f3fc'  # Cyan protéine
C_PHYS   = '#93c5fd'  # Bleu clair adaptateur
C_LIG    = '#fed7aa'  # Orange pastel ligand
C_RDKIT  = '#fbcfe8'  # Rose RDKit
C_BACK   = '#ddd6fe'  # Violet clair backbone
C_CROSS  = '#bbf7d0'  # Vert cross-attn
C_POOL   = '#fef08a'  # Jaune pooling
C_FUSION = '#cbd5e1'  # Gris fusion
C_P1_OUT = '#fca5a5'  # Rouge Phase 1
C_P2_OUT = '#93c5fd'  # Bleu Phase 2

# ==========================================
# 1. ENTRÉES (INPUTS)
# ==========================================
# Séquence de base Protéine
draw_premium_tensor(ax, x=0.5, y=8.4, w=1.5, h=1.2, d=0.4, color=C_PROT, 
                    label='Protein\nSequence\nFeatures', dims=['Batch', 'N_prot', '2583'])
# Métriques Physiques (FoldX/SASA)
draw_premium_tensor(ax, x=0.5, y=6.6, w=1.5, h=0.8, d=0.4, color=C_PHYS, 
                    label='Biophysical\nMetrics', dims=['Batch', 'N_prot', '6'])
# Coordonnées 3D Protéine
draw_premium_tensor(ax, x=0.5, y=4.8, w=1.5, h=0.8, d=0.4, color=C_PROT, 
                    label='Protein\nGeometry', dims=['Batch', 'N_prot', 'Coords'])

# Ligand Features
draw_premium_tensor(ax, x=0.5, y=3.0, w=1.5, h=1.0, d=0.4, color=C_LIG, 
                    label='Ligand\nFeatures', dims=['Batch', 'N_lig', '23'])
# Ligand Coordonnées 3D
draw_premium_tensor(ax, x=0.5, y=1.5, w=1.5, h=0.8, d=0.4, color=C_LIG, 
                    label='Ligand\nGeometry', dims=['Batch', 'N_lig', 'Coords'])
# RDKit Global
draw_premium_tensor(ax, x=0.5, y=0.1, w=1.5, h=0.5, d=0.4, color=C_RDKIT, 
                    label='RDKit\nGlobal', dims=['Batch', '1', '15'])

# ==========================================
# 2. ENCODEURS & PROJECTIONS
# ==========================================
# Projection Séquence
draw_premium_tensor(ax, x=4.0, y=8.4, w=1.5, h=0.8, d=0.4, color=C_PROT, 
                    label='Linear Proj.\n(2583 → 256)')
# Adaptateur Physique (Zéro-Init)
draw_premium_tensor(ax, x=4.0, y=6.6, w=1.5, h=0.8, d=0.4, color=C_PHYS, 
                    label='Physics Adapter\n(6 → 256)')
# Sommation des Embeddings
draw_premium_tensor(ax, x=6.2, y=7.7, w=1.0, h=0.8, d=0.4, color=C_FUSION, 
                    label='hp Sum')

# Projection Ligand
draw_premium_tensor(ax, x=4.0, y=3.0, w=1.5, h=0.8, d=0.4, color=C_LIG, 
                    label='Linear Proj.\n(23 → 256)')

# Connexions projections/adaptateur
draw_premium_connection(ax, (2.05, 9.0), (3.9, 9.0))
draw_premium_connection(ax, (2.05, 7.0), (3.9, 7.0), label='Phase 2 Only', label_pos=(3.0, 7.1))
draw_premium_connection(ax, (5.55, 8.8), (6.1, 8.35))
draw_premium_connection(ax, (5.55, 7.0), (6.1, 7.7))

draw_premium_connection(ax, (2.05, 3.5), (3.9, 3.5))

# ==========================================
# 3. BACKBONES (X4 LAYERS)
# ==========================================
# Protein Backbone (x4 Blocks)
draw_premium_tensor(ax, x=8.2, y=6.0, w=1.8, h=2.0, d=0.5, color=C_BACK, 
                    label='AEGIS\nProtein\nBackbone\n(4 Blocks)', dims=['Batch', 'N_prot', '256'])
# Ligand Backbone
draw_premium_tensor(ax, x=8.2, y=2.5, w=1.8, h=1.4, d=0.5, color=C_BACK, 
                    label='AEGIS\nLigand\nBackbone', dims=['Batch', 'N_lig', '256'])

draw_premium_connection(ax, (7.25, 8.1), (8.1, 7.6))
draw_premium_connection(ax, (2.05, 5.2), (8.1, 6.5), style='--') # Géométrie protéine
draw_premium_connection(ax, (5.55, 3.4), (8.1, 3.4))
draw_premium_connection(ax, (2.05, 1.9), (8.1, 2.9), style='--') # Géométrie ligand

# ==========================================
# 4. TARGET POOLING & CROSS-ATTENTION
# ==========================================
# Target Pooling Phase 1 (Foyer Mutation)
draw_premium_tensor(ax, x=12.2, y=7.2, w=1.2, h=1.2, d=0.4, color=C_POOL, 
                    label='Target\nPooling\n(Mutation)', dims=['Batch', '1', '256'])
# Symmetric Cross-Attention (Induced-Fit)
draw_premium_tensor(ax, x=12.2, y=2.5, w=1.8, h=1.5, d=0.5, color=C_CROSS, 
                    label='Symmetric\nCross-Attention\n(Induced-Fit)', dims=['Batch', 'N_lig', '256'])

draw_premium_connection(ax, (10.05, 7.0), (12.1, 7.8), label='Index Slicing', label_pos=(11.0, 7.9))
draw_premium_connection(ax, (10.05, 3.2), (12.1, 3.25)) # Ligand vers Cross-Attn
draw_premium_connection(ax, (9.1, 6.0), (12.1, 3.5), label='K, V from Protein', label_pos=(10.8, 4.85)) # Protéine vers Cross-Attn

# ==========================================
# 5. SORTIES PHASE 1 (Physique)
# ==========================================
draw_premium_tensor(ax, x=15.0, y=7.2, w=1.5, h=1.2, d=0.4, color=C_P1_OUT, 
                    label='Phase 1\nOutputs\n(7 Regressions)', dims=['Batch', '1', '7'])
draw_premium_connection(ax, (13.45, 7.8), (14.9, 7.8), label='Phase 1 Only', label_pos=(14.2, 7.9))

# ==========================================
# 6. POOLING PHASE 2 & CONCATÉNATION
# ==========================================
# Target Pooling Protéine (Consciente du Ligand)
draw_premium_tensor(ax, x=15.2, y=4.4, w=1.2, h=1.2, d=0.4, color=C_POOL, 
                    label='Target\nPooling\n(Aware)', dims=['Batch', '1', '256'])
# Gated Pooling Ligand
draw_premium_tensor(ax, x=15.2, y=2.5, w=1.2, h=1.2, d=0.4, color=C_POOL, 
                    label='Gated\nPooling', dims=['Batch', '1', '256'])

draw_premium_connection(ax, (14.05, 3.5), (15.1, 4.8))
draw_premium_connection(ax, (14.05, 3.1), (15.1, 3.1))

# Bloc de Concaténation (Fusion globale)
draw_premium_tensor(ax, x=18.0, y=2.4, w=1.4, h=1.4, d=0.4, color=C_FUSION, 
                    label='FUSION\n(Concat)', dims=['Batch', '1', '527'])

draw_premium_connection(ax, (16.45, 5.0), (17.9, 3.4), label='Mut_Vec', label_pos=(17.0, 4.5))
draw_premium_connection(ax, (16.45, 3.1), (17.9, 3.1), label='Lig_Vec', label_pos=(17.15, 3.2))

# RDKit Global vers la fusion (Ligne pointillée rose)
ax.annotate('', xy=(18.1, 2.3), xytext=(2.05, 0.35),
            arrowprops=dict(arrowstyle="->", lw=1.0, ls="--", color='#db2777', shrinkA=4, shrinkB=4))

# ==========================================
# 7. SORTIE PHASE 2 (Affinité delta_pAff)
# ==========================================
draw_premium_tensor(ax, x=21.0, y=2.8, w=1.4, h=1.0, d=0.3, color=C_P2_OUT, 
                    label='delta_pAff\nRegression\n(Pure)', dims=['Batch', '1', '1'])
draw_premium_connection(ax, (19.45, 3.1), (20.9, 3.1))

# Titre du diagramme
ax.text(11.5, 10.4, "AEGIS-GT: 3D Volumetric Tensor Flow Diagram (Pure Regression)", ha='center', va='center', 
        fontsize=18, fontweight='bold', color='#0f172a')

plt.tight_layout()
plt.savefig("aegis_volumetric_architecture_premium.png", dpi=300, bbox_inches='tight')
plt.show()

## **Partie Entrainement Phase 1 du modèle Transformer Géométrique**

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pickle
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
from tqdm.auto import tqdm

# =============================================================================
# 1. Dénormalisation (pour des métriques biologiquement lisibles)
# =============================================================================
def denormalize_deltas(y_deltas):
    """
    Applique la transformation inverse en tenant compte de l'approche Hybride.
    y_deltas contient 8 colonnes : ['rmsd', sasa', 'pack', 'solve_hydro', 'electro', 'clash', 'ddg']
    """
    try:
        with open("phase1_scaler.pkl", "rb") as f:
            scaler = pickle.load(f)
    except Exception as e:
        print(f"Scalers non trouvés, les métriques seront calculées sur les données normalisées. Erreur: {e}")
        return y_deltas
    
    # Transformation inverse directe et instantanée sur les 7 colonnes
    return scaler.inverse_transform(y_deltas)

# =============================================================================
# 2. Calcul des métriques (Scikit-learn avec Post-Binning pour la RMSD)
# =============================================================================
def calculate_metrics(y_true_deltas, y_pred_deltas):
    metrics = {}
    
    # Dénormalisation avant calcul
    y_true_deltas_real = denormalize_deltas(y_true_deltas)
    y_pred_deltas_real = denormalize_deltas(y_pred_deltas)
    
    # Métrique de régression(sur les valeurs biologiques réelles)
    r2_list, mae_list = [], []
    tasks = ['rmsd', 'sasa', 'pack', 'solv_hydro', 'electro', 'clash', 'ddg']
    
    for i, name in enumerate(tasks):
        r2 = r2_score(y_true_deltas_real[:, i], y_pred_deltas_real[:, i])
        mae = mean_absolute_error(y_true_deltas_real[:, i], y_pred_deltas_real[:, i])
        metrics[f'R2_{name}'] = r2
        metrics[f'MAE_{name}'] = mae

        # on calcule la moyenne des deltas physiques (hors RMSD qui est la structure)
        if name != 'rmsd':
            r2_list.append(r2)
            mae_list.append(mae)
    
    metrics['R2_Mean'] = np.mean(r2_list)
    metrics['MAE_Mean'] = np.mean(mae_list)

    # Post-Bining classification de la RMSD (3 Classes)
    q33 = 0.2550
    q66 = 0.5215
    
    y_true_rmsd_real = y_true_deltas_real[:, 0]
    y_pred_rmsd_real = y_pred_deltas_real[:, 0]
    
    y_true_rmsd_class = np.digitize(y_true_rmsd_real, bins=[q33, q66])
    y_pred_rmsd_class = np.digitize(y_pred_rmsd_real, bins=[q33, q66])

    metrics['Accuracy'] = accuracy_score(y_true_rmsd_class, y_pred_rmsd_class)
    metrics['Macro_F1'] = f1_score(y_true_rmsd_class, y_pred_rmsd_class, average='macro')

    prec = precision_score(y_true_rmsd_class, y_pred_rmsd_class, average=None, labels=[0, 1, 2], zero_division=0)
    rec = recall_score(y_true_rmsd_class, y_pred_rmsd_class, average=None, labels=[0, 1, 2], zero_division=0)
    f1_cls = f1_score(y_true_rmsd_class, y_pred_rmsd_class, average=None, labels=[0, 1, 2], zero_division=0)
    
    for c in range(3):
        metrics[f'P_cls{c}'] = prec[c]
        metrics[f'R_cls{c}'] = rec[c]
        metrics[f'F1_cls{c}'] = f1_cls[c]

    return metrics

# =============================================================================
# 3. Fonction d'affichage détaillée
# =============================================================================
def print_detailed_metrics(metrics):

    # Ajout dynamique des pertes si elles sont fournies
    train_loss_str = f"Training Loss: {metrics.get('Train_Loss', 0.0):.4f} | " if 'Train_Loss' in metrics else ""
    val_loss_str = f"Validation Loss: {metrics.get('Val_Loss', 0.0):.4f} | " if 'Val_Loss' in metrics else ""

    
    # 1. Ligne globale (Les moyennes)
    global_str = (f"{train_loss_str}{val_loss_str}Accuracy : {metrics['Accuracy']:.3f} | F1: {metrics['Macro_F1']:.3f} | "
                  f"Mean R²: {metrics['R2_Mean']:.3f} | Mean MAE: {metrics['MAE_Mean']:.3f}")
    
    # 2. Ligne des classes (RMSD)
    cls_str = " | ".join([f"Classe {c} (Precision :{metrics[f'P_cls{c}']:.2f} Recall :{metrics[f'R_cls{c}']:.2f} F1:{metrics[f'F1_cls{c}']:.2f})" for c in range(3)])
    class_line = f"RMSD Classes : {cls_str}"
    
    # 3. Ligne des régressions (Deltas)
    #tasks = ['plddt', 'sasa', 'packing', 'curv', 'hydro', 'charge', 'vol']
    tasks = ['rmsd', 'sasa', 'pack', 'solv_hydro', 'electro', 'clash', 'ddg']
    reg_str = " | ".join([f"{t[:7]} (R²:{metrics[f'R2_{t}']:.2f} MAE:{metrics[f'MAE_{t}']:.2f})" for t in tasks])
    reg_line = f"Deltas Physique  : {reg_str}"
    
    # Affichage final avec un petit séparateur pour la clarté
    print(f"\n{global_str}\n{class_line}\n{reg_line}\n")

# =============================================================================
# 4. Moteur de validation
# =============================================================================
def validate(model, val_loader, loss_manager, device, max_batches=None):
    model.eval()
    #all_true_rmsd, all_pred_rmsd = [], []
    all_true_deltas, all_pred_deltas = [], []

    total_val_loss = 0.0
    valid_batches = 0

    total_steps = len(val_loader)
    if max_batches is not None and max_batches < total_steps:
        total_steps = max_batches
    
    with torch.no_grad():
        for i, batch in enumerate(tqdm(val_loader, total=total_steps, desc="Évaluation", leave=False)):

            if max_batches is not None and i >= max_batches:
                break
                
            batch = batch.to(device)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                outputs, _, _ = model(batch)

                preds_deltas = outputs['pred_deltas_tensor'] # Shape: [B, 7]

                #targets_rmsd = batch.y_rmsd_class
                targets_deltas = batch.y_deltas.view(-1, 7)

                # Calcul de la Validation Loss
                loss, _ = loss_manager(preds_deltas, targets_deltas)
                total_val_loss += loss.item()
                valid_batches += 1

            all_true_deltas.append(targets_deltas.cpu().numpy())
            all_pred_deltas.append(preds_deltas.float().cpu().numpy())

    metrics = calculate_metrics(
        np.concatenate(all_true_deltas), 
        np.concatenate(all_pred_deltas)
    )

    # on ajoute la Validation Loss moyenne au dictionnaire des métriques
    metrics['Val_Loss'] = total_val_loss / max(1, valid_batches)
    
    return metrics 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================================================
# 2. Manager de perte Multi-Taches (Phase 1)
# =============================================================================
class Phase1LossManager(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.loss_fn = nn.HuberLoss(reduction='none', delta=1.0)
        self.device = device

        # Initialisés à 0.0, ce qui équivaut au départ à un poids de 1.0 (car exp(0) = 1) [6]
        self.log_vars = nn.Parameter(torch.zeros(7, device=device))


    def forward(self, preds_deltas, targets_deltas, target_mask=None):
        # preds_deltas: [B, 7], targets_deltas: [B, 7]
        individual_losses = self.loss_fn(preds_deltas, targets_deltas) # [B, 7]
        
        if target_mask is not None:
            # on s'assure que le masque est sur le bon device
            target_mask = target_mask.to(preds_deltas.device)
            
            # 1. Application du masque (les pertes des tâches ayant échoué tombent à 0.0)
            masked_losses = individual_losses * target_mask
    
            
            # 2. Moyenne par tâche pour le dictionnaire de debug
            sum_losses = masked_losses.sum(dim=0)
            count_valid = target_mask.sum(dim=0)
            mean_losses = sum_losses / (count_valid + 1e-6)
            # on détecte si la tâche a au moins 1 exemple valide dans ce batch
            task_active = (count_valid > 0).float()
        else:
            mean_losses = individual_losses.mean(dim=0)
            task_active = torch.ones(7, device=preds_deltas.device)

        # 2. Application de la formule de KENDALL ET AL. (CVPR 2018) [5, 6]
        # Loss_i = 0.5 * exp(-log_var_i) * Mean_Loss_i + 0.5 * log_var_i [6]
        # on utilise exp(-log_var) pour assurer mathématiquement que le diviseur reste positif [6]
        weighted_losses = (0.5 * torch.exp(-self.log_vars) * mean_losses + 0.5 * self.log_vars) * task_active
        
        # 3. La perte totale est la somme de toutes les pertes équilibrées [6]
        total_loss = weighted_losses.sum()
            
        losses = {
            'rmsd': mean_losses[0].item(),
            'true_delta_sasa': mean_losses[1].item(),
            'true_delta_packing': mean_losses[2].item(),
            'true_delta_solv_hydro': mean_losses[3].item(),
            'true_delta_electro': mean_losses[4].item(),
            'true_delta_clash': mean_losses[5].item(),
            'true_delta_ddg': mean_losses[6].item()
        }
        return total_loss, losses    

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
import gc
import os

def train_phase1(model, train_loader, val_loader, test_loader, epochs=80, device='cuda', accum_steps=1, eval_every_n_steps=300, resume_ckpt_path=None):
    print("\nLancement de l'entraînement Phase 1...")
    
    loss_manager = Phase1LossManager(device=device)
    
    # optimizer 
    optimizer = optim.AdamW([
        {'params': model.parameters(), 'weight_decay': 0.05},      # Modèle avec Weight Decay [2.1, 6]
        {'params': loss_manager.parameters(), 'weight_decay': 0.0} # Loss sans Weight Decay [6]
    ], lr=1e-4)
    
    total_steps = (len(train_loader) // accum_steps) * epochs
    scheduler = OneCycleLR(optimizer, max_lr=1e-4, total_steps=total_steps, pct_start=0.1)

    start_epoch = 0 
    global_step = 0
    best_val_loss = float('inf')
    patience_counter = 0
    patience_limit = 10 # Early Stopping
    best_ckpt_path = None


    if resume_ckpt_path is not None and os.path.exists(resume_ckpt_path):
        print(f"\nReprise de l'entraînement détectée depuis : {resume_ckpt_path}")

        # Sécurité PyTorch 2.6 : weights_only=False
        checkpoint = torch.load(resume_ckpt_path, map_location=device, weights_only=False)

        # Restauration des poids et de la "mémoire" de l'entraînement
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])

        # on restaure les compteurs pour reprendre exactement là où on s'est arrêté
        start_epoch = checkpoint.get('epoch', 1) - 1  # -1 car la boucle range() commence à 0
        global_step = checkpoint.get('step', 0)
        best_val_loss = checkpoint.get('best_score', float('inf'))
        best_ckpt_path = resume_ckpt_path
        print(f"Modèle restauré avec succès !")
        print(f"Reprise à l'Époque {start_epoch+1}/{epochs} | Step global : {global_step} | Meilleur Perte : {best_val_loss:.4f}\n")
    else:
        print("\nLancement d'un nouvel entraînement Phase 1...")

    steps_per_epoch = len(train_loader) // accum_steps
    
    try:
        for epoch in range(start_epoch, epochs):
            model.train()
            running_train_loss = 0.0
            running_steps = 0

            # 1. Calcul du décalage (offset) si on reprend au milieu de l'époque
            if epoch == start_epoch:
                batches_to_skip = (global_step % steps_per_epoch) * accum_steps
            else:
                batches_to_skip = 0
        
            # 2. Configuration de tqdm avec 'initial' pour l'affichage visuel
            pbar = tqdm(total=len(train_loader), initial=batches_to_skip, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        
            for i, batch in enumerate(train_loader):

                if epoch == start_epoch and i < batches_to_skip:
                    pbar.update(1)
                    continue
                    
                batch = batch.to(device)
            
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    outputs, _, _ = model(batch)
                                  
                    preds_deltas = outputs['pred_deltas_tensor']
                    targets_deltas = batch.y_deltas.view(-1, 7)
            
                    loss, _ = loss_manager(preds_deltas, targets_deltas, batch.target_mask)
                    loss = loss / accum_steps
            
                loss.backward()
            
                # Application des gradients selon accum_steps
                if (i + 1) % accum_steps == 0 or (i + 1) == len(train_loader):
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                    optimizer.step()
                    optimizer.zero_grad()
                    scheduler.step()
                
                    global_step += 1
                    running_train_loss += (loss.item() * accum_steps)
                    running_steps += 1
                
                    # =================================================================
                    # Validation intra-époque (Échantillonnage régulier)
                    # =================================================================
                    if global_step % eval_every_n_steps == 0:

                        # on calcule la moyenne de la train loss sur les 10 000 derniers steps
                        avg_train_loss = running_train_loss / max(1, running_steps)
                    
                        metrics = validate(model, val_loader, loss_manager, device, max_batches=1000)

                        metrics['Train_Loss'] = avg_train_loss
                    
                        print_detailed_metrics(metrics)

                        # on remet les compteurs à zéro pour la prochaine fenêtre de 10 000 steps !
                        running_train_loss = 0.0
                        running_steps = 0
                    
                        # Sauvegarde du meilleur modèle avec minimisation de la Val Loss
                        current_val_loss = metrics['Val_Loss']
                    
                        if current_val_loss < best_val_loss:
                            best_val_loss = current_val_loss
                            patience_counter = 0
                        
                            # 1. Création du checkpoint complet
                            checkpoint = {
                                'epoch': epoch + 1, 
                                'step': global_step, 
                                'model_state': model.state_dict(), 
                                'optimizer_state': optimizer.state_dict(), 
                                'scheduler_state': scheduler.state_dict(), 
                                'best_score': best_val_loss
                            }

                            # 2. Nom dynamique du nouveau fichier
                            new_ckpt_path = f"aegis_best_ep{epoch+1}_step{global_step}_mae{best_val_loss:.4f}.pth"

                            # 3. Suppression de l'ancien checkpoint (pour ne pas saturer le disque)
                            # on ne supprime l'ancien fichier que s'il ne provient pas de '/kaggle/input/'
                            if best_ckpt_path is not None and os.path.exists(best_ckpt_path):
                                if "input" not in best_ckpt_path: 
                                    os.remove(best_ckpt_path)
                                
                            # 4. Sauvegarde du nouveau
                            torch.save(checkpoint, new_ckpt_path)
                            best_ckpt_path = new_ckpt_path # Mise à jour de la variable pour le prochain tour
                        
                            print(f"📉 Nouveau Record de Perte (Val Loss: {best_val_loss:.4f}). Checkpoint sauvegardé ({new_ckpt_path}) !")
                            
                        else:
                            patience_counter += 1
                            print(f"Patience: {patience_counter}/{patience_limit}")
                        
                        if patience_counter >= patience_limit:
                            print(f"Early stopping déclenché au step {global_step}.")
                            break # sort de la boucle des batchs
                        
                        model.train() # retour en mode train

                # Mise à jour de l'affichage de la barre
                pbar.update(1)
                
                pbar.set_postfix({'Loss': f"{loss.item() * accum_steps:.4f}"})  

                # on arrête de piocher quand on a atteint le compte total de l'époque
                if (i + 1) >= len(train_loader):
                    break

        
            pbar.close() # on ferme proprement la barre à la fin de l'époque    
            
            # Nettoyage VRAM fin d'époque
            torch.cuda.empty_cache()
            gc.collect()
        
            if patience_counter >= patience_limit:
                break # sort de la boucle des époques

    except KeyboardInterrupt:
        print(f"\n\nBouton stop pressé! Sauvegarde d'urgence au step {global_step} en cours...")
        checkpoint_urgence = {
            'epoch': epoch + 1, 
            'step': global_step, 
            'model_state': model.state_dict(), 
            'optimizer_state': optimizer.state_dict(), 
            'scheduler_state': scheduler.state_dict(), 
            'best_score': best_val_loss 
        }
        nom_fichier = f"aegis_reprise_step{global_step}.pth"
        torch.save(checkpoint_urgence, nom_fichier)
        print(f"Sauvegarde d'urgence réussie : {nom_fichier}")
        return model
            
    # =================================================================
    # Validation Finale (sur le validation set entier)
    # =================================================================
    print("\nEntrainement Terminé.")
    print("Chargement des meilleurs poids pour l'évaluation finale sur le validation Set...")
    try:
        # on charge le dictionnaire complet, puis on extrait uniquement les poids du modèle
        checkpoint = torch.load(best_ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state'])
        print(f"Checkpoint restauré avec succès depuis {best_ckpt_path}")
    except Exception as e:
        print(f"Restauration échouée, utilisation des poids actuels. Erreur: {e}")
        
    final_metrics = validate(model, test_loader, loss_manager, device)
    print_detailed_metrics(final_metrics)
    
    return model

In [ ]:
import os
import gc

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from torch_geometric.loader import DataLoader

if 'train_loader' in globals(): del globals()['train_loader']
if 'val_loader' in globals(): del globals()['val_loader']
if 'test_loader' in globals(): del globals()['test_loader']
gc.collect() # Force le garbage collector à faire le ménage proprement en silence

if __name__ == "__main__":
    
    # 1. Paramètres d'entraînement
    BATCH_SIZE = 32
    ACCUM_STEPS = 2  # Batch effectif = 32 * 2 = 64
    EPOCHS = 50       # 50 époques suffisent sur 4205 variants

    # on autorise jusqu'à 8 workers
    OPTIMAL_WORKERS = min(os.cpu_count(), 8) 
    print(f"Déploiement de la puissance brute : {OPTIMAL_WORKERS} workers activés.")
    
    # 2. Création des DataLoaders Géométriques
    # train_dataset et val_dataset ont été créés dans votre fichier de données
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        sampler=train_sampler, 
        num_workers=OPTIMAL_WORKERS,
        persistent_workers=True if OPTIMAL_WORKERS > 0 else False,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=0,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=0,               
        pin_memory=True
    )
    
    # 3. Lancement de l'entraînement
    # on valide tous les 300 steps

    RESUME_CHECKPOINT = None
    
    train_phase1(
        model=model, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        test_loader=test_loader, 
        epochs=EPOCHS, 
        device='cuda', 
        accum_steps=ACCUM_STEPS, 
        eval_every_n_steps=300,
        resume_ckpt_path=RESUME_CHECKPOINT
    )

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
 
# ── Données extraites des logs d'entraînement Phase 1 ──────────────────────────
epochs     = [5,      10,     14,     19,     23,     28,     32,     37,     41,     45,     50]
val_loss   = [1.4017, 1.3467, 1.2834, 1.1910, 1.1694, 1.1416, 1.1155, 1.0833, 1.0748, 1.0781, 1.0754]
train_loss = [0.1349, 0.0606,-0.0874,-0.1977,-0.2788,-0.3589,-0.4193,-0.4670,-0.4889,-0.4920,-0.4980]
accuracy   = [0.494,  0.519,  0.501,  0.519,  0.511,  0.492,  0.515,  0.523,  0.535,  0.532,  0.535]
f1         = [0.501,  0.521,  0.507,  0.524,  0.516,  0.501,  0.523,  0.530,  0.539,  0.538,  0.540]
mean_r2    = [0.063,  0.077,  0.101,  0.147,  0.106,  0.119,  0.107,  0.134,  0.132,  0.126,  0.130]
 
best_epoch = 41
best_val   = 1.0748
 
r2_labels = ["Solvatation", "Packing", "SASA", "RMSD", "Électro", "Clash", "ΔΔG"]
r2_values = [0.30, 0.25, 0.18, 0.14, 0.03, 0.08, -0.10]
r2_colors = ["#2ECC71" if v > 0.15 else ("#F0A500" if v >= 0 else "#FF6B6B") for v in r2_values]
 
# ── Style ───────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0F0F1A",
    "axes.facecolor":   "#0F0F1A",
    "axes.edgecolor":   "#2A2A40",
    "axes.labelcolor":  "#C8C8E0",
    "xtick.color":      "#8888AA",
    "ytick.color":      "#8888AA",
    "text.color":       "#E0E0F0",
    "grid.color":       "#1E1E30",
    "grid.linewidth":   0.6,
    "grid.linestyle":   "--",
    "font.family":      "DejaVu Sans",
    "font.size":        11,
})
 
PURPLE = "#7B68EE"
TEAL   = "#2ECC71"
CORAL  = "#FF6B6B"
AMBER  = "#F0A500"
GRAY   = "#555577"
 
# ── Layout : 2 lignes × 2 colonnes ─────────────────────────────────────────────
fig = plt.figure(figsize=(18, 11))
fig.patch.set_facecolor("#0F0F1A")
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.32)
 
ax1 = fig.add_subplot(gs[0, 0])   # Val Loss & Train Loss
ax2 = fig.add_subplot(gs[0, 1])   # Accuracy & F1
ax3 = fig.add_subplot(gs[1, 0])   # Mean R²
ax4 = fig.add_subplot(gs[1, 1])   # R² par tâche (barh indépendant)
 
fig.suptitle(
    "AEGIS-GT — Courbes d'entraînement Phase 1",
    fontsize=15, fontweight="bold", color="#E0E0F0", y=0.98
)
 
# ── Panneau 1 : Val Loss & Train Loss ──────────────────────────────────────────
ax1.set_facecolor("#0F0F1A")
ax1.grid(True)
 
ax1.plot(epochs, val_loss,   color=PURPLE, lw=2.2, marker="o", ms=5,
         label="Validation Loss", zorder=3)
ax1.plot(epochs, train_loss, color=TEAL,   lw=2.0, marker="s", ms=4,
         linestyle="--", label="Training Loss", zorder=3)
ax1.fill_between(epochs, train_loss, val_loss, alpha=0.07, color=PURPLE)
 
ax1.axvline(x=best_epoch, color=AMBER, lw=1.2, linestyle=":", alpha=0.8)
ax1.scatter([best_epoch], [best_val], color=AMBER, s=90, zorder=5,
            label=f"Best ep{best_epoch}  (Val={best_val})")
ax1.annotate(
    f"  ep{best_epoch}\n  {best_val}",
    xy=(best_epoch, best_val),
    xytext=(best_epoch + 1.8, best_val + 0.05),
    color=AMBER, fontsize=9,
    arrowprops=dict(arrowstyle="->", color=AMBER, lw=1.0)
)
ax1.axvspan(45, 50, alpha=0.06, color=CORAL)
ax1.text(45.3, 1.35, "patience\n1–2/10", color=CORAL, fontsize=8, alpha=0.85)
 
ax1.text(0.03, 0.04,
    "* Train Loss négative :\nformulation Kendall et al.\n(log_var apprenables)",
    transform=ax1.transAxes, fontsize=7.5, color="#8888AA", va="bottom",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#1A1A2E",
              edgecolor="#333355", alpha=0.7))
 
ax1.set_title("Pertes d'entraînement & validation", color="#C8C8E0", fontsize=12, pad=8)
ax1.set_xlabel("Époque")
ax1.set_ylabel("Loss")
ax1.legend(fontsize=9, framealpha=0.2,
           facecolor="#1A1A2E", edgecolor="#333355", labelcolor="#C8C8E0")
 
# ── Panneau 2 : Accuracy & F1 ──────────────────────────────────────────────────
ax2.set_facecolor("#0F0F1A")
ax2.grid(True)
 
ax2.plot(epochs, accuracy, color=CORAL,  lw=2.2, marker="o", ms=5, label="Accuracy")
ax2.plot(epochs, f1,       color=AMBER,  lw=2.0, marker="s", ms=4,
         linestyle="--", label="F1-macro")
ax2.axhline(y=1/3, color=GRAY, lw=1.0, linestyle=":", alpha=0.7, label="Hasard (33.3%)")
ax2.text(5, 1/3 + 0.005, "baseline aléatoire", color=GRAY, fontsize=8, alpha=0.8)
 
ax2.annotate("",
    xy=(41, 0.535), xytext=(41, 1/3),
    arrowprops=dict(arrowstyle="<->", color="#AAAACC", lw=1.2))
ax2.text(42.5, (0.535 + 1/3) / 2,
    f"+{(0.535 - 1/3)*100:.1f}pp\nvs hasard",
    color="#AAAACC", fontsize=8, va="center")
 
ax2.axvline(x=best_epoch, color=AMBER, lw=1.2, linestyle=":", alpha=0.8)
ax2.set_ylim(0.28, 0.62)
ax2.set_title("Accuracy & F1-macro (classification RMSD)", color="#C8C8E0", fontsize=12, pad=8)
ax2.set_xlabel("Époque")
ax2.set_ylabel("Score")
ax2.legend(fontsize=9, framealpha=0.2,
           facecolor="#1A1A2E", edgecolor="#333355", labelcolor="#C8C8E0")
 
# ── Panneau 3 : Mean R² ────────────────────────────────────────────────────────
ax3.set_facecolor("#0F0F1A")
ax3.grid(True)
 
ax3.plot(epochs, mean_r2, color=TEAL, lw=2.2, marker="D", ms=5, label="Mean R² (7 tâches)")
ax3.fill_between(epochs, mean_r2, alpha=0.12, color=TEAL)
ax3.axvline(x=best_epoch, color=AMBER, lw=1.2, linestyle=":", alpha=0.8)
ax3.scatter([best_epoch], [0.132], color=AMBER, s=80, zorder=5,
            label=f"Best ep{best_epoch}  R²=0.132")
 
ax3.set_ylim(-0.02, 0.22)
ax3.set_title("Mean R² — 7 tâches biophysiques", color="#C8C8E0", fontsize=12, pad=8)
ax3.set_xlabel("Époque")
ax3.set_ylabel("Mean R²")
ax3.legend(fontsize=9, framealpha=0.2,
           facecolor="#1A1A2E", edgecolor="#333355", labelcolor="#C8C8E0")
 
# ── Panneau 4 : R² par tâche — axe numérique ───────────────────────────────────
ax4.set_facecolor("#0F0F1A")
ax4.grid(True, axis="x")
 
y_pos = np.arange(len(r2_labels))
bars  = ax4.barh(y_pos, r2_values, color=r2_colors,
                 edgecolor="none", height=0.55)
 
ax4.set_yticks(y_pos)
ax4.set_yticklabels(r2_labels, fontsize=11, color="#C8C8E0")
ax4.axvline(x=0, color="#555577", lw=1.0)
ax4.set_xlim(-0.20, 0.40)
ax4.set_xlabel("R²")
ax4.set_title("R² par tâche biophysique — checkpoint ep41",
              color="#C8C8E0", fontsize=12, pad=8)
 
for bar, val in zip(bars, r2_values):
    offset = 0.012 if val >= 0 else -0.012
    ha     = "left"  if val >= 0 else "right"
    ax4.text(val + offset,
             bar.get_y() + bar.get_height() / 2,
             f"{val:.2f}", va="center", ha=ha,
             fontsize=10, color="#E0E0F0")
 
legend_patches = [
    mpatches.Patch(color=TEAL,  label="R² > 0.15 — bien appris"),
    mpatches.Patch(color=AMBER, label="R² 0–0.15 — signal faible"),
    mpatches.Patch(color=CORAL, label="R² < 0 — labels bruités"),
]
ax4.legend(handles=legend_patches, fontsize=9, framealpha=0.2, loc="lower right",
           facecolor="#1A1A2E", edgecolor="#333355", labelcolor="#C8C8E0")
 
# Bandeau récapitulatif 
fig.text(
    0.5, 0.01,
    f"Best checkpoint : époque {best_epoch}  |  Val Loss = {best_val}  |  "
    f"Accuracy = 51.4%  |  F1 = 0.518  |  "
    f"Protocole : Gene-Disjoint OOD  |  AEGIS-GT (22.9M params)",
    ha="center", va="bottom", fontsize=9, color="#7777AA",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#111122",
              edgecolor="#2A2A40", alpha=0.8)
)
 
plt.savefig("aegis_phase1_training_curves.png",
            dpi=180, bbox_inches="tight",
            facecolor="#0F0F1A", edgecolor="none")
plt.show()
print("Figure sauvegardée : aegis_phase1_training_curves.png")

In [ ]:
import numpy as np
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

def generate_and_plot_confusion_matrix(model, loader, device, q33=0.2550, q66=0.5215):
    print("Collecte des prédictions sur le Test Set...")
    model.eval()
    
    all_true_real = []
    all_pred_real = []
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                outputs, _, _ = model(batch)
                
                # on collecte toutes les 7 colonnes pour satisfaire le Scaler [2.1]
                preds = outputs['pred_deltas_tensor'] # Shape: [B, 7]
                targets = batch.y_deltas.view(-1, 7)  # Shape: [B, 7]
                
            all_true_real.append(targets.cpu().numpy())
            all_pred_real.append(preds.float().cpu().numpy())
            
    # Dénormalisation globale des 7 cibles d'un coup (Tenseur [N, 7] valide pour le scaler) [2.1]
    y_true_real_all = denormalize_deltas(np.concatenate(all_true_real)) # [N, 7]
    y_pred_real_all = denormalize_deltas(np.concatenate(all_pred_real)) # [N, 7]
    
    # on extrait maintenant l'index 0 (RMSD réel dénormalisé) [2.1]
    y_true_raw = y_true_real_all[:, 0]
    y_pred_raw = y_pred_real_all[:, 0]
    
    # Post-Binning selon vos terciles réels de Cellule 7
    y_true_class = np.digitize(y_true_raw, bins=[q33, q66])
    y_pred_class = np.digitize(y_pred_raw, bins=[q33, q66])
    
    # Calcul de la matrice brute et en pourcentages
    cm = confusion_matrix(y_true_class, y_pred_class, labels=[0, 1, 2])
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Configuration graphique Premium (Sleek)
    plt.figure(figsize=(10, 8), dpi=300)
    labels = [
        f'Faible (< {q33:.4f} Å)', 
        f'Moyen\n({q33:.4f} - {q66:.4f} Å)', 
        f'Fort (>= {q66:.4f} Å)'
    ]
    
    # Création des annotations textuelles (Effectif + Pourcentage)
    annot = np.empty_like(cm, dtype=object)
    for i in range(3):
        for j in range(3):
            annot[i, j] = f"{cm[i, j]:,}\n({cm_percent[i, j]:.1f}%)"
            
    # Tracé du Heatmap Indigo
    sns.heatmap(cm, annot=annot, fmt='', cmap='Purples', xticklabels=labels, yticklabels=labels, 
                cbar=True, square=True, cbar_kws={"shrink": .8})
    
    plt.title("AEGIS-GT : Matrice de Confusion de la Déformation Structurelle (RMSD)", 
              fontsize=13, fontweight='bold', pad=20, color='#0f172a')
    plt.xlabel("Classes Prédites", fontsize=10, fontweight='bold', labelpad=12, color='#1e293b')
    plt.ylabel("Classes Réelles (Vérité Terrain)", fontsize=10, fontweight='bold', labelpad=12, color='#1e293b')
    
    plt.tight_layout()
    plt.savefig("confusion_matrix_rmsd.png", dpi=300, bbox_inches='tight')
    plt.show()

# Appel de la fonction (utilise le modèle chargé avec ses meilleurs poids et votre test_loader)
generate_and_plot_confusion_matrix(model, test_loader, device='cuda')